# bitfit codet5 770m 1024 token

In [ ]:

import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
)

warnings.filterwarnings("ignore")

os.environ["CUDA_VISIBLE_DEVICES"]    = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"]  = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


MODEL_NAME    = "Salesforce/codet5p-770m"
TRAIN_CSV     = "/code vul/trqw.csv"
TEST_CSV      = "/code vul/teqw.csv"
MAX_LENGTH    = 1024
BATCH_SIZE    = 16
NUM_EPOCHS    = 16
LEARNING_RATE = 1e-3
WARMUP_RATIO  = 0.06
VAL_SPLIT     = 0.1
NUM_LABELS    = 2
SEED          = 42
CHECKPOINT    = "best_bitfit_codet5p_checkpoint.pt"
LOSS_PLOT     = "bitfit_codet5p_loss_curve.png"


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


class CodeVulnerabilityDataset(Dataset):
    def __init__(
        self,
        dataframe:   pd.DataFrame,
        tokenizer,
        max_length:  int,
    ) -> None:
        self.texts      = dataframe["func_cleaned"].astype(str).tolist()
        self.labels     = dataframe["label"].astype(int).tolist()
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> dict:
        encoding = self.tokenizer(
            self.texts[idx],
            max_length      = self.max_length,
            padding         = "max_length",
            truncation      = True,
            return_tensors  = "pt",
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }


class BitFitCodeT5pClassifier(nn.Module):


    def __init__(self, model_name: str, num_labels: int) -> None:
        super().__init__()
        self.num_labels = num_labels

        full_model = AutoModel.from_pretrained(
            model_name,
            trust_remote_code = True,
            torch_dtype       = torch.float16,
        )

        self.encoder = full_model.encoder

        hidden_size = full_model.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Dropout(p=0.1),
            nn.Linear(hidden_size, num_labels),
        )

        for layer in self.classifier:
            if isinstance(layer, nn.Linear):
                nn.init.normal_(layer.weight, std=0.02)
                nn.init.zeros_(layer.bias)

        del full_model
        torch.cuda.empty_cache()

        self._apply_bitfit_freezing()

    def _apply_bitfit_freezing(self) -> None:
        for param in self.encoder.parameters():
            param.requires_grad = False

        unfrozen_bias_count = 0
        for name, param in self.encoder.named_parameters():
            if name.endswith(".bias") and param is not None:
                param.data          = param.data.float()
                param.requires_grad = True
                unfrozen_bias_count += param.numel()

        for layer in self.classifier:
            if isinstance(layer, nn.Linear):
                layer.weight.data = layer.weight.data.float()
                layer.bias.data   = layer.bias.data.float()
                layer.weight.requires_grad = True
                layer.bias.requires_grad   = True

        total_encoder     = sum(p.numel() for p in self.encoder.parameters())
        total_all         = sum(p.numel() for p in self.parameters())
        trainable_all     = sum(p.numel() for p in self.parameters() if p.requires_grad)
        classifier_params = sum(p.numel() for p in self.classifier.parameters())

        print("=" * 64)
        print("  BitFit  |  Salesforce/codet5p-770m  |  Parameter Summary")
        print("=" * 64)
        print(f"  Encoder total            : {total_encoder:>14,}")
        print(f"  Encoder bias (unfrozen)  : {unfrozen_bias_count:>14,}")
        print(f"  Classifier head          : {classifier_params:>14,}")
        print(f"  Total parameters         : {total_all:>14,}")
        print(f"  Trainable parameters     : {trainable_all:>14,}")
        print(f"  Trainable fraction       : "
              f"{100.0 * trainable_all / total_all:>12.6f}%")
        print("=" * 64)

        if trainable_all == 0:
            raise RuntimeError(
                "No trainable parameters detected after BitFit freezing. "
                "Verify that the encoder contains .bias tensors."
            )

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        labels:         torch.Tensor = None,
    ):
        encoder_outputs = self.encoder(
            input_ids      = input_ids,
            attention_mask = attention_mask,
        )

        hidden_states = encoder_outputs.last_hidden_state.float()

        mask_expanded = attention_mask.unsqueeze(-1).float()
        sum_hidden    = (hidden_states * mask_expanded).sum(dim=1)
        token_counts  = mask_expanded.sum(dim=1).clamp(min=1e-9)
        pooled        = sum_hidden / token_counts

        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return loss, logits


def train_epoch(
    model:     nn.Module,
    loader:    DataLoader,
    optimizer: torch.optim.Optimizer,
    scheduler,
    scaler:    GradScaler,
    device:    torch.device,
):
    model.train()
    total_loss = total_correct = total_samples = 0

    for batch in loader:
        input_ids      = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels         = batch["labels"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type="cuda", dtype=torch.float16):
            loss, logits = model(input_ids, attention_mask, labels)

        if loss.dim() > 0:
            loss = loss.mean()

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            max_norm=1.0,
        )

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        bs             = labels.size(0)
        total_loss    += loss.item() * bs
        preds          = logits.detach().argmax(dim=-1)
        total_correct += (preds == labels).sum().item()
        total_samples += bs

    return total_loss / total_samples, total_correct / total_samples


@torch.no_grad()
def evaluate_epoch(
    model:  nn.Module,
    loader: DataLoader,
    device: torch.device,
):
    model.eval()
    total_loss = total_correct = total_samples = 0
    all_preds  = []
    all_labels = []

    for batch in loader:
        input_ids      = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels         = batch["labels"].to(device, non_blocking=True)

        with autocast(device_type="cuda", dtype=torch.float16):
            loss, logits = model(input_ids, attention_mask, labels)

        if loss.dim() > 0:
            loss = loss.mean()

        bs             = labels.size(0)
        total_loss    += loss.item() * bs
        preds          = logits.argmax(dim=-1)
        total_correct += (preds == labels).sum().item()
        total_samples += bs
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    f1       = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, accuracy, f1, all_preds, all_labels


def plot_loss_curves(
    train_losses: list,
    val_losses:   list,
    save_path:    str,
) -> None:
    epochs = list(range(1, len(train_losses) + 1))
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(epochs, train_losses, marker="o", linewidth=1.8,
            markersize=4, label="Train Loss", color="#1f77b4")
    ax.plot(epochs, val_losses,   marker="s", linewidth=1.8,
            markersize=4, label="Validation Loss", color="#d62728")
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("Cross-Entropy Loss", fontsize=12)
    ax.set_title(
        "BitFit  |  Salesforce/codet5p-770m  |  Code Vulnerability Detection\n"
        "Train vs Validation Loss per Epoch",
        fontsize=11,
    )
    ax.legend(fontsize=11)
    ax.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"\nLoss curve saved -> {save_path}")


def main() -> None:
    set_seed(SEED)

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is not available. At least one GPU is required to run "
            "this script."
        )

    device = torch.device("cuda")
    n_gpus = torch.cuda.device_count()
    print(f"Device: {device}  |  GPUs visible: {n_gpus}")
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}  |  "
              f"Memory: {props.total_memory / 1e9:.1f} GB")

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        trust_remote_code = True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "right"

    train_df_full = pd.read_csv(TRAIN_CSV)
    test_df       = pd.read_csv(TEST_CSV)

    print(f"\nRaw data -> Train: {len(train_df_full)}  |  Test: {len(test_df)}")
    print(f"Label distribution (train):\n{train_df_full['label'].value_counts()}")

    train_df, val_df = train_test_split(
        train_df_full,
        test_size    = VAL_SPLIT,
        random_state = SEED,
        stratify     = train_df_full["label"],
    )
    train_df = train_df.reset_index(drop=True)
    val_df   = val_df.reset_index(drop=True)
    test_df  = test_df.reset_index(drop=True)

    print(
        f"Split -> Train: {len(train_df)}  |  "
        f"Val: {len(val_df)}  |  Test: {len(test_df)}"
    )

    num_workers = min(4, os.cpu_count() or 1)

    train_loader = DataLoader(
        CodeVulnerabilityDataset(train_df, tokenizer, MAX_LENGTH),
        batch_size  = BATCH_SIZE,
        shuffle     = True,
        num_workers = num_workers,
        pin_memory  = True,
        drop_last   = False,
    )
    val_loader = DataLoader(
        CodeVulnerabilityDataset(val_df, tokenizer, MAX_LENGTH),
        batch_size  = BATCH_SIZE,
        shuffle     = False,
        num_workers = num_workers,
        pin_memory  = True,
    )
    test_loader = DataLoader(
        CodeVulnerabilityDataset(test_df, tokenizer, MAX_LENGTH),
        batch_size  = BATCH_SIZE,
        shuffle     = False,
        num_workers = num_workers,
        pin_memory  = True,
    )

    model = BitFitCodeT5pClassifier(MODEL_NAME, NUM_LABELS)

    if n_gpus > 1:
        model = nn.DataParallel(model)

    model = model.to(device)

    trainable_params = [p for p in model.parameters() if p.requires_grad]

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr           = LEARNING_RATE,
        betas        = (0.9, 0.999),
        eps          = 1e-8,
        weight_decay = 0.0,
    )

    total_steps  = len(train_loader) * NUM_EPOCHS
    warmup_steps = int(WARMUP_RATIO * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps   = warmup_steps,
        num_training_steps = total_steps,
    )

    scaler = GradScaler(device="cuda")

    print(
        f"\nTraining configuration"
        f"\n  Model       : {MODEL_NAME}"
        f"\n  PEFT method : BitFit (Ben-Zaken et al., ACL 2022)"
        f"\n  Epochs      : {NUM_EPOCHS}"
        f"\n  LR          : {LEARNING_RATE}"
        f"\n  Batch size  : {BATCH_SIZE}"
        f"\n  Max length  : {MAX_LENGTH}"
        f"\n  Total steps : {total_steps}"
        f"\n  Warmup steps: {warmup_steps}"
        f"\n  GPUs in use : {n_gpus}"
    )

    train_losses  = []
    val_losses    = []
    best_val_f1   = -1.0
    best_val_loss = float("inf")

    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss, tr_acc = train_epoch(
            model, train_loader, optimizer, scheduler, scaler, device
        )
        vl_loss, vl_acc, vl_f1, _, _ = evaluate_epoch(
            model, val_loader, device
        )

        train_losses.append(tr_loss)
        val_losses.append(vl_loss)

        print(
            f"Epoch {epoch:02d}/{NUM_EPOCHS}"
            f"  |  Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f}"
            f"  |  Val Loss: {vl_loss:.4f}  Acc: {vl_acc:.4f}"
            f"  F1: {vl_f1:.4f}"
        )

        if vl_f1 > best_val_f1 or (vl_f1 == best_val_f1 and vl_loss < best_val_loss):
            best_val_f1   = vl_f1
            best_val_loss = vl_loss
            state_to_save = (
                model.module.state_dict()
                if isinstance(model, nn.DataParallel)
                else model.state_dict()
            )
            torch.save(state_to_save, CHECKPOINT)
            print(
                f"           checkpoint saved  "
                f"(val_f1={best_val_f1:.4f}  val_loss={best_val_loss:.4f})"
            )

    plot_loss_curves(train_losses, val_losses, LOSS_PLOT)

    raw_model = (
        model.module if isinstance(model, nn.DataParallel) else model
    )
    raw_model.load_state_dict(
        torch.load(CHECKPOINT, map_location=device, weights_only=True),
        strict=True,
    )

    if n_gpus > 1:
        model = nn.DataParallel(raw_model)
    else:
        model = raw_model
    model = model.to(device)

    te_loss, te_acc, te_f1, te_preds, te_labels = evaluate_epoch(
        model, test_loader, device
    )

    print("\n" + "=" * 64)
    print("  FINAL TEST RESULTS  (best checkpoint by macro-F1 + val loss)")
    print("=" * 64)
    print(f"  Test Loss       : {te_loss:.4f}")
    print(f"  Test Accuracy   : {te_acc:.4f}")
    print(f"  Test F1 (macro) : {te_f1:.4f}")
    print()
    print(
        classification_report(
            te_labels,
            te_preds,
            target_names = ["Not Vulnerable", "Vulnerable"],
            digits       = 4,
        )
    )
    print("=" * 64)


if __name__ == "__main__":
    main()

# bit fit codet5 220M 1024token

In [ ]:


import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    get_linear_schedule_with_warmup,
)

warnings.filterwarnings("ignore")

os.environ["CUDA_VISIBLE_DEVICES"]    = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"]  = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


MODEL_NAME    = "Salesforce/codet5p-220m"
TRAIN_CSV                   = 'code vul/trqw.csv'
TEST_CSV                    = 'code vul/teqw.csv'
MAX_LENGTH    = 1024
BATCH_SIZE    = 16
NUM_EPOCHS    = 16
LEARNING_RATE = 1e-3
WARMUP_RATIO  = 0.06
VAL_SPLIT     = 0.1
NUM_LABELS    = 2
SEED          = 42
CHECKPOINT    = "best_bitfit_codet5p_checkpoint.pt"
LOSS_PLOT     = "bitfit_codet5p_loss_curve.png"


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


class CodeVulnerabilityDataset(Dataset):
    def __init__(
        self,
        dataframe:   pd.DataFrame,
        tokenizer,
        max_length:  int,
    ) -> None:
        self.texts      = dataframe["func_cleaned"].astype(str).tolist()
        self.labels     = dataframe["label"].astype(int).tolist()
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> dict:
        encoding = self.tokenizer(
            self.texts[idx],
            max_length      = self.max_length,
            padding         = "max_length",
            truncation      = True,
            return_tensors  = "pt",
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }


class BitFitCodeT5pClassifier(nn.Module):


    def __init__(self, model_name: str, num_labels: int) -> None:
        super().__init__()
        self.num_labels = num_labels

        full_model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name,
            torch_dtype = torch.float16,
        )

        self.encoder = full_model.encoder

        hidden_size = full_model.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Dropout(p=0.1),
            nn.Linear(hidden_size, num_labels),
        )

        for layer in self.classifier:
            if isinstance(layer, nn.Linear):
                nn.init.normal_(layer.weight, std=0.02)
                nn.init.zeros_(layer.bias)

        del full_model
        torch.cuda.empty_cache()

        self._apply_bitfit_freezing()

    def _apply_bitfit_freezing(self) -> None:
        for param in self.encoder.parameters():
            param.requires_grad = False

        unfrozen_bias_count = 0
        for name, param in self.encoder.named_parameters():
            if name.endswith(".bias") and param is not None:
                param.data          = param.data.float()
                param.requires_grad = True
                unfrozen_bias_count += param.numel()

        for layer in self.classifier:
            if isinstance(layer, nn.Linear):
                layer.weight.data = layer.weight.data.float()
                layer.bias.data   = layer.bias.data.float()
                layer.weight.requires_grad = True
                layer.bias.requires_grad   = True

        total_encoder     = sum(p.numel() for p in self.encoder.parameters())
        total_all         = sum(p.numel() for p in self.parameters())
        trainable_all     = sum(p.numel() for p in self.parameters() if p.requires_grad)
        classifier_params = sum(p.numel() for p in self.classifier.parameters())

        print("=" * 64)
        print("  BitFit  |  Salesforce/codet5p-220m  |  Parameter Summary")
        print("=" * 64)
        print(f"  Encoder total            : {total_encoder:>14,}")
        print(f"  Encoder bias (unfrozen)  : {unfrozen_bias_count:>14,}")
        print(f"  Classifier head          : {classifier_params:>14,}")
        print(f"  Total parameters         : {total_all:>14,}")
        print(f"  Trainable parameters     : {trainable_all:>14,}")
        print(f"  Trainable fraction       : "
              f"{100.0 * trainable_all / total_all:>12.6f}%")
        print("=" * 64)

        if trainable_all == 0:
            raise RuntimeError(
                "No trainable parameters detected after BitFit freezing. "
                "Verify that the encoder contains .bias tensors."
            )

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        labels:         torch.Tensor = None,
    ):
        encoder_outputs = self.encoder(
            input_ids      = input_ids,
            attention_mask = attention_mask,
        )

        hidden_states = encoder_outputs.last_hidden_state.float()

        mask_expanded = attention_mask.unsqueeze(-1).float()
        sum_hidden    = (hidden_states * mask_expanded).sum(dim=1)
        token_counts  = mask_expanded.sum(dim=1).clamp(min=1e-9)
        pooled        = sum_hidden / token_counts

        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return loss, logits


def train_epoch(
    model:     nn.Module,
    loader:    DataLoader,
    optimizer: torch.optim.Optimizer,
    scheduler,
    scaler:    GradScaler,
    device:    torch.device,
):
    model.train()
    total_loss = total_correct = total_samples = 0

    for batch in loader:
        input_ids      = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels         = batch["labels"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type="cuda", dtype=torch.float16):
            loss, logits = model(input_ids, attention_mask, labels)

        if loss.dim() > 0:
            loss = loss.mean()

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            max_norm=1.0,
        )

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        bs             = labels.size(0)
        total_loss    += loss.item() * bs
        preds          = logits.detach().argmax(dim=-1)
        total_correct += (preds == labels).sum().item()
        total_samples += bs

    return total_loss / total_samples, total_correct / total_samples


@torch.no_grad()
def evaluate_epoch(
    model:  nn.Module,
    loader: DataLoader,
    device: torch.device,
):
    model.eval()
    total_loss = total_correct = total_samples = 0
    all_preds  = []
    all_labels = []

    for batch in loader:
        input_ids      = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels         = batch["labels"].to(device, non_blocking=True)

        with autocast(device_type="cuda", dtype=torch.float16):
            loss, logits = model(input_ids, attention_mask, labels)

        if loss.dim() > 0:
            loss = loss.mean()

        bs             = labels.size(0)
        total_loss    += loss.item() * bs
        preds          = logits.argmax(dim=-1)
        total_correct += (preds == labels).sum().item()
        total_samples += bs
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    f1       = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, accuracy, f1, all_preds, all_labels


def plot_loss_curves(
    train_losses: list,
    val_losses:   list,
    save_path:    str,
) -> None:
    epochs = list(range(1, len(train_losses) + 1))
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(epochs, train_losses, marker="o", linewidth=1.8,
            markersize=4, label="Train Loss", color="#1f77b4")
    ax.plot(epochs, val_losses,   marker="s", linewidth=1.8,
            markersize=4, label="Validation Loss", color="#d62728")
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("Cross-Entropy Loss", fontsize=12)
    ax.set_title(
        "BitFit  |  Salesforce/codet5p-220m  |  Code Vulnerability Detection\n"
        "Train vs Validation Loss per Epoch",
        fontsize=11,
    )
    ax.legend(fontsize=11)
    ax.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"\nLoss curve saved -> {save_path}")


def main() -> None:
    set_seed(SEED)

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is not available. At least one GPU is required to run "
            "this script."
        )

    device = torch.device("cuda")
    n_gpus = torch.cuda.device_count()
    print(f"Device: {device}  |  GPUs visible: {n_gpus}")
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}  |  "
              f"Memory: {props.total_memory / 1e9:.1f} GB")

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "right"

    train_df_full = pd.read_csv(TRAIN_CSV)
    test_df       = pd.read_csv(TEST_CSV)

    print(f"\nRaw data -> Train: {len(train_df_full)}  |  Test: {len(test_df)}")
    print(f"Label distribution (train):\n{train_df_full['label'].value_counts()}")

    train_df, val_df = train_test_split(
        train_df_full,
        test_size    = VAL_SPLIT,
        random_state = SEED,
        stratify     = train_df_full["label"],
    )
    train_df = train_df.reset_index(drop=True)
    val_df   = val_df.reset_index(drop=True)
    test_df  = test_df.reset_index(drop=True)

    print(
        f"Split -> Train: {len(train_df)}  |  "
        f"Val: {len(val_df)}  |  Test: {len(test_df)}"
    )

    num_workers = min(4, os.cpu_count() or 1)

    train_loader = DataLoader(
        CodeVulnerabilityDataset(train_df, tokenizer, MAX_LENGTH),
        batch_size  = BATCH_SIZE,
        shuffle     = True,
        num_workers = num_workers,
        pin_memory  = True,
        drop_last   = False,
    )
    val_loader = DataLoader(
        CodeVulnerabilityDataset(val_df, tokenizer, MAX_LENGTH),
        batch_size  = BATCH_SIZE,
        shuffle     = False,
        num_workers = num_workers,
        pin_memory  = True,
    )
    test_loader = DataLoader(
        CodeVulnerabilityDataset(test_df, tokenizer, MAX_LENGTH),
        batch_size  = BATCH_SIZE,
        shuffle     = False,
        num_workers = num_workers,
        pin_memory  = True,
    )

    model = BitFitCodeT5pClassifier(MODEL_NAME, NUM_LABELS)

    if n_gpus > 1:
        model = nn.DataParallel(model)

    model = model.to(device)

    trainable_params = [p for p in model.parameters() if p.requires_grad]

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr           = LEARNING_RATE,
        betas        = (0.9, 0.999),
        eps          = 1e-8,
        weight_decay = 0.0,
    )

    total_steps  = len(train_loader) * NUM_EPOCHS
    warmup_steps = int(WARMUP_RATIO * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps   = warmup_steps,
        num_training_steps = total_steps,
    )

    scaler = GradScaler(device="cuda")

    print(
        f"\nTraining configuration"
        f"\n  Model       : {MODEL_NAME}"
        f"\n  PEFT method : BitFit (Ben-Zaken et al., ACL 2022)"
        f"\n  Epochs      : {NUM_EPOCHS}"
        f"\n  LR          : {LEARNING_RATE}"
        f"\n  Batch size  : {BATCH_SIZE}"
        f"\n  Max length  : {MAX_LENGTH}"
        f"\n  Total steps : {total_steps}"
        f"\n  Warmup steps: {warmup_steps}"
        f"\n  GPUs in use : {n_gpus}"
    )

    train_losses  = []
    val_losses    = []
    best_val_f1   = -1.0
    best_val_loss = float("inf")

    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss, tr_acc = train_epoch(
            model, train_loader, optimizer, scheduler, scaler, device
        )
        vl_loss, vl_acc, vl_f1, _, _ = evaluate_epoch(
            model, val_loader, device
        )

        train_losses.append(tr_loss)
        val_losses.append(vl_loss)

        print(
            f"Epoch {epoch:02d}/{NUM_EPOCHS}"
            f"  |  Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f}"
            f"  |  Val Loss: {vl_loss:.4f}  Acc: {vl_acc:.4f}"
            f"  F1: {vl_f1:.4f}"
        )

        if vl_f1 > best_val_f1 or (vl_f1 == best_val_f1 and vl_loss < best_val_loss):
            best_val_f1   = vl_f1
            best_val_loss = vl_loss
            state_to_save = (
                model.module.state_dict()
                if isinstance(model, nn.DataParallel)
                else model.state_dict()
            )
            torch.save(state_to_save, CHECKPOINT)
            print(
                f"           checkpoint saved  "
                f"(val_f1={best_val_f1:.4f}  val_loss={best_val_loss:.4f})"
            )

    plot_loss_curves(train_losses, val_losses, LOSS_PLOT)

    raw_model = (
        model.module if isinstance(model, nn.DataParallel) else model
    )
    raw_model.load_state_dict(
        torch.load(CHECKPOINT, map_location=device, weights_only=True),
        strict=True,
    )

    if n_gpus > 1:
        model = nn.DataParallel(raw_model)
    else:
        model = raw_model
    model = model.to(device)

    te_loss, te_acc, te_f1, te_preds, te_labels = evaluate_epoch(
        model, test_loader, device
    )

    print("\n" + "=" * 64)
    print("  FINAL TEST RESULTS  (best checkpoint by macro-F1 + val loss)")
    print("=" * 64)
    print(f"  Test Loss       : {te_loss:.4f}")
    print(f"  Test Accuracy   : {te_acc:.4f}")
    print(f"  Test F1 (macro) : {te_f1:.4f}")
    print()
    print(
        classification_report(
            te_labels,
            te_preds,
            target_names = ["Not Vulnerable", "Vulnerable"],
            digits       = 4,
        )
    )
    print("=" * 64)


if __name__ == "__main__":
    main()

# bitfit Qwen2.5-Coder-1.5B codevul 1024 token

In [ ]:

import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
)

warnings.filterwarnings("ignore")

os.environ["CUDA_VISIBLE_DEVICES"]     = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"]   = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"]  = "expandable_segments:True"


MODEL_NAME    = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
TRAIN_CSV     = "/code vul/trqw.csv"
TEST_CSV      = "/code vul/teqw.csv"

MAX_SEQ_LEN   = 128
BATCH_SIZE    = 32
NUM_EPOCHS    = 16
LEARNING_RATE = 1e-3
WARMUP_RATIO  = 0.1
VAL_SPLIT     = 0.1
NUM_LABELS    = 2
SEED          = 42
CHECKPOINT    = "best_bitfit_qwen_checkpoint.pt"
LOSS_PLOT     = "bitfit_qwen_loss_curve.png"


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


class CodeVulnerabilityDataset(Dataset):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        tokenizer,
        max_seq_len: int,
    ) -> None:
        self.texts       = dataframe["func_cleaned"].astype(str).tolist()
        self.labels      = dataframe["label"].astype(int).tolist()
        self.tokenizer   = tokenizer
        self.max_seq_len = max_seq_len

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> dict:
        encoding = self.tokenizer(
            self.texts[idx],
            max_length     = self.max_seq_len,
            padding        = "max_length",
            truncation     = True,
            return_tensors = "pt",
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }


class BitFitQwenClassifier(nn.Module):
    """
    BitFit procedure (Ben-Zaken et al. 2022, Section 3):

    Step 1 — Freeze every backbone parameter (all W weight matrices and
             layer-norm scale vectors g).
    Step 2 — Unfreeze ONLY the additive bias terms b^{(l)}_{(.)} across
             every transformer layer.  Discovered programmatically via
             name.endswith('.bias') — no hardcoded layer names — consistent
             with the paper's task-invariant property: the same set of
             parameters is modified regardless of the downstream task.
    Step 3 — Cast every trainable bias tensor from float16 to float32.
             The backbone is loaded in fp16 for memory efficiency, but
             GradScaler.unscale_() raises ValueError if it encounters fp16
             gradients.  Casting only the bias terms (and the classifier)
             to fp32 resolves this while keeping all frozen weight matrices
             in fp16.
    Step 4 — Add a task-specific linear classification head that is always
             fully trainable (paper Section 3 explicitly includes the
             classification layer).

    Decoder-only pooling:
             Qwen2.5-Coder is a causal LM with no [CLS] token.  The hidden
             state at the last non-padding token position is used as the
             sequence representation, which is the standard approach for
             decoder-only sequence classifiers.
    """

    def __init__(self, model_name: str, num_labels: int) -> None:
        super().__init__()
        self.num_labels = num_labels

        self.backbone = AutoModel.from_pretrained(
            model_name,
            trust_remote_code = True,
            torch_dtype       = torch.float16,
        )

        hidden_size = self.backbone.config.hidden_size
        self.classifier = nn.Linear(hidden_size, num_labels)
        nn.init.normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)

        self._apply_bitfit_freezing()

    def _apply_bitfit_freezing(self) -> None:
        for param in self.backbone.parameters():
            param.requires_grad = False

        unfrozen_count = 0
        for name, param in self.backbone.named_parameters():
            if name.endswith(".bias") and param is not None:
                param.data          = param.data.float()
                param.requires_grad = True
                unfrozen_count     += param.numel()

        for param in self.classifier.parameters():
            param.data          = param.data.float()
            param.requires_grad = True

        total_backbone    = sum(p.numel() for p in self.backbone.parameters())
        total_all         = sum(p.numel() for p in self.parameters())
        trainable_all     = sum(
            p.numel() for p in self.parameters() if p.requires_grad
        )
        classifier_params = sum(
            p.numel() for p in self.classifier.parameters()
        )

        print("=" * 60)
        print("BitFit Qwen2.5-Coder Parameter Summary")
        print("=" * 60)
        print(f"  Backbone total          : {total_backbone:>14,}")
        print(f"  Backbone bias (unfrozen): {unfrozen_count:>14,}")
        print(f"  Classifier head         : {classifier_params:>14,}")
        print(f"  Total parameters        : {total_all:>14,}")
        print(f"  Trainable parameters    : {trainable_all:>14,}")
        print(f"  Trainable fraction      : "
              f"{100.0 * trainable_all / total_all:>12.4f}%")
        print("=" * 60)

        if trainable_all == 0:
            raise RuntimeError(
                "No trainable parameters found after BitFit freezing. "
                "Verify that the model contains .bias tensors."
            )

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        labels:         torch.Tensor = None,
    ):
        outputs = self.backbone(
            input_ids      = input_ids,
            attention_mask = attention_mask,
        )

        hidden_states  = outputs.last_hidden_state
        last_token_idx = (attention_mask.sum(dim=1) - 1).clamp(min=0)
        batch_range    = torch.arange(
            hidden_states.size(0), device=hidden_states.device
        )
        pooled = hidden_states[batch_range, last_token_idx].float()
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return loss, logits


def train_epoch(
    model:     nn.Module,
    loader:    DataLoader,
    optimizer: torch.optim.Optimizer,
    scheduler,
    scaler:    GradScaler,
    device:    torch.device,
):
    model.train()
    total_loss = total_correct = total_samples = 0

    for batch in loader:
        input_ids      = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels         = batch["labels"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type="cuda", dtype=torch.float16):
            loss, logits = model(input_ids, attention_mask, labels)

        if loss.dim() > 0:
            loss = loss.mean()

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            max_norm=1.0,
        )

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        bs             = labels.size(0)
        total_loss    += loss.item() * bs
        preds          = logits.detach().argmax(dim=-1)
        total_correct += (preds == labels).sum().item()
        total_samples += bs

    return total_loss / total_samples, total_correct / total_samples


@torch.no_grad()
def evaluate_epoch(
    model:  nn.Module,
    loader: DataLoader,
    device: torch.device,
):
    model.eval()
    total_loss = total_correct = total_samples = 0
    all_preds  = []
    all_labels = []

    for batch in loader:
        input_ids      = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels         = batch["labels"].to(device, non_blocking=True)

        with autocast(device_type="cuda", dtype=torch.float16):
            loss, logits = model(input_ids, attention_mask, labels)

        if loss.dim() > 0:
            loss = loss.mean()

        bs             = labels.size(0)
        total_loss    += loss.item() * bs
        preds          = logits.argmax(dim=-1)
        total_correct += (preds == labels).sum().item()
        total_samples += bs
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    f1       = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, accuracy, f1, all_preds, all_labels


def plot_loss_curves(
    train_losses: list,
    val_losses:   list,
    save_path:    str,
) -> None:
    epochs = list(range(1, len(train_losses) + 1))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(epochs, train_losses, marker="o", linewidth=1.8,
            markersize=4, label="Train Loss")
    ax.plot(epochs, val_losses,   marker="s", linewidth=1.8,
            markersize=4, label="Validation Loss")
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("Cross-Entropy Loss", fontsize=12)
    ax.set_title(
        "BitFit  |  Qwen2.5-Coder-1.5B  |  Code Vulnerability Detection\n"
        "Train / Validation Loss per Epoch",
        fontsize=11,
    )
    ax.legend(fontsize=11)
    ax.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"\nLoss curve saved -> {save_path}")


def main() -> None:
    set_seed(SEED)

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is not available. At least one GPU is required."
        )

    device = torch.device("cuda")
    n_gpus = torch.cuda.device_count()
    print(f"Device: {device}  |  GPUs available: {n_gpus}")

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME, trust_remote_code=True
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "left"

    train_df_full = pd.read_csv(TRAIN_CSV)
    test_df       = pd.read_csv(TEST_CSV)

    train_df, val_df = train_test_split(
        train_df_full,
        test_size    = VAL_SPLIT,
        random_state = SEED,
        stratify     = train_df_full["label"],
    )
    train_df = train_df.reset_index(drop=True)
    val_df   = val_df.reset_index(drop=True)
    test_df  = test_df.reset_index(drop=True)

    print(
        f"Split -> Train: {len(train_df)}  |  "
        f"Val: {len(val_df)}  |  Test: {len(test_df)}"
    )

    num_workers = min(4, os.cpu_count() or 1)

    train_loader = DataLoader(
        CodeVulnerabilityDataset(train_df, tokenizer, MAX_SEQ_LEN),
        batch_size  = BATCH_SIZE,
        shuffle     = True,
        num_workers = num_workers,
        pin_memory  = True,
        drop_last   = False,
    )
    val_loader = DataLoader(
        CodeVulnerabilityDataset(val_df, tokenizer, MAX_SEQ_LEN),
        batch_size  = BATCH_SIZE,
        shuffle     = False,
        num_workers = num_workers,
        pin_memory  = True,
    )
    test_loader = DataLoader(
        CodeVulnerabilityDataset(test_df, tokenizer, MAX_SEQ_LEN),
        batch_size  = BATCH_SIZE,
        shuffle     = False,
        num_workers = num_workers,
        pin_memory  = True,
    )

    model = BitFitQwenClassifier(MODEL_NAME, NUM_LABELS)

    if n_gpus > 1:
        model = nn.DataParallel(model)

    model = model.to(device)

    trainable_params = [p for p in model.parameters() if p.requires_grad]

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr           = LEARNING_RATE,
        betas        = (0.9, 0.999),
        eps          = 1e-8,
        weight_decay = 0.0,
    )

    total_steps  = len(train_loader) * NUM_EPOCHS
    warmup_steps = int(WARMUP_RATIO * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps   = warmup_steps,
        num_training_steps = total_steps,
    )

    scaler = GradScaler(device="cuda")

    print(
        f"\nTraining config"
        f"\n  epochs={NUM_EPOCHS}  lr={LEARNING_RATE}  "
        f"batch={BATCH_SIZE}  max_len={MAX_SEQ_LEN}"
        f"\n  total_steps={total_steps}  warmup_steps={warmup_steps}"
    )

    train_losses  = []
    val_losses    = []
    best_val_loss = float("inf")

    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss, tr_acc = train_epoch(
            model, train_loader, optimizer, scheduler, scaler, device
        )
        vl_loss, vl_acc, vl_f1, _, _ = evaluate_epoch(
            model, val_loader, device
        )

        train_losses.append(tr_loss)
        val_losses.append(vl_loss)

        print(
            f"Epoch {epoch:02d}/{NUM_EPOCHS}"
            f"  |  Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f}"
            f"  |  Val Loss: {vl_loss:.4f}  Acc: {vl_acc:.4f}"
            f"  F1: {vl_f1:.4f}"
        )

        if vl_loss < best_val_loss:
            best_val_loss = vl_loss
            state_to_save = (
                model.module.state_dict()
                if isinstance(model, nn.DataParallel)
                else model.state_dict()
            )
            torch.save(state_to_save, CHECKPOINT)
            print(
                f"           checkpoint saved  "
                f"(val_loss={best_val_loss:.4f})"
            )

    plot_loss_curves(train_losses, val_losses, LOSS_PLOT)

    raw_model = (
        model.module if isinstance(model, nn.DataParallel) else model
    )
    raw_model.load_state_dict(
        torch.load(CHECKPOINT, map_location=device, weights_only=True),
        strict=True,
    )

    if n_gpus > 1:
        model = nn.DataParallel(raw_model)
    else:
        model = raw_model
    model = model.to(device)

    te_loss, te_acc, te_f1, te_preds, te_labels = evaluate_epoch(
        model, test_loader, device
    )

    print("\n" + "=" * 60)
    print("FINAL TEST RESULTS  (best checkpoint by validation loss)")
    print("=" * 60)
    print(f"  Test Loss       : {te_loss:.4f}")
    print(f"  Test Accuracy   : {te_acc:.4f}")
    print(f"  Test F1 (macro) : {te_f1:.4f}")
    print()
    print(
        classification_report(
            te_labels,
            te_preds,
            target_names = ["Not Vulnerable", "Vulnerable"],
            digits       = 4,
        )
    )
    print("=" * 60)


if __name__ == "__main__":
    main()

# BitFit codequen 3b, 1024 token

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
warnings.filterwarnings('ignore')
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"
TRAIN_CSV                   = "code vul/trqw.csv"
TEST_CSV                    = "code vul/teqw.csv"
MAX_SEQ_LEN = 1024
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8
NUM_EPOCHS = 16
LEARNING_RATE = 1e-3
WARMUP_RATIO = 0.1
VAL_SPLIT = 0.1
NUM_LABELS = 2
SEED = 42
CHECKPOINT = "best_bitfit_qwen3b_checkpoint.pt"
LOSS_PLOT = "bitfit_qwen3b_loss_curve.png"
INPUT_DEVICE = torch.device("cuda:0")
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
class CodeVulnerabilityDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_seq_len):
        self.texts = dataframe["func_cleaned"].astype(str).tolist()
        self.labels = dataframe["label"].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_seq_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }
class BitFitQwenClassifier(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.num_labels = num_labels
        self.backbone = AutoModel.from_pretrained(
            model_name,
            trust_remote_code=True,
            torch_dtype=torch.float16,
            device_map="auto",
        )
        self.backbone.config.use_cache = False
        self.backbone.gradient_checkpointing_enable()
        hidden_size = self.backbone.config.hidden_size
        self.classifier = nn.Linear(hidden_size, num_labels).to(INPUT_DEVICE)
        nn.init.normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)
        self._apply_bitfit_freezing()
    def _apply_bitfit_freezing(self):
        for param in self.backbone.parameters():
            param.requires_grad = False
        unfrozen_count = 0
        for name, param in self.backbone.named_parameters():
            if name.endswith(".bias") and param is not None:
                param.data = param.data.float()
                param.requires_grad = True
                unfrozen_count += param.numel()
        for param in self.classifier.parameters():
            param.data = param.data.float()
            param.requires_grad = True
        total_backbone = sum(p.numel() for p in self.backbone.parameters())
        total_all = sum(p.numel() for p in self.parameters())
        trainable_all = sum(p.numel() for p in self.parameters() if p.requires_grad)
        classifier_params = sum(p.numel() for p in self.classifier.parameters())
        print("=" * 60)
        print("BitFit Qwen2.5-Coder-3B Parameter Summary")
        print("=" * 60)
        print(f"  Backbone total          : {total_backbone:>14,}")
        print(f"  Backbone bias (unfrozen): {unfrozen_count:>14,}")
        print(f"  Classifier head         : {classifier_params:>14,}")
        print(f"  Total parameters        : {total_all:>14,}")
        print(f"  Trainable parameters    : {trainable_all:>14,}")
        print(f"  Trainable fraction      : {100.0 * trainable_all / total_all:>12.4f}%")
        print("=" * 60)
        if trainable_all == 0:
            raise RuntimeError("No trainable parameters found after BitFit freezing.")
    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state
        last_token_idx = (attention_mask.sum(dim=1) - 1).clamp(min=0)
        batch_range = torch.arange(hidden_states.size(0), device=hidden_states.device)
        pooled = hidden_states[batch_range, last_token_idx].float().to(INPUT_DEVICE)
        logits = self.classifier(pooled)
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels.to(INPUT_DEVICE))
        return loss, logits
def free_gpu_cache():
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    import gc
    gc.collect()
def train_epoch(model, loader, optimizer, scheduler, scaler, grad_accum_steps):
    model.train()
    total_loss = total_correct = total_samples = 0
    optimizer.zero_grad(set_to_none=True)
    for step, batch in enumerate(loader):
        input_ids = batch["input_ids"].to(INPUT_DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(INPUT_DEVICE, non_blocking=True)
        labels = batch["labels"].to(INPUT_DEVICE, non_blocking=True)
        with autocast(device_type="cuda", dtype=torch.float16):
            loss, logits = model(input_ids, attention_mask, labels)
        if loss.dim() > 0:
            loss = loss.mean()
        scaled_loss = loss / grad_accum_steps
        scaler.scale(scaled_loss).backward()
        bs = labels.size(0)
        total_loss += loss.item() * bs
        preds = logits.detach().argmax(dim=-1)
        total_correct += (preds == labels).sum().item()
        total_samples += bs
        if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            free_gpu_cache()
    return total_loss / total_samples, total_correct / total_samples
@torch.no_grad()
def evaluate_epoch(model, loader):
    model.eval()
    total_loss = total_correct = total_samples = 0
    all_preds = []
    all_labels = []
    for batch in loader:
        input_ids = batch["input_ids"].to(INPUT_DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(INPUT_DEVICE, non_blocking=True)
        labels = batch["labels"].to(INPUT_DEVICE, non_blocking=True)
        with autocast(device_type="cuda", dtype=torch.float16):
            loss, logits = model(input_ids, attention_mask, labels)
        if loss.dim() > 0:
            loss = loss.mean()
        bs = labels.size(0)
        total_loss += loss.item() * bs
        preds = logits.argmax(dim=-1)
        total_correct += (preds == labels).sum().item()
        total_samples += bs
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
    free_gpu_cache()
    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, accuracy, f1, all_preds, all_labels
def plot_loss_curves(train_losses, val_losses, save_path):
    epochs = list(range(1, len(train_losses) + 1))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(epochs, train_losses, marker="o", linewidth=1.8, markersize=4, label="Train Loss")
    ax.plot(epochs, val_losses, marker="s", linewidth=1.8, markersize=4, label="Validation Loss")
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("Cross-Entropy Loss", fontsize=12)
    ax.set_title("BitFit  |  Qwen2.5-Coder-3B  |  Code Vulnerability Detection\nTrain / Validation Loss per Epoch", fontsize=11)
    ax.legend(fontsize=11)
    ax.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"\nLoss curve saved -> {save_path}")
def main():
    set_seed(SEED)
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available. At least one GPU is required.")
    n_gpus = torch.cuda.device_count()
    print(f"GPUs available: {n_gpus}  |  Strategy: device_map=auto (pipeline across all GPUs)")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "left"
    train_df_full = pd.read_csv(TRAIN_CSV)
    test_df = pd.read_csv(TEST_CSV)
    train_df, val_df = train_test_split(
        train_df_full,
        test_size=VAL_SPLIT,
        random_state=SEED,
        stratify=train_df_full["label"],
    )
    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    print(f"Split -> Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}")
    num_workers = min(4, os.cpu_count() or 1)
    train_loader = DataLoader(
        CodeVulnerabilityDataset(train_df, tokenizer, MAX_SEQ_LEN),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers, pin_memory=False, drop_last=False,
    )
    val_loader = DataLoader(
        CodeVulnerabilityDataset(val_df, tokenizer, MAX_SEQ_LEN),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=False,
    )
    test_loader = DataLoader(
        CodeVulnerabilityDataset(test_df, tokenizer, MAX_SEQ_LEN),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=False,
    )
    model = BitFitQwenClassifier(MODEL_NAME, NUM_LABELS)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        trainable_params, lr=LEARNING_RATE, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0,
    )
    effective_steps_per_epoch = (len(train_loader) + GRAD_ACCUM_STEPS - 1) // GRAD_ACCUM_STEPS
    total_steps = effective_steps_per_epoch * NUM_EPOCHS
    warmup_steps = int(WARMUP_RATIO * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps,
    )
    scaler = GradScaler(device="cuda")
    print(f"\nTraining config\n  epochs={NUM_EPOCHS}  lr={LEARNING_RATE}  batch={BATCH_SIZE}  grad_accum={GRAD_ACCUM_STEPS}  effective_batch={BATCH_SIZE * GRAD_ACCUM_STEPS}  max_len={MAX_SEQ_LEN}\n  total_steps={total_steps}  warmup_steps={warmup_steps}")
    train_losses = []
    val_losses = []
    best_val_loss = float("inf")
    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, scheduler, scaler, GRAD_ACCUM_STEPS)
        vl_loss, vl_acc, vl_f1, _, _ = evaluate_epoch(model, val_loader)
        train_losses.append(tr_loss)
        val_losses.append(vl_loss)
        print(f"Epoch {epoch:02d}/{NUM_EPOCHS}  |  Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f}  |  Val Loss: {vl_loss:.4f}  Acc: {vl_acc:.4f}  F1: {vl_f1:.4f}")
        if vl_loss < best_val_loss:
            best_val_loss = vl_loss
            trainable_state = {k: v for k, v in model.state_dict().items() if any(k == n for n, p in model.named_parameters() if p.requires_grad)}
            torch.save(trainable_state, CHECKPOINT)
            print(f"           checkpoint saved  (val_loss={best_val_loss:.4f})")
        free_gpu_cache()
    plot_loss_curves(train_losses, val_losses, LOSS_PLOT)
    saved_state = torch.load(CHECKPOINT, map_location="cpu", weights_only=True)
    current_state = model.state_dict()
    current_state.update(saved_state)
    model.load_state_dict(current_state, strict=True)
    te_loss, te_acc, te_f1, te_preds, te_labels = evaluate_epoch(model, test_loader)
    print("\n" + "=" * 60)
    print("FINAL TEST RESULTS  (best checkpoint by validation loss)")
    print("=" * 60)
    print(f"  Test Loss       : {te_loss:.4f}")
    print(f"  Test Accuracy   : {te_acc:.4f}")
    print(f"  Test F1 (macro) : {te_f1:.4f}")
    print()
    print(classification_report(te_labels, te_preds, target_names=["Not Vulnerable", "Vulnerable"], digits=4))
    print("=" * 60)
if __name__ == "__main__":
    main()

# adapter codet5+ 770M, 1024 token

In [ ]:
import os
import gc
import warnings
import random
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix,
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

MODEL_NAME                  = "Salesforce/codet5p-770m"
TRAIN_CSV                   = '/trqw.csv'
TEST_CSV                    = '/teqw.csv'
MAX_LENGTH                  = 1024
SEED                        = 42
BATCH_SIZE                  = 2
GRADIENT_ACCUMULATION_STEPS = 16
NUM_EPOCHS                  = 10
LEARNING_RATE               = 1e-4
WARMUP_RATIO                = 0.1
WEIGHT_DECAY                = 0.01
MAX_GRAD_NORM               = 1.0
ADAPTER_REDUCTION_FACTOR    = 16
NUM_LABELS                  = 2
CLASSIFIER_DROPOUT          = 0.1
NON_LINEARITY               = "gelu"
ADAPTER_INIT_SCALE          = 1e-3
SAVE_DIR                    = "./adapter_tuning_codet5p_checkpoints"

if torch.cuda.is_available():
    DEVICE         = torch.device("cuda:0")
    NUM_WORKERS    = 4
    PIN_MEMORY     = True
    BACKBONE_DTYPE = torch.bfloat16
elif torch.backends.mps.is_available():
    DEVICE         = torch.device("mps")
    NUM_WORKERS    = 0
    PIN_MEMORY     = False
    BACKBONE_DTYPE = torch.float32
else:
    DEVICE         = torch.device("cpu")
    NUM_WORKERS    = 0
    PIN_MEMORY     = False
    BACKBONE_DTYPE = torch.float32

os.makedirs(SAVE_DIR, exist_ok=True)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


set_seed(SEED)


def free_gpu_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()


def print_gpu_memory():
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            alloc  = torch.cuda.memory_allocated(i) / 1024 ** 3
            reserv = torch.cuda.memory_reserved(i)  / 1024 ** 3
            print(f"  GPU {i}: allocated={alloc:.2f}GB  reserved={reserv:.2f}GB")
    else:
        print(f"  Running on: {DEVICE}")


print(f"Device: {DEVICE}")
print("Initial state:")
print_gpu_memory()


def _get_hidden_size(config) -> int:
    if hasattr(config, "hidden_size"):
        return config.hidden_size
    if hasattr(config, "d_model"):
        return config.d_model
    raise AttributeError("Cannot determine hidden_size from model config.")


def _get_num_encoder_layers(config) -> int:
    if hasattr(config, "num_layers"):
        return config.num_layers
    if hasattr(config, "num_hidden_layers"):
        return config.num_hidden_layers
    raise AttributeError("Cannot determine number of encoder layers from model config.")


class HoulsbyBottleneckAdapter(nn.Module):


    def __init__(self, hidden_size: int, reduction_factor: int, non_linearity: str):
        super().__init__()
        self.hidden_size     = hidden_size
        self.bottleneck_size = hidden_size // reduction_factor

        self.down_proj = nn.Linear(hidden_size, self.bottleneck_size)
        self.up_proj   = nn.Linear(self.bottleneck_size, hidden_size)

        if non_linearity == "gelu":
            self.activation = nn.GELU()
        elif non_linearity == "relu":
            self.activation = nn.ReLU()
        elif non_linearity == "swish":
            self.activation = nn.SiLU()
        else:
            raise ValueError(f"Unsupported non-linearity: {non_linearity}")

        nn.init.normal_(self.down_proj.weight, std=ADAPTER_INIT_SCALE)
        nn.init.zeros_(self.down_proj.bias)
        nn.init.normal_(self.up_proj.weight, std=ADAPTER_INIT_SCALE)
        nn.init.zeros_(self.up_proj.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual   = x
        orig_dtype = x.dtype
        x = self.down_proj(x.float())
        x = self.activation(x)
        x = self.up_proj(x)
        return x.to(orig_dtype) + residual


def inject_houlsby_adapters(
    backbone: AutoModelForSeq2SeqLM,
    reduction_factor: int,
    non_linearity: str,
) -> None:

    hidden_size = _get_hidden_size(backbone.config)

    for block in backbone.encoder.block:
        attn_adapter = HoulsbyBottleneckAdapter(hidden_size, reduction_factor, non_linearity)
        ff_adapter   = HoulsbyBottleneckAdapter(hidden_size, reduction_factor, non_linearity)

        block.layer[0].add_module("adapter", attn_adapter)
        block.layer[1].add_module("adapter", ff_adapter)

        def _make_hook(adp: HoulsbyBottleneckAdapter):
            def hook(module, inp, output):
                h = output[0] if isinstance(output, tuple) else output
                target_device = h.device
                if next(adp.parameters()).device != target_device:
                    adp.to(target_device)
                adapted = adp(h)
                if isinstance(output, tuple):
                    return (adapted,) + output[1:]
                return adapted
            return hook

        block.layer[0].register_forward_hook(_make_hook(attn_adapter))
        block.layer[1].register_forward_hook(_make_hook(ff_adapter))


def freeze_backbone_unfreeze_adapters_and_norms(model: nn.Module) -> None:

    for param in model.parameters():
        param.requires_grad = False

    for name, param in model.named_parameters():
        if "adapter" in name or "layer_norm" in name or "classifier" in name:
            param.requires_grad = True


def count_parameters(model: nn.Module):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen    = total - trainable
    pct       = 100.0 * trainable / total if total > 0 else 0.0
    return total, trainable, frozen, pct


class VulnerabilityDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, tokenizer, max_length: int):
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.texts      = dataframe["func_cleaned"].astype(str).tolist()
        self.labels     = dataframe["label"].astype(int).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }


class HoulsbyAdapterVulnerabilityClassifier(nn.Module):
 

    def __init__(
        self,
        model_name: str,
        num_labels: int,
        reduction_factor: int,
        non_linearity: str,
        classifier_dropout: float,
    ):
        super().__init__()

        self.backbone = AutoModelForSeq2SeqLM.from_pretrained(
            model_name,
            torch_dtype=BACKBONE_DTYPE,
            trust_remote_code=True,
        )
        self.backbone.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
        self.backbone.config.use_cache = False

        inject_houlsby_adapters(self.backbone, reduction_factor, non_linearity)

        hidden_size = _get_hidden_size(self.backbone.config)

        self.classifier = nn.Sequential(
            nn.Dropout(classifier_dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(classifier_dropout),
            nn.Linear(hidden_size // 2, num_labels),
        )

        self._init_classifier()
        freeze_backbone_unfreeze_adapters_and_norms(self)
        self.to(DEVICE)

    def _init_classifier(self):
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    @staticmethod
    def mean_pool(
        last_hidden_state: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        mask   = attention_mask.unsqueeze(-1).to(last_hidden_state.device).float()
        summed = (last_hidden_state.float() * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        labels: torch.Tensor = None,
        class_weights: torch.Tensor = None,
    ):
        encoder_outputs = self.backbone.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        pooled = self.mean_pool(encoder_outputs.last_hidden_state, attention_mask)
        logits = self.classifier(pooled.to(DEVICE))

        loss = None
        if labels is not None:
            labels = labels.to(logits.device)
            cw     = class_weights.to(logits.device) if class_weights is not None else None
            loss   = nn.CrossEntropyLoss(weight=cw)(logits, labels)

        return loss, logits


def load_data(train_csv: str, test_csv: str):
    train_df = pd.read_csv(train_csv)
    test_df  = pd.read_csv(test_csv)

    required = {"func_cleaned", "label"}
    assert required.issubset(set(train_df.columns)), \
        f"Train CSV must contain {required}. Found: {set(train_df.columns)}"
    assert required.issubset(set(test_df.columns)), \
        f"Test CSV must contain {required}. Found: {set(test_df.columns)}"

    train_df = train_df.dropna(subset=list(required)).reset_index(drop=True)
    test_df  = test_df.dropna(subset=list(required)).reset_index(drop=True)
    train_df["label"] = train_df["label"].astype(int)
    test_df["label"]  = test_df["label"].astype(int)

    print(f"Train samples : {len(train_df)}")
    print(f"Test  samples : {len(test_df)}")
    print(f"Train label dist:\n{train_df['label'].value_counts()}")
    print(f"Test  label dist:\n{test_df['label'].value_counts()}")
    return train_df, test_df


def compute_class_weights_tensor(labels: list, num_classes: int) -> torch.Tensor:
    classes = np.arange(num_classes)
    weights = compute_class_weight("balanced", classes=classes, y=np.array(labels))
    return torch.tensor(weights, dtype=torch.float32)


def train_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    optimizer: AdamW,
    scheduler,
    class_weights: torch.Tensor,
    epoch: int,
    gradient_accumulation_steps: int,
):
    model.train()
    total_loss                       = 0.0
    all_preds, all_labels, all_probs = [], [], []
    optimizer.zero_grad()

    for step, batch in enumerate(dataloader):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"]

        loss, logits = model(input_ids, attention_mask, labels, class_weights)

        scaled_loss = loss / gradient_accumulation_steps
        scaled_loss.backward()

        if (step + 1) % gradient_accumulation_steps == 0 or (step + 1) == len(dataloader):
            nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                MAX_GRAD_NORM,
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item()

        with torch.no_grad():
            probs = F.softmax(logits.float(), dim=-1)
            preds = torch.argmax(probs, dim=-1)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(batch["labels"].tolist())
        all_probs.extend(probs[:, 1].cpu().tolist())

        if (step + 1) % 50 == 0:
            print(
                f"  Epoch {epoch} | Step {step+1}/{len(dataloader)} | "
                f"Avg Loss: {total_loss/(step+1):.4f}"
            )

        if (step + 1) % 100 == 0:
            free_gpu_cache()

    return total_loss / len(dataloader), compute_metrics(all_labels, all_preds, all_probs)


@torch.no_grad()
def evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    class_weights: torch.Tensor,
):
    model.eval()
    total_loss                       = 0.0
    all_preds, all_labels, all_probs = [], [], []

    for batch in dataloader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"]

        loss, logits = model(input_ids, attention_mask, labels, class_weights)
        total_loss  += loss.item()

        probs = F.softmax(logits.float(), dim=-1)
        preds = torch.argmax(probs, dim=-1)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(batch["labels"].tolist())
        all_probs.extend(probs[:, 1].cpu().tolist())

    return total_loss / len(dataloader), compute_metrics(all_labels, all_preds, all_probs)


def compute_metrics(labels, preds, probs):
    acc    = accuracy_score(labels, preds)
    prec   = precision_score(labels, preds, average="binary", zero_division=0)
    rec    = recall_score(labels, preds, average="binary", zero_division=0)
    f1     = f1_score(labels, preds, average="binary", zero_division=0)
    f1_mac = f1_score(labels, preds, average="macro",  zero_division=0)
    mcc    = matthews_corrcoef(labels, preds)
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = 0.0
    cm = confusion_matrix(labels, preds)
    if cm.size == 4:
        tn, fp, fn, tp = cm.ravel()
    else:
        tn = fp = fn = 0
        tp = int(cm[0, 0])
    fpr = float(fp) / (float(fp) + float(tn) + 1e-9)
    return {
        "accuracy":  float(acc),
        "precision": float(prec),
        "recall":    float(rec),
        "f1":        float(f1),
        "f1_macro":  float(f1_mac),
        "mcc":       float(mcc),
        "auc_roc":   float(auc),
        "fpr":       float(fpr),
        "tp":        int(tp),
        "tn":        int(tn),
        "fp":        int(fp),
        "fn":        int(fn),
    }


def print_metrics(split: str, loss: float, m: dict, epoch: int = None):
    hdr = f"[{split}]" if epoch is None else f"[Epoch {epoch} | {split}]"
    print(f"\n{hdr}")
    print(f"  Loss      : {loss:.4f}")
    print(f"  Accuracy  : {m['accuracy']*100:.2f}%")
    print(f"  Precision : {m['precision']*100:.2f}%")
    print(f"  Recall    : {m['recall']*100:.2f}%")
    print(f"  F1 (Bin)  : {m['f1']*100:.2f}%")
    print(f"  F1 (Mac)  : {m['f1_macro']*100:.2f}%")
    print(f"  MCC       : {m['mcc']:.4f}")
    print(f"  AUC-ROC   : {m['auc_roc']:.4f}")
    print(f"  FPR       : {m['fpr']*100:.2f}%")
    print(f"  TP={m['tp']} | TN={m['tn']} | FP={m['fp']} | FN={m['fn']}")


def print_parameter_summary(model: nn.Module):
    total, trainable, frozen, pct = count_parameters(model)
    print("\n" + "=" * 68)
    print("HOULSBY ADAPTER-TUNING  —  PARAMETER SUMMARY")
    print("=" * 68)
    print(f"  Total parameters                       : {total:>15,}")
    print(f"  Trainable (adapters + LayerNorm + head): {trainable:>15,}  ({pct:.4f}%)")
    print(f"  Frozen    (CodeT5+ backbone)           : {frozen:>15,}")
    print("=" * 68)
    print("\nTrainable parameter groups:")
    for name, param in model.named_parameters():
        if param.requires_grad:
            print(f"  {name:<80s} | {param.numel():>10,}")
    print("=" * 68 + "\n")


def main():
    hidden_size = 1024
    num_enc_layers = 12
    bottleneck_dim = hidden_size // ADAPTER_REDUCTION_FACTOR

    print("=" * 68)
    print("ADAPTER-TUNING  (Houlsby et al., ICML 2019)")
    print("Parameter-Efficient Transfer Learning for NLP")
    print("Task      : Code Vulnerability Detection (Binary)")
    print(f"Backbone  : {MODEL_NAME}")
    print(f"Arch      : CodeT5+ encoder-decoder | {num_enc_layers} enc layers | hidden={hidden_size}")
    print(f"Reduction : {ADAPTER_REDUCTION_FACTOR}x  → bottleneck m={bottleneck_dim}")
    print(f"Adapters  : 2 per encoder layer × {num_enc_layers} = {num_enc_layers * 2} total")
    print(f"Backbone dtype           : {BACKBONE_DTYPE}")
    print(f"Device                   : {DEVICE}")
    print(f"Gradient checkpointing   : ENABLED")
    print(f"MAX_LENGTH               : {MAX_LENGTH}")
    print(f"Effective batch size     : {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
    print("=" * 68 + "\n")

    train_df, test_df = load_data(TRAIN_CSV, TEST_CSV)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "left"

    train_dataset = VulnerabilityDataset(train_df, tokenizer, MAX_LENGTH)
    test_dataset  = VulnerabilityDataset(test_df,  tokenizer, MAX_LENGTH)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    train_labels  = train_df["label"].astype(int).tolist()
    class_weights = compute_class_weights_tensor(train_labels, NUM_LABELS)
    print(f"Class weights (balanced): {[round(w, 4) for w in class_weights.tolist()]}\n")

    print("Loading CodeT5+ backbone and injecting Houlsby adapters ...")
    free_gpu_cache()

    model = HoulsbyAdapterVulnerabilityClassifier(
        model_name=MODEL_NAME,
        num_labels=NUM_LABELS,
        reduction_factor=ADAPTER_REDUCTION_FACTOR,
        non_linearity=NON_LINEARITY,
        classifier_dropout=CLASSIFIER_DROPOUT,
    )

    hidden_size    = _get_hidden_size(model.backbone.config)
    num_enc_layers = _get_num_encoder_layers(model.backbone.config)
    bottleneck_dim = hidden_size // ADAPTER_REDUCTION_FACTOR

    print("Adapter injection complete.\n")
    print("State after model load:")
    print_gpu_memory()

    print_parameter_summary(model)
    free_gpu_cache()

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(
        trainable_params,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        betas=(0.9, 0.999),
        eps=1e-8,
    )

    total_steps  = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS) * NUM_EPOCHS
    warmup_steps = int(WARMUP_RATIO * total_steps)
    print(f"Total optimiser steps : {total_steps}")
    print(f"Warmup steps          : {warmup_steps}\n")

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    best_f1    = 0.0
    best_epoch = 0
    history    = []

    print("=" * 68)
    print("TRAINING")
    print("=" * 68)

    for epoch in range(1, NUM_EPOCHS + 1):
        free_gpu_cache()
        print(f"\nState at start of epoch {epoch}:")
        print_gpu_memory()

        train_loss, train_metrics = train_epoch(
            model=model,
            dataloader=train_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            class_weights=class_weights,
            epoch=epoch,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        )
        print_metrics("TRAIN", train_loss, train_metrics, epoch)

        free_gpu_cache()

        test_loss, test_metrics = evaluate(
            model=model,
            dataloader=test_loader,
            class_weights=class_weights,
        )
        print_metrics("TEST", test_loss, test_metrics, epoch)

        history.append({
            "epoch":      epoch,
            "train_loss": train_loss,
            "test_loss":  test_loss,
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"test_{k}":  v for k, v in test_metrics.items()},
        })

        if test_metrics["f1"] > best_f1:
            best_f1    = test_metrics["f1"]
            best_epoch = epoch

            trainable_state = {
                k: v for k, v in model.state_dict().items()
                if any(tag in k for tag in ("adapter", "layer_norm", "classifier"))
            }
            torch.save(
                {
                    "epoch":                epoch,
                    "trainable_state_dict": trainable_state,
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "best_f1":              best_f1,
                    "test_metrics":         test_metrics,
                    "config": {
                        "model_name":       MODEL_NAME,
                        "reduction_factor": ADAPTER_REDUCTION_FACTOR,
                        "non_linearity":    NON_LINEARITY,
                        "max_length":       MAX_LENGTH,
                        "num_labels":       NUM_LABELS,
                    },
                },
                os.path.join(SAVE_DIR, "best_adapter_model.pt"),
            )
            print(f"  ✓ Best checkpoint saved  (epoch {epoch} | F1={best_f1*100:.2f}%)")

    pd.DataFrame(history).to_csv(
        os.path.join(SAVE_DIR, "training_history.csv"), index=False
    )

    print("\n" + "=" * 68)
    print("FINAL EVALUATION — BEST CHECKPOINT")
    print("=" * 68)

    free_gpu_cache()

    ckpt = torch.load(
        os.path.join(SAVE_DIR, "best_adapter_model.pt"),
        map_location="cpu",
    )
    model.load_state_dict(ckpt["trainable_state_dict"], strict=False)

    final_loss, final_metrics = evaluate(
        model=model,
        dataloader=test_loader,
        class_weights=class_weights,
    )
    print_metrics("FINAL TEST", final_loss, final_metrics)

    _, trainable, _, pct = count_parameters(model)

    print("\n" + "=" * 68)
    print("HOULSBY ADAPTER-TUNING — COMPLETE RESULTS SUMMARY")
    print("=" * 68)
    print(f"  Model               : {MODEL_NAME}")
    print(f"  Best Epoch          : {best_epoch} / {NUM_EPOCHS}")
    print(f"  Accuracy            : {final_metrics['accuracy']*100:.2f}%")
    print(f"  Precision           : {final_metrics['precision']*100:.2f}%")
    print(f"  Recall              : {final_metrics['recall']*100:.2f}%")
    print(f"  F1 (Binary)         : {final_metrics['f1']*100:.2f}%")
    print(f"  F1 (Macro)          : {final_metrics['f1_macro']*100:.2f}%")
    print(f"  MCC                 : {final_metrics['mcc']:.4f}")
    print(f"  AUC-ROC             : {final_metrics['auc_roc']:.4f}")
    print(f"  FPR                 : {final_metrics['fpr']*100:.2f}%")
    print(f"  TP={final_metrics['tp']} | TN={final_metrics['tn']} | "
          f"FP={final_metrics['fp']} | FN={final_metrics['fn']}")
    print(f"\n  Encoder layers      : {num_enc_layers}")
    print(f"  Adapters injected   : {num_enc_layers * 2}  (2 per layer × {num_enc_layers})")
    print(f"  d_model             : {hidden_size}")
    print(f"  Bottleneck dim (m)  : {bottleneck_dim}  (d={hidden_size} / {ADAPTER_REDUCTION_FACTOR})")
    print(f"  Trainable params    : {trainable:,}  ({pct:.4f}% of total)")
    print(f"  Backbone dtype      : {BACKBONE_DTYPE}")
    print(f"  Gradient ckpt       : ENABLED")
    print("=" * 68)

    results_df = pd.DataFrame([{
        "method":            "Adapter-Tuning (Houlsby et al., ICML 2019)",
        "backbone":          MODEL_NAME,
        "reduction_factor":  ADAPTER_REDUCTION_FACTOR,
        "non_linearity":     NON_LINEARITY,
        "bottleneck_dim":    bottleneck_dim,
        "num_adapter_pairs": num_enc_layers,
        "total_adapters":    num_enc_layers * 2,
        "backbone_dtype":    str(BACKBONE_DTYPE),
        "grad_checkpointing": True,
        "max_length":        MAX_LENGTH,
        "effective_batch":   BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
        "trainable_params":  trainable,
        "trainable_pct":     round(pct, 6),
        "best_epoch":        best_epoch,
        **{f"final_{k}": round(v, 6) if isinstance(v, float) else v
           for k, v in final_metrics.items()},
    }])
    results_df.to_csv(os.path.join(SAVE_DIR, "final_results.csv"), index=False)

    print(f"\nArtifacts saved to : {SAVE_DIR}/")
    print(f"  final_results.csv")
    print(f"  training_history.csv")
    print(f"  best_adapter_model.pt")


if __name__ == "__main__":
    main()

# adapter codet5+ 220m 1024 token

In [ ]:
import os
import gc
import warnings
import random
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, matthews_corrcoef, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
warnings.filterwarnings("ignore")
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
MODEL_NAME = "Salesforce/codet5p-220m"
TRAIN_CSV = "code vul/trqw.csv"
TEST_CSV = "code vul/teqw.csv"
MAX_LENGTH = 1024
SEED = 42
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 16
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
ADAPTER_REDUCTION_FACTOR = 16
NUM_LABELS = 2
CLASSIFIER_DROPOUT = 0.1
NON_LINEARITY = "gelu"
ADAPTER_INIT_SCALE = 1e-3
BACKBONE_DTYPE = torch.bfloat16
HEAD_DEVICE = "cuda:0"
SAVE_DIR = "./adapter_tuning_codet5p220m_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(SEED)
def free_gpu_cache():
    gc.collect()
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
def print_gpu_memory():
    for i in range(torch.cuda.device_count()):
        alloc = torch.cuda.memory_allocated(i) / 1024 ** 3
        reserv = torch.cuda.memory_reserved(i) / 1024 ** 3
        print(f"  GPU {i}: allocated={alloc:.2f}GB  reserved={reserv:.2f}GB")
print(f"GPUs visible: {torch.cuda.device_count()}")
print("Initial GPU state:")
print_gpu_memory()
class HoulsbyBottleneckAdapter(nn.Module):
    def __init__(self, hidden_size, reduction_factor, non_linearity):
        super().__init__()
        self.hidden_size = hidden_size
        self.bottleneck_size = hidden_size // reduction_factor
        self.down_proj = nn.Linear(hidden_size, self.bottleneck_size)
        self.up_proj = nn.Linear(self.bottleneck_size, hidden_size)
        if non_linearity == "gelu":
            self.activation = nn.GELU()
        elif non_linearity == "relu":
            self.activation = nn.ReLU()
        elif non_linearity == "swish":
            self.activation = nn.SiLU()
        else:
            raise ValueError(f"Unsupported non-linearity: {non_linearity}")
        nn.init.normal_(self.down_proj.weight, std=ADAPTER_INIT_SCALE)
        nn.init.zeros_(self.down_proj.bias)
        nn.init.normal_(self.up_proj.weight, std=ADAPTER_INIT_SCALE)
        nn.init.zeros_(self.up_proj.bias)
    def forward(self, x):
        residual = x
        orig_dtype = x.dtype
        x = self.down_proj(x.float())
        x = self.activation(x)
        x = self.up_proj(x)
        return x.to(orig_dtype) + residual
def inject_houlsby_adapters(backbone, reduction_factor, non_linearity):
    hidden_size = backbone.config.hidden_size
    for layer in backbone.encoder.block:
        attn_adapter = HoulsbyBottleneckAdapter(hidden_size, reduction_factor, non_linearity)
        ff_adapter = HoulsbyBottleneckAdapter(hidden_size, reduction_factor, non_linearity)
        layer.layer[0].add_module("adapter", attn_adapter)
        layer.layer[1].add_module("adapter", ff_adapter)
        def _make_hook(adp):
            def hook(module, inp, output):
                h = output[0] if isinstance(output, tuple) else output
                target_device = h.device
                if next(adp.parameters()).device != target_device:
                    adp.to(target_device)
                adapted = adp(h)
                if isinstance(output, tuple):
                    return (adapted,) + output[1:]
                return adapted
            return hook
        layer.layer[0].register_forward_hook(_make_hook(attn_adapter))
        layer.layer[1].register_forward_hook(_make_hook(ff_adapter))
def freeze_backbone_unfreeze_adapters_and_norms(model):
    for param in model.parameters():
        param.requires_grad = False
    for name, param in model.named_parameters():
        if "adapter" in name or "norm" in name or "classifier" in name:
            param.requires_grad = True
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - trainable
    pct = 100.0 * trainable / total if total > 0 else 0.0
    return total, trainable, frozen, pct
class VulnerabilityDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.texts = dataframe["func_cleaned"].astype(str).tolist()
        self.labels = dataframe["label"].astype(int).tolist()
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0), "labels": torch.tensor(self.labels[idx], dtype=torch.long)}
class HoulsbyAdapterVulnerabilityClassifier(nn.Module):
    def __init__(self, model_name, num_labels, reduction_factor, non_linearity, classifier_dropout):
        super().__init__()
        self.backbone = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=BACKBONE_DTYPE, device_map="auto", trust_remote_code=True)
        self.backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        self.backbone.config.use_cache = False
        inject_houlsby_adapters(self.backbone, reduction_factor, non_linearity)
        hidden_size = self.backbone.config.hidden_size
        self.head_device = torch.device(HEAD_DEVICE)
        self.classifier = nn.Sequential(nn.Dropout(classifier_dropout), nn.Linear(hidden_size, hidden_size // 2), nn.GELU(), nn.Dropout(classifier_dropout), nn.Linear(hidden_size // 2, num_labels)).to(self.head_device)
        self._init_classifier()
        freeze_backbone_unfreeze_adapters_and_norms(self)
    def _init_classifier(self):
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    @staticmethod
    def mean_pool(last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).to(last_hidden_state.device).float()
        summed = (last_hidden_state.float() * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts
    def forward(self, input_ids, attention_mask, labels=None, class_weights=None):
        out = self.backbone.encoder(input_ids=input_ids, attention_mask=attention_mask, output_attentions=False, output_hidden_states=False)
        pooled = self.mean_pool(out.last_hidden_state, attention_mask)
        pooled = pooled.to(self.head_device)
        logits = self.classifier(pooled)
        loss = None
        if labels is not None:
            labels = labels.to(self.head_device)
            cw = class_weights.to(self.head_device) if class_weights is not None else None
            loss = nn.CrossEntropyLoss(weight=cw)(logits, labels)
        return loss, logits
def load_data(train_csv, test_csv):
    train_df = pd.read_csv(train_csv)
    test_df = pd.read_csv(test_csv)
    required = {"func_cleaned", "label"}
    assert required.issubset(set(train_df.columns)), f"Train CSV must contain {required}."
    assert required.issubset(set(test_df.columns)), f"Test CSV must contain {required}."
    train_df = train_df.dropna(subset=list(required)).reset_index(drop=True)
    test_df = test_df.dropna(subset=list(required)).reset_index(drop=True)
    train_df["label"] = train_df["label"].astype(int)
    test_df["label"] = test_df["label"].astype(int)
    print(f"Train samples : {len(train_df)}")
    print(f"Test  samples : {len(test_df)}")
    print(f"Train label dist:\n{train_df['label'].value_counts()}")
    print(f"Test  label dist:\n{test_df['label'].value_counts()}")
    return train_df, test_df
def compute_class_weights_tensor(labels, num_classes):
    classes = np.arange(num_classes)
    weights = compute_class_weight("balanced", classes=classes, y=np.array(labels))
    return torch.tensor(weights, dtype=torch.float32)
def train_epoch(model, dataloader, optimizer, scheduler, class_weights, epoch, gradient_accumulation_steps):
    model.train()
    total_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []
    optimizer.zero_grad()
    input_device = torch.device("cuda:0")
    for step, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(input_device)
        attention_mask = batch["attention_mask"].to(input_device)
        labels = batch["labels"]
        loss, logits = model(input_ids, attention_mask, labels, class_weights)
        scaled_loss = loss / gradient_accumulation_steps
        scaled_loss.backward()
        if (step + 1) % gradient_accumulation_steps == 0 or (step + 1) == len(dataloader):
            nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        total_loss += loss.item()
        with torch.no_grad():
            probs = F.softmax(logits.float(), dim=-1)
            preds = torch.argmax(probs, dim=-1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(batch["labels"].tolist())
        all_probs.extend(probs[:, 1].cpu().tolist())
        if (step + 1) % 50 == 0:
            print(f"  Epoch {epoch} | Step {step+1}/{len(dataloader)} | Avg Loss: {total_loss/(step+1):.4f}")
        if (step + 1) % 100 == 0:
            free_gpu_cache()
    return total_loss / len(dataloader), compute_metrics(all_labels, all_preds, all_probs)
@torch.no_grad()
def evaluate(model, dataloader, class_weights):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []
    input_device = torch.device("cuda:0")
    for batch in dataloader:
        input_ids = batch["input_ids"].to(input_device)
        attention_mask = batch["attention_mask"].to(input_device)
        labels = batch["labels"]
        loss, logits = model(input_ids, attention_mask, labels, class_weights)
        total_loss += loss.item()
        probs = F.softmax(logits.float(), dim=-1)
        preds = torch.argmax(probs, dim=-1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(batch["labels"].tolist())
        all_probs.extend(probs[:, 1].cpu().tolist())
    return total_loss / len(dataloader), compute_metrics(all_labels, all_preds, all_probs)
def compute_metrics(labels, preds, probs):
    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, average="binary", zero_division=0)
    rec = recall_score(labels, preds, average="binary", zero_division=0)
    f1 = f1_score(labels, preds, average="binary", zero_division=0)
    f1_mac = f1_score(labels, preds, average="macro", zero_division=0)
    mcc = matthews_corrcoef(labels, preds)
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = 0.0
    cm = confusion_matrix(labels, preds)
    if cm.size == 4:
        tn, fp, fn, tp = cm.ravel()
    else:
        tn = fp = fn = 0
        tp = int(cm[0, 0])
    fpr = float(fp) / (float(fp) + float(tn) + 1e-9)
    return {"accuracy": float(acc), "precision": float(prec), "recall": float(rec), "f1": float(f1), "f1_macro": float(f1_mac), "mcc": float(mcc), "auc_roc": float(auc), "fpr": float(fpr), "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)}
def print_metrics(split, loss, m, epoch=None):
    hdr = f"[{split}]" if epoch is None else f"[Epoch {epoch} | {split}]"
    print(f"\n{hdr}")
    print(f"  Loss      : {loss:.4f}")
    print(f"  Accuracy  : {m['accuracy']*100:.2f}%")
    print(f"  Precision : {m['precision']*100:.2f}%")
    print(f"  Recall    : {m['recall']*100:.2f}%")
    print(f"  F1 (Bin)  : {m['f1']*100:.2f}%")
    print(f"  F1 (Mac)  : {m['f1_macro']*100:.2f}%")
    print(f"  MCC       : {m['mcc']:.4f}")
    print(f"  AUC-ROC   : {m['auc_roc']:.4f}")
    print(f"  FPR       : {m['fpr']*100:.2f}%")
    print(f"  TP={m['tp']} | TN={m['tn']} | FP={m['fp']} | FN={m['fn']}")
def print_parameter_summary(model):
    total, trainable, frozen, pct = count_parameters(model)
    print("\n" + "=" * 68)
    print("HOULSBY ADAPTER-TUNING  —  PARAMETER SUMMARY")
    print("=" * 68)
    print(f"  Total parameters                       : {total:>15,}")
    print(f"  Trainable (adapters + LayerNorm + head): {trainable:>15,}  ({pct:.4f}%)")
    print(f"  Frozen    (CodeT5+ encoder backbone)   : {frozen:>15,}")
    print("=" * 68)
    print("\nTrainable parameter groups:")
    for name, param in model.named_parameters():
        if param.requires_grad:
            print(f"  {name:<80s} | {param.numel():>10,}")
    print("=" * 68 + "\n")
def main():
    print("=" * 68)
    print("ADAPTER-TUNING  (Houlsby et al., ICML 2019)")
    print("Parameter-Efficient Transfer Learning for NLP")
    print("Task      : Code Vulnerability Detection (Binary)")
    print(f"Backbone  : {MODEL_NAME}")
    print(f"Arch      : CodeT5+ encoder-decoder | encoder used for classification")
    print(f"Reduction : {ADAPTER_REDUCTION_FACTOR}x  → bottleneck m=hidden_size/{ADAPTER_REDUCTION_FACTOR}")
    print(f"Adapters  : 2 per encoder layer (self-attn + FF)")
    print(f"Backbone dtype           : {BACKBONE_DTYPE}")
    print(f"device_map               : auto (spread across all GPUs)")
    print(f"Gradient checkpointing   : ENABLED")
    print(f"MAX_LENGTH               : {MAX_LENGTH}")
    print(f"Effective batch size     : {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
    print("=" * 68 + "\n")
    train_df, test_df = load_data(TRAIN_CSV, TEST_CSV)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "left"
    train_dataset = VulnerabilityDataset(train_df, tokenizer, MAX_LENGTH)
    test_dataset = VulnerabilityDataset(test_df, tokenizer, MAX_LENGTH)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=False, drop_last=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=False)
    train_labels = train_df["label"].astype(int).tolist()
    class_weights = compute_class_weights_tensor(train_labels, NUM_LABELS)
    print(f"Class weights (balanced): {[round(w, 4) for w in class_weights.tolist()]}\n")
    print("Loading backbone with device_map=auto and injecting Houlsby adapters ...")
    free_gpu_cache()
    model = HoulsbyAdapterVulnerabilityClassifier(model_name=MODEL_NAME, num_labels=NUM_LABELS, reduction_factor=ADAPTER_REDUCTION_FACTOR, non_linearity=NON_LINEARITY, classifier_dropout=CLASSIFIER_DROPOUT)
    print("Adapter injection complete.\n")
    print("GPU state after model load:")
    print_gpu_memory()
    print_parameter_summary(model)
    free_gpu_cache()
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.999), eps=1e-8)
    total_steps = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS) * NUM_EPOCHS
    warmup_steps = int(WARMUP_RATIO * total_steps)
    print(f"Total optimiser steps : {total_steps}")
    print(f"Warmup steps          : {warmup_steps}\n")
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    best_f1 = 0.0
    best_epoch = 0
    history = []
    print("=" * 68)
    print("TRAINING")
    print("=" * 68)
    for epoch in range(1, NUM_EPOCHS + 1):
        free_gpu_cache()
        print(f"\nGPU state at start of epoch {epoch}:")
        print_gpu_memory()
        train_loss, train_metrics = train_epoch(model=model, dataloader=train_loader, optimizer=optimizer, scheduler=scheduler, class_weights=class_weights, epoch=epoch, gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS)
        print_metrics("TRAIN", train_loss, train_metrics, epoch)
        free_gpu_cache()
        test_loss, test_metrics = evaluate(model=model, dataloader=test_loader, class_weights=class_weights)
        print_metrics("TEST", test_loss, test_metrics, epoch)
        history.append({"epoch": epoch, "train_loss": train_loss, "test_loss": test_loss, **{f"train_{k}": v for k, v in train_metrics.items()}, **{f"test_{k}": v for k, v in test_metrics.items()}})
        if test_metrics["f1"] > best_f1:
            best_f1 = test_metrics["f1"]
            best_epoch = epoch
            trainable_state = {k: v for k, v in model.state_dict().items() if any(tag in k for tag in ("adapter", "norm", "classifier"))}
            torch.save({"epoch": epoch, "trainable_state_dict": trainable_state, "optimizer_state_dict": optimizer.state_dict(), "scheduler_state_dict": scheduler.state_dict(), "best_f1": best_f1, "test_metrics": test_metrics, "config": {"model_name": MODEL_NAME, "reduction_factor": ADAPTER_REDUCTION_FACTOR, "non_linearity": NON_LINEARITY, "max_length": MAX_LENGTH, "num_labels": NUM_LABELS}}, os.path.join(SAVE_DIR, "best_adapter_model.pt"))
            print(f"  ✓ Best checkpoint saved  (epoch {epoch} | F1={best_f1*100:.2f}%)")
    pd.DataFrame(history).to_csv(os.path.join(SAVE_DIR, "training_history.csv"), index=False)
    print("\n" + "=" * 68)
    print("FINAL EVALUATION — BEST CHECKPOINT")
    print("=" * 68)
    free_gpu_cache()
    ckpt = torch.load(os.path.join(SAVE_DIR, "best_adapter_model.pt"), map_location="cpu")
    model.load_state_dict(ckpt["trainable_state_dict"], strict=False)
    final_loss, final_metrics = evaluate(model=model, dataloader=test_loader, class_weights=class_weights)
    print_metrics("FINAL TEST", final_loss, final_metrics)
    _, trainable, _, pct = count_parameters(model)
    d_model = model.backbone.config.hidden_size
    bottleneck_dim = d_model // ADAPTER_REDUCTION_FACTOR
    num_layers = len(model.backbone.encoder.block)
    print("\n" + "=" * 68)
    print("HOULSBY ADAPTER-TUNING — COMPLETE RESULTS SUMMARY")
    print("=" * 68)
    print(f"  Model               : {MODEL_NAME}")
    print(f"  Best Epoch          : {best_epoch} / {NUM_EPOCHS}")
    print(f"  Accuracy            : {final_metrics['accuracy']*100:.2f}%")
    print(f"  Precision           : {final_metrics['precision']*100:.2f}%")
    print(f"  Recall              : {final_metrics['recall']*100:.2f}%")
    print(f"  F1 (Binary)         : {final_metrics['f1']*100:.2f}%")
    print(f"  F1 (Macro)          : {final_metrics['f1_macro']*100:.2f}%")
    print(f"  MCC                 : {final_metrics['mcc']:.4f}")
    print(f"  AUC-ROC             : {final_metrics['auc_roc']:.4f}")
    print(f"  FPR                 : {final_metrics['fpr']*100:.2f}%")
    print(f"  TP={final_metrics['tp']} | TN={final_metrics['tn']} | FP={final_metrics['fp']} | FN={final_metrics['fn']}")
    print(f"\n  Encoder layers      : {num_layers}")
    print(f"  Adapters injected   : {num_layers * 2}  (2 per layer × {num_layers})")
    print(f"  d_model             : {d_model}")
    print(f"  Bottleneck dim (m)  : {bottleneck_dim}  (d={d_model} / {ADAPTER_REDUCTION_FACTOR})")
    print(f"  Trainable params    : {trainable:,}  ({pct:.4f}% of total)")
    print(f"  Backbone dtype      : {BACKBONE_DTYPE}")
    print(f"  Gradient ckpt       : ENABLED")
    print("=" * 68)
    results_df = pd.DataFrame([{"method": "Adapter-Tuning (Houlsby et al., ICML 2019)", "backbone": MODEL_NAME, "reduction_factor": ADAPTER_REDUCTION_FACTOR, "non_linearity": NON_LINEARITY, "bottleneck_dim": bottleneck_dim, "num_adapter_pairs": num_layers, "total_adapters": num_layers * 2, "backbone_dtype": str(BACKBONE_DTYPE), "grad_checkpointing": True, "max_length": MAX_LENGTH, "effective_batch": BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS, "trainable_params": trainable, "trainable_pct": round(pct, 6), "best_epoch": best_epoch, **{f"final_{k}": round(v, 6) if isinstance(v, float) else v for k, v in final_metrics.items()}}])
    results_df.to_csv(os.path.join(SAVE_DIR, "final_results.csv"), index=False)
    print(f"\nArtifacts saved to : {SAVE_DIR}/")
    print(f"  final_results.csv")
    print(f"  training_history.csv")
    print(f"  best_adapter_model.pt")
if __name__ == "__main__":
    main()

# adapter Qwen2.5-Coder-1.5B code vul 1024 token

In [ ]:
import os
import gc
import warnings
import random
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix,
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")
os.environ["CUDA_VISIBLE_DEVICES"]      = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"]    = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"]  = "expandable_segments:True"

MODEL_NAME                  = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
TRAIN_CSV                   = "/code vul/trqw.csv"
TEST_CSV                    = "code vul/teqw.csv"
MAX_LENGTH                  = 1024
SEED                        = 42
BATCH_SIZE                  = 2
GRADIENT_ACCUMULATION_STEPS = 16
NUM_EPOCHS                  = 10
LEARNING_RATE               = 1e-4
WARMUP_RATIO                = 0.1
WEIGHT_DECAY                = 0.01
MAX_GRAD_NORM               = 1.0
ADAPTER_REDUCTION_FACTOR    = 16
NUM_LABELS                  = 2
CLASSIFIER_DROPOUT          = 0.1
NON_LINEARITY               = "gelu"
ADAPTER_INIT_SCALE          = 1e-3
BACKBONE_DTYPE              = torch.bfloat16
HEAD_DEVICE                 = "cuda:0"
SAVE_DIR                    = "./adapter_tuning_qwen_checkpoints"

os.makedirs(SAVE_DIR, exist_ok=True)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


set_seed(SEED)


def free_gpu_cache():
    gc.collect()
    torch.cuda.synchronize()
    torch.cuda.empty_cache()


def print_gpu_memory():
    for i in range(torch.cuda.device_count()):
        alloc  = torch.cuda.memory_allocated(i)  / 1024 ** 3
        reserv = torch.cuda.memory_reserved(i)   / 1024 ** 3
        print(f"  GPU {i}: allocated={alloc:.2f}GB  reserved={reserv:.2f}GB")


print(f"GPUs visible: {torch.cuda.device_count()}")
print("Initial GPU state:")
print_gpu_memory()


class HoulsbyBottleneckAdapter(nn.Module):
    """
    Houlsby et al. (ICML 2019) — "Parameter-Efficient Transfer Learning for NLP"
    Section 2.1 & Figure 2 — bottleneck adapter.

    Flow:  x → down_proj(d→m) → non-linearity → up_proj(m→d) → + x

    where  m = d / reduction_factor  (bottleneck dimension).

    Near-zero weight init: module starts as approximate identity ψ_{w,v0}(x) ≈ φ_w(x).
    Parameters per adapter: 2·m·d + m + d.
    For Qwen2.5-1.5B (d=1536, factor=16 → m=96): 296,544 params per adapter.
    """

    def __init__(self, hidden_size: int, reduction_factor: int, non_linearity: str):
        super().__init__()
        self.hidden_size     = hidden_size
        self.bottleneck_size = hidden_size // reduction_factor

        self.down_proj = nn.Linear(hidden_size, self.bottleneck_size)
        self.up_proj   = nn.Linear(self.bottleneck_size, hidden_size)

        if non_linearity == "gelu":
            self.activation = nn.GELU()
        elif non_linearity == "relu":
            self.activation = nn.ReLU()
        elif non_linearity == "swish":
            self.activation = nn.SiLU()
        else:
            raise ValueError(f"Unsupported non-linearity: {non_linearity}")

        nn.init.normal_(self.down_proj.weight, std=ADAPTER_INIT_SCALE)
        nn.init.zeros_(self.down_proj.bias)
        nn.init.normal_(self.up_proj.weight, std=ADAPTER_INIT_SCALE)
        nn.init.zeros_(self.up_proj.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual   = x
        orig_dtype = x.dtype
        x = self.down_proj(x.float())
        x = self.activation(x)
        x = self.up_proj(x)
        return x.to(orig_dtype) + residual


def inject_houlsby_adapters(
    backbone: AutoModelForCausalLM,
    reduction_factor: int,
    non_linearity: str,
) -> None:

    hidden_size = backbone.config.hidden_size

    for layer in backbone.model.layers:
        attn_adapter = HoulsbyBottleneckAdapter(hidden_size, reduction_factor, non_linearity)
        mlp_adapter  = HoulsbyBottleneckAdapter(hidden_size, reduction_factor, non_linearity)

        layer.self_attn.add_module("adapter", attn_adapter)
        layer.mlp.add_module("adapter", mlp_adapter)

        def _make_hook(adp: HoulsbyBottleneckAdapter):
            def hook(module, inp, output):
                h = output[0] if isinstance(output, tuple) else output
                target_device = h.device
                if next(adp.parameters()).device != target_device:
                    adp.to(target_device)
                adapted = adp(h)
                if isinstance(output, tuple):
                    return (adapted,) + output[1:]
                return adapted
            return hook

        layer.self_attn.register_forward_hook(_make_hook(attn_adapter))
        layer.mlp.register_forward_hook(_make_hook(mlp_adapter))


def freeze_backbone_unfreeze_adapters_and_norms(model: nn.Module) -> None:

    for param in model.parameters():
        param.requires_grad = False

    for name, param in model.named_parameters():
        if "adapter" in name or "norm" in name or "classifier" in name:
            param.requires_grad = True


def count_parameters(model: nn.Module):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen    = total - trainable
    pct       = 100.0 * trainable / total if total > 0 else 0.0
    return total, trainable, frozen, pct


class VulnerabilityDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, tokenizer, max_length: int):
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.texts      = dataframe["func_cleaned"].astype(str).tolist()
        self.labels     = dataframe["label"].astype(int).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }


class HoulsbyAdapterVulnerabilityClassifier(nn.Module):

    def __init__(
        self,
        model_name: str,
        num_labels: int,
        reduction_factor: int,
        non_linearity: str,
        classifier_dropout: float,
    ):
        super().__init__()

        self.backbone = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=BACKBONE_DTYPE,
            device_map="auto",
            trust_remote_code=True,
        )

        self.backbone.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
        self.backbone.config.use_cache = False

        inject_houlsby_adapters(self.backbone, reduction_factor, non_linearity)

        hidden_size      = self.backbone.config.hidden_size
        self.head_device = torch.device(HEAD_DEVICE)

        self.classifier = nn.Sequential(
            nn.Dropout(classifier_dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(classifier_dropout),
            nn.Linear(hidden_size // 2, num_labels),
        ).to(self.head_device)

        self._init_classifier()
        freeze_backbone_unfreeze_adapters_and_norms(self)

    def _init_classifier(self):
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    @staticmethod
    def mean_pool(
        last_hidden_state: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        mask    = attention_mask.unsqueeze(-1).to(last_hidden_state.device).float()
        summed  = (last_hidden_state.float() * mask).sum(dim=1)
        counts  = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        labels: torch.Tensor = None,
        class_weights: torch.Tensor = None,
    ):
        out = self.backbone.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            output_attentions=False,
            output_hidden_states=False,
        )

        pooled = self.mean_pool(out.last_hidden_state, attention_mask)
        pooled = pooled.to(self.head_device)
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            labels = labels.to(self.head_device)
            cw     = class_weights.to(self.head_device) if class_weights is not None else None
            loss   = nn.CrossEntropyLoss(weight=cw)(logits, labels)

        return loss, logits


def load_data(train_csv: str, test_csv: str):
    train_df = pd.read_csv(train_csv)
    test_df  = pd.read_csv(test_csv)

    required = {"func_cleaned", "label"}
    assert required.issubset(set(train_df.columns)), \
        f"Train CSV must contain {required}. Found: {set(train_df.columns)}"
    assert required.issubset(set(test_df.columns)), \
        f"Test CSV must contain {required}. Found: {set(test_df.columns)}"

    train_df = train_df.dropna(subset=list(required)).reset_index(drop=True)
    test_df  = test_df.dropna(subset=list(required)).reset_index(drop=True)
    train_df["label"] = train_df["label"].astype(int)
    test_df["label"]  = test_df["label"].astype(int)

    print(f"Train samples : {len(train_df)}")
    print(f"Test  samples : {len(test_df)}")
    print(f"Train label dist:\n{train_df['label'].value_counts()}")
    print(f"Test  label dist:\n{test_df['label'].value_counts()}")
    return train_df, test_df


def compute_class_weights_tensor(labels: list, num_classes: int) -> torch.Tensor:
    classes = np.arange(num_classes)
    weights = compute_class_weight("balanced", classes=classes, y=np.array(labels))
    return torch.tensor(weights, dtype=torch.float32)


def train_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    optimizer: AdamW,
    scheduler,
    class_weights: torch.Tensor,
    epoch: int,
    gradient_accumulation_steps: int,
):
    model.train()
    total_loss                    = 0.0
    all_preds, all_labels, all_probs = [], [], []
    optimizer.zero_grad()

    input_device = torch.device("cuda:0")

    for step, batch in enumerate(dataloader):
        input_ids      = batch["input_ids"].to(input_device)
        attention_mask = batch["attention_mask"].to(input_device)
        labels         = batch["labels"]

        loss, logits = model(input_ids, attention_mask, labels, class_weights)

        scaled_loss = loss / gradient_accumulation_steps
        scaled_loss.backward()

        if (step + 1) % gradient_accumulation_steps == 0 or (step + 1) == len(dataloader):
            nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                MAX_GRAD_NORM,
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item()

        with torch.no_grad():
            probs = F.softmax(logits.float(), dim=-1)
            preds = torch.argmax(probs, dim=-1)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(batch["labels"].tolist())
        all_probs.extend(probs[:, 1].cpu().tolist())

        if (step + 1) % 50 == 0:
            print(
                f"  Epoch {epoch} | Step {step+1}/{len(dataloader)} | "
                f"Avg Loss: {total_loss/(step+1):.4f}"
            )

        if (step + 1) % 100 == 0:
            free_gpu_cache()

    return total_loss / len(dataloader), compute_metrics(all_labels, all_preds, all_probs)


@torch.no_grad()
def evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    class_weights: torch.Tensor,
):
    model.eval()
    total_loss                    = 0.0
    all_preds, all_labels, all_probs = [], [], []
    input_device                  = torch.device("cuda:0")

    for batch in dataloader:
        input_ids      = batch["input_ids"].to(input_device)
        attention_mask = batch["attention_mask"].to(input_device)
        labels         = batch["labels"]

        loss, logits = model(input_ids, attention_mask, labels, class_weights)
        total_loss  += loss.item()

        probs = F.softmax(logits.float(), dim=-1)
        preds = torch.argmax(probs, dim=-1)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(batch["labels"].tolist())
        all_probs.extend(probs[:, 1].cpu().tolist())

    return total_loss / len(dataloader), compute_metrics(all_labels, all_preds, all_probs)


def compute_metrics(labels, preds, probs):
    acc    = accuracy_score(labels, preds)
    prec   = precision_score(labels, preds, average="binary", zero_division=0)
    rec    = recall_score(labels, preds, average="binary", zero_division=0)
    f1     = f1_score(labels, preds, average="binary", zero_division=0)
    f1_mac = f1_score(labels, preds, average="macro",  zero_division=0)
    mcc    = matthews_corrcoef(labels, preds)
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = 0.0
    cm = confusion_matrix(labels, preds)
    if cm.size == 4:
        tn, fp, fn, tp = cm.ravel()
    else:
        tn = fp = fn = 0
        tp = int(cm[0, 0])
    fpr = float(fp) / (float(fp) + float(tn) + 1e-9)
    return {
        "accuracy":  float(acc),
        "precision": float(prec),
        "recall":    float(rec),
        "f1":        float(f1),
        "f1_macro":  float(f1_mac),
        "mcc":       float(mcc),
        "auc_roc":   float(auc),
        "fpr":       float(fpr),
        "tp":        int(tp),
        "tn":        int(tn),
        "fp":        int(fp),
        "fn":        int(fn),
    }


def print_metrics(split: str, loss: float, m: dict, epoch: int = None):
    hdr = f"[{split}]" if epoch is None else f"[Epoch {epoch} | {split}]"
    print(f"\n{hdr}")
    print(f"  Loss      : {loss:.4f}")
    print(f"  Accuracy  : {m['accuracy']*100:.2f}%")
    print(f"  Precision : {m['precision']*100:.2f}%")
    print(f"  Recall    : {m['recall']*100:.2f}%")
    print(f"  F1 (Bin)  : {m['f1']*100:.2f}%")
    print(f"  F1 (Mac)  : {m['f1_macro']*100:.2f}%")
    print(f"  MCC       : {m['mcc']:.4f}")
    print(f"  AUC-ROC   : {m['auc_roc']:.4f}")
    print(f"  FPR       : {m['fpr']*100:.2f}%")
    print(f"  TP={m['tp']} | TN={m['tn']} | FP={m['fp']} | FN={m['fn']}")


def print_parameter_summary(model: nn.Module):
    total, trainable, frozen, pct = count_parameters(model)
    print("\n" + "=" * 68)
    print("HOULSBY ADAPTER-TUNING  —  PARAMETER SUMMARY")
    print("=" * 68)
    print(f"  Total parameters                       : {total:>15,}")
    print(f"  Trainable (adapters + RMSNorm + head)  : {trainable:>15,}  ({pct:.4f}%)")
    print(f"  Frozen    (Qwen2 backbone)             : {frozen:>15,}")
    print("=" * 68)
    print("\nTrainable parameter groups:")
    for name, param in model.named_parameters():
        if param.requires_grad:
            print(f"  {name:<80s} | {param.numel():>10,}")
    print("=" * 68 + "\n")


def main():
    print("=" * 68)
    print("ADAPTER-TUNING  (Houlsby et al., ICML 2019)")
    print("Parameter-Efficient Transfer Learning for NLP")
    print("Task      : Code Vulnerability Detection (Binary)")
    print(f"Backbone  : {MODEL_NAME}")
    print(f"Arch      : Qwen2 decoder-only | 28 layers | hidden=1536")
    print(f"Reduction : {ADAPTER_REDUCTION_FACTOR}x  → bottleneck m={1536//ADAPTER_REDUCTION_FACTOR}")
    print(f"Adapters  : 2 per decoder layer × 28 = 56 total")
    print(f"Backbone dtype           : {BACKBONE_DTYPE}")
    print(f"device_map               : auto (spread across all GPUs)")
    print(f"Gradient checkpointing   : ENABLED")
    print(f"MAX_LENGTH               : {MAX_LENGTH}")
    print(f"Effective batch size     : {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
    print("=" * 68 + "\n")

    train_df, test_df = load_data(TRAIN_CSV, TEST_CSV)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "left"

    train_dataset = VulnerabilityDataset(train_df, tokenizer, MAX_LENGTH)
    test_dataset  = VulnerabilityDataset(test_df,  tokenizer, MAX_LENGTH)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=False,
        drop_last=False,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=False,
    )

    train_labels  = train_df["label"].astype(int).tolist()
    class_weights = compute_class_weights_tensor(train_labels, NUM_LABELS)
    print(f"Class weights (balanced): {[round(w, 4) for w in class_weights.tolist()]}\n")

    print("Loading backbone with device_map=auto and injecting Houlsby adapters ...")
    free_gpu_cache()

    model = HoulsbyAdapterVulnerabilityClassifier(
        model_name=MODEL_NAME,
        num_labels=NUM_LABELS,
        reduction_factor=ADAPTER_REDUCTION_FACTOR,
        non_linearity=NON_LINEARITY,
        classifier_dropout=CLASSIFIER_DROPOUT,
    )

    print("Adapter injection complete.\n")
    print("GPU state after model load:")
    print_gpu_memory()

    print_parameter_summary(model)
    free_gpu_cache()

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(
        trainable_params,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        betas=(0.9, 0.999),
        eps=1e-8,
    )

    total_steps  = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS) * NUM_EPOCHS
    warmup_steps = int(WARMUP_RATIO * total_steps)
    print(f"Total optimiser steps : {total_steps}")
    print(f"Warmup steps          : {warmup_steps}\n")

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    best_f1    = 0.0
    best_epoch = 0
    history    = []

    print("=" * 68)
    print("TRAINING")
    print("=" * 68)

    for epoch in range(1, NUM_EPOCHS + 1):
        free_gpu_cache()
        print(f"\nGPU state at start of epoch {epoch}:")
        print_gpu_memory()

        train_loss, train_metrics = train_epoch(
            model=model,
            dataloader=train_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            class_weights=class_weights,
            epoch=epoch,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        )
        print_metrics("TRAIN", train_loss, train_metrics, epoch)

        free_gpu_cache()

        test_loss, test_metrics = evaluate(
            model=model,
            dataloader=test_loader,
            class_weights=class_weights,
        )
        print_metrics("TEST", test_loss, test_metrics, epoch)

        history.append({
            "epoch":      epoch,
            "train_loss": train_loss,
            "test_loss":  test_loss,
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"test_{k}":  v for k, v in test_metrics.items()},
        })

        if test_metrics["f1"] > best_f1:
            best_f1    = test_metrics["f1"]
            best_epoch = epoch

            trainable_state = {
                k: v for k, v in model.state_dict().items()
                if any(tag in k for tag in ("adapter", "norm", "classifier"))
            }
            torch.save(
                {
                    "epoch":                epoch,
                    "trainable_state_dict": trainable_state,
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "best_f1":              best_f1,
                    "test_metrics":         test_metrics,
                    "config": {
                        "model_name":       MODEL_NAME,
                        "reduction_factor": ADAPTER_REDUCTION_FACTOR,
                        "non_linearity":    NON_LINEARITY,
                        "max_length":       MAX_LENGTH,
                        "num_labels":       NUM_LABELS,
                    },
                },
                os.path.join(SAVE_DIR, "best_adapter_model.pt"),
            )
            print(f"  ✓ Best checkpoint saved  (epoch {epoch} | F1={best_f1*100:.2f}%)")

    pd.DataFrame(history).to_csv(
        os.path.join(SAVE_DIR, "training_history.csv"), index=False
    )

    print("\n" + "=" * 68)
    print("FINAL EVALUATION — BEST CHECKPOINT")
    print("=" * 68)

    free_gpu_cache()

    ckpt = torch.load(
        os.path.join(SAVE_DIR, "best_adapter_model.pt"),
        map_location="cpu",
    )
    model.load_state_dict(ckpt["trainable_state_dict"], strict=False)

    final_loss, final_metrics = evaluate(
        model=model,
        dataloader=test_loader,
        class_weights=class_weights,
    )
    print_metrics("FINAL TEST", final_loss, final_metrics)

    _, trainable, _, pct = count_parameters(model)
    d_model        = model.backbone.config.hidden_size
    bottleneck_dim = d_model // ADAPTER_REDUCTION_FACTOR
    num_layers     = model.backbone.config.num_hidden_layers

    print("\n" + "=" * 68)
    print("HOULSBY ADAPTER-TUNING — COMPLETE RESULTS SUMMARY")
    print("=" * 68)
    print(f"  Model               : {MODEL_NAME}")
    print(f"  Best Epoch          : {best_epoch} / {NUM_EPOCHS}")
    print(f"  Accuracy            : {final_metrics['accuracy']*100:.2f}%")
    print(f"  Precision           : {final_metrics['precision']*100:.2f}%")
    print(f"  Recall              : {final_metrics['recall']*100:.2f}%")
    print(f"  F1 (Binary)         : {final_metrics['f1']*100:.2f}%")
    print(f"  F1 (Macro)          : {final_metrics['f1_macro']*100:.2f}%")
    print(f"  MCC                 : {final_metrics['mcc']:.4f}")
    print(f"  AUC-ROC             : {final_metrics['auc_roc']:.4f}")
    print(f"  FPR                 : {final_metrics['fpr']*100:.2f}%")
    print(f"  TP={final_metrics['tp']} | TN={final_metrics['tn']} | "
          f"FP={final_metrics['fp']} | FN={final_metrics['fn']}")
    print(f"\n  Decoder layers      : {num_layers}")
    print(f"  Adapters injected   : {num_layers * 2}  (2 per layer × {num_layers})")
    print(f"  d_model             : {d_model}")
    print(f"  Bottleneck dim (m)  : {bottleneck_dim}  (d={d_model} / {ADAPTER_REDUCTION_FACTOR})")
    print(f"  Trainable params    : {trainable:,}  ({pct:.4f}% of total)")
    print(f"  Backbone dtype      : {BACKBONE_DTYPE}")
    print(f"  Gradient ckpt       : ENABLED")
    print("=" * 68)

    results_df = pd.DataFrame([{
        "method":            "Adapter-Tuning (Houlsby et al., ICML 2019)",
        "backbone":          MODEL_NAME,
        "reduction_factor":  ADAPTER_REDUCTION_FACTOR,
        "non_linearity":     NON_LINEARITY,
        "bottleneck_dim":    bottleneck_dim,
        "num_adapter_pairs": num_layers,
        "total_adapters":    num_layers * 2,
        "backbone_dtype":    str(BACKBONE_DTYPE),
        "grad_checkpointing": True,
        "max_length":        MAX_LENGTH,
        "effective_batch":   BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
        "trainable_params":  trainable,
        "trainable_pct":     round(pct, 6),
        "best_epoch":        best_epoch,
        **{f"final_{k}": round(v, 6) if isinstance(v, float) else v
           for k, v in final_metrics.items()},
    }])
    results_df.to_csv(os.path.join(SAVE_DIR, "final_results.csv"), index=False)

    print(f"\nArtifacts saved to : {SAVE_DIR}/")
    print(f"  final_results.csv")
    print(f"  training_history.csv")
    print(f"  best_adapter_model.pt")


if __name__ == "__main__":
    main()

# adpater Qwen2.5-Coder-3B, 1024 token

In [ ]:
import os
import gc
import warnings
import random
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, matthews_corrcoef, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
warnings.filterwarnings("ignore")
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"
TRAIN_CSV     = "/code vul/trqw.csv"
TEST_CSV      = "/code vul/teqw.csv"
MAX_LENGTH    = 1024
SEED = 42
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 16
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
ADAPTER_REDUCTION_FACTOR = 16
NUM_LABELS = 2
CLASSIFIER_DROPOUT = 0.1
NON_LINEARITY = "gelu"
ADAPTER_INIT_SCALE = 1e-3
BACKBONE_DTYPE = torch.bfloat16
HEAD_DEVICE = "cuda:0"
SAVE_DIR = "./adapter_tuning_qwen3b_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(SEED)
def free_gpu_cache():
    gc.collect()
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
def print_gpu_memory():
    for i in range(torch.cuda.device_count()):
        alloc = torch.cuda.memory_allocated(i) / 1024 ** 3
        reserv = torch.cuda.memory_reserved(i) / 1024 ** 3
        print(f"  GPU {i}: allocated={alloc:.2f}GB  reserved={reserv:.2f}GB")
print(f"GPUs visible: {torch.cuda.device_count()}")
print("Initial GPU state:")
print_gpu_memory()
class HoulsbyBottleneckAdapter(nn.Module):
    def __init__(self, hidden_size, reduction_factor, non_linearity):
        super().__init__()
        self.hidden_size = hidden_size
        self.bottleneck_size = hidden_size // reduction_factor
        self.down_proj = nn.Linear(hidden_size, self.bottleneck_size)
        self.up_proj = nn.Linear(self.bottleneck_size, hidden_size)
        if non_linearity == "gelu":
            self.activation = nn.GELU()
        elif non_linearity == "relu":
            self.activation = nn.ReLU()
        elif non_linearity == "swish":
            self.activation = nn.SiLU()
        else:
            raise ValueError(f"Unsupported non-linearity: {non_linearity}")
        nn.init.normal_(self.down_proj.weight, std=ADAPTER_INIT_SCALE)
        nn.init.zeros_(self.down_proj.bias)
        nn.init.normal_(self.up_proj.weight, std=ADAPTER_INIT_SCALE)
        nn.init.zeros_(self.up_proj.bias)
    def forward(self, x):
        residual = x
        orig_dtype = x.dtype
        x = self.down_proj(x.float())
        x = self.activation(x)
        x = self.up_proj(x)
        return x.to(orig_dtype) + residual
def inject_houlsby_adapters(backbone, reduction_factor, non_linearity):
    hidden_size = backbone.config.hidden_size
    for layer in backbone.model.layers:
        attn_adapter = HoulsbyBottleneckAdapter(hidden_size, reduction_factor, non_linearity)
        mlp_adapter = HoulsbyBottleneckAdapter(hidden_size, reduction_factor, non_linearity)
        layer.self_attn.add_module("adapter", attn_adapter)
        layer.mlp.add_module("adapter", mlp_adapter)
        def _make_hook(adp):
            def hook(module, inp, output):
                h = output[0] if isinstance(output, tuple) else output
                target_device = h.device
                if next(adp.parameters()).device != target_device:
                    adp.to(target_device)
                adapted = adp(h)
                if isinstance(output, tuple):
                    return (adapted,) + output[1:]
                return adapted
            return hook
        layer.self_attn.register_forward_hook(_make_hook(attn_adapter))
        layer.mlp.register_forward_hook(_make_hook(mlp_adapter))
def freeze_backbone_unfreeze_adapters_and_norms(model):
    for param in model.parameters():
        param.requires_grad = False
    for name, param in model.named_parameters():
        if "adapter" in name or "norm" in name or "classifier" in name:
            param.requires_grad = True
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - trainable
    pct = 100.0 * trainable / total if total > 0 else 0.0
    return total, trainable, frozen, pct
class VulnerabilityDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.texts = dataframe["code"].astype(str).tolist()
        self.labels = dataframe["label"].astype(int).tolist()
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0), "labels": torch.tensor(self.labels[idx], dtype=torch.long)}
class HoulsbyAdapterVulnerabilityClassifier(nn.Module):
    def __init__(self, model_name, num_labels, reduction_factor, non_linearity, classifier_dropout):
        super().__init__()
        self.backbone = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=BACKBONE_DTYPE, device_map="auto", trust_remote_code=True)
        self.backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        self.backbone.config.use_cache = False
        inject_houlsby_adapters(self.backbone, reduction_factor, non_linearity)
        hidden_size = self.backbone.config.hidden_size
        self.head_device = torch.device(HEAD_DEVICE)
        self.classifier = nn.Sequential(nn.Dropout(classifier_dropout), nn.Linear(hidden_size, hidden_size // 2), nn.GELU(), nn.Dropout(classifier_dropout), nn.Linear(hidden_size // 2, num_labels)).to(self.head_device)
        self._init_classifier()
        freeze_backbone_unfreeze_adapters_and_norms(self)
    def _init_classifier(self):
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    @staticmethod
    def mean_pool(last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).to(last_hidden_state.device).float()
        summed = (last_hidden_state.float() * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts
    def forward(self, input_ids, attention_mask, labels=None, class_weights=None):
        out = self.backbone.model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False, output_attentions=False, output_hidden_states=False)
        pooled = self.mean_pool(out.last_hidden_state, attention_mask)
        pooled = pooled.to(self.head_device)
        logits = self.classifier(pooled)
        loss = None
        if labels is not None:
            labels = labels.to(self.head_device)
            cw = class_weights.to(self.head_device) if class_weights is not None else None
            loss = nn.CrossEntropyLoss(weight=cw)(logits, labels)
        return loss, logits
def load_data(train_csv, test_csv):
    train_df = pd.read_csv(train_csv)
    test_df = pd.read_csv(test_csv)
    required = {"code", "label"}
    assert required.issubset(set(train_df.columns)), f"Train CSV must contain {required}."
    assert required.issubset(set(test_df.columns)), f"Test CSV must contain {required}."
    train_df = train_df.dropna(subset=list(required)).reset_index(drop=True)
    test_df = test_df.dropna(subset=list(required)).reset_index(drop=True)
    train_df["label"] = train_df["label"].astype(int)
    test_df["label"] = test_df["label"].astype(int)
    print(f"Train samples : {len(train_df)}")
    print(f"Test  samples : {len(test_df)}")
    print(f"Train label dist:\n{train_df['label'].value_counts()}")
    print(f"Test  label dist:\n{test_df['label'].value_counts()}")
    return train_df, test_df
def compute_class_weights_tensor(labels, num_classes):
    classes = np.arange(num_classes)
    weights = compute_class_weight("balanced", classes=classes, y=np.array(labels))
    return torch.tensor(weights, dtype=torch.float32)
def train_epoch(model, dataloader, optimizer, scheduler, class_weights, epoch, gradient_accumulation_steps):
    model.train()
    total_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []
    optimizer.zero_grad()
    input_device = torch.device("cuda:0")
    for step, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(input_device)
        attention_mask = batch["attention_mask"].to(input_device)
        labels = batch["labels"]
        loss, logits = model(input_ids, attention_mask, labels, class_weights)
        scaled_loss = loss / gradient_accumulation_steps
        scaled_loss.backward()
        if (step + 1) % gradient_accumulation_steps == 0 or (step + 1) == len(dataloader):
            nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        total_loss += loss.item()
        with torch.no_grad():
            probs = F.softmax(logits.float(), dim=-1)
            preds = torch.argmax(probs, dim=-1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(batch["labels"].tolist())
        all_probs.extend(probs[:, 1].cpu().tolist())
        if (step + 1) % 50 == 0:
            print(f"  Epoch {epoch} | Step {step+1}/{len(dataloader)} | Avg Loss: {total_loss/(step+1):.4f}")
        if (step + 1) % 100 == 0:
            free_gpu_cache()
    return total_loss / len(dataloader), compute_metrics(all_labels, all_preds, all_probs)
@torch.no_grad()
def evaluate(model, dataloader, class_weights):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []
    input_device = torch.device("cuda:0")
    for batch in dataloader:
        input_ids = batch["input_ids"].to(input_device)
        attention_mask = batch["attention_mask"].to(input_device)
        labels = batch["labels"]
        loss, logits = model(input_ids, attention_mask, labels, class_weights)
        total_loss += loss.item()
        probs = F.softmax(logits.float(), dim=-1)
        preds = torch.argmax(probs, dim=-1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(batch["labels"].tolist())
        all_probs.extend(probs[:, 1].cpu().tolist())
    return total_loss / len(dataloader), compute_metrics(all_labels, all_preds, all_probs)
def compute_metrics(labels, preds, probs):
    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, average="binary", zero_division=0)
    rec = recall_score(labels, preds, average="binary", zero_division=0)
    f1 = f1_score(labels, preds, average="binary", zero_division=0)
    f1_mac = f1_score(labels, preds, average="macro", zero_division=0)
    mcc = matthews_corrcoef(labels, preds)
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = 0.0
    cm = confusion_matrix(labels, preds)
    if cm.size == 4:
        tn, fp, fn, tp = cm.ravel()
    else:
        tn = fp = fn = 0
        tp = int(cm[0, 0])
    fpr = float(fp) / (float(fp) + float(tn) + 1e-9)
    return {"accuracy": float(acc), "precision": float(prec), "recall": float(rec), "f1": float(f1), "f1_macro": float(f1_mac), "mcc": float(mcc), "auc_roc": float(auc), "fpr": float(fpr), "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)}
def print_metrics(split, loss, m, epoch=None):
    hdr = f"[{split}]" if epoch is None else f"[Epoch {epoch} | {split}]"
    print(f"\n{hdr}")
    print(f"  Loss      : {loss:.4f}")
    print(f"  Accuracy  : {m['accuracy']*100:.2f}%")
    print(f"  Precision : {m['precision']*100:.2f}%")
    print(f"  Recall    : {m['recall']*100:.2f}%")
    print(f"  F1 (Bin)  : {m['f1']*100:.2f}%")
    print(f"  F1 (Mac)  : {m['f1_macro']*100:.2f}%")
    print(f"  MCC       : {m['mcc']:.4f}")
    print(f"  AUC-ROC   : {m['auc_roc']:.4f}")
    print(f"  FPR       : {m['fpr']*100:.2f}%")
    print(f"  TP={m['tp']} | TN={m['tn']} | FP={m['fp']} | FN={m['fn']}")
def print_parameter_summary(model):
    total, trainable, frozen, pct = count_parameters(model)
    print("\n" + "=" * 68)
    print("HOULSBY ADAPTER-TUNING  —  PARAMETER SUMMARY")
    print("=" * 68)
    print(f"  Total parameters                       : {total:>15,}")
    print(f"  Trainable (adapters + RMSNorm + head)  : {trainable:>15,}  ({pct:.4f}%)")
    print(f"  Frozen    (Qwen2 backbone)             : {frozen:>15,}")
    print("=" * 68)
    print("\nTrainable parameter groups:")
    for name, param in model.named_parameters():
        if param.requires_grad:
            print(f"  {name:<80s} | {param.numel():>10,}")
    print("=" * 68 + "\n")
def main():
    print("=" * 68)
    print("ADAPTER-TUNING  (Houlsby et al., ICML 2019)")
    print("Parameter-Efficient Transfer Learning for NLP")
    print("Task      : Code Vulnerability Detection (Binary)")
    print(f"Backbone  : {MODEL_NAME}")
    print(f"Arch      : Qwen2 decoder-only | 36 layers | hidden=2048")
    print(f"Reduction : {ADAPTER_REDUCTION_FACTOR}x  → bottleneck m={2048//ADAPTER_REDUCTION_FACTOR}")
    print(f"Adapters  : 2 per decoder layer × 36 = 72 total")
    print(f"Backbone dtype           : {BACKBONE_DTYPE}")
    print(f"device_map               : auto (spread across all GPUs)")
    print(f"Gradient checkpointing   : ENABLED")
    print(f"MAX_LENGTH               : {MAX_LENGTH}")
    print(f"Effective batch size     : {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
    print("=" * 68 + "\n")
    train_df, test_df = load_data(TRAIN_CSV, TEST_CSV)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "left"
    train_dataset = VulnerabilityDataset(train_df, tokenizer, MAX_LENGTH)
    test_dataset = VulnerabilityDataset(test_df, tokenizer, MAX_LENGTH)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=False, drop_last=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=False)
    train_labels = train_df["label"].astype(int).tolist()
    class_weights = compute_class_weights_tensor(train_labels, NUM_LABELS)
    print(f"Class weights (balanced): {[round(w, 4) for w in class_weights.tolist()]}\n")
    print("Loading backbone with device_map=auto and injecting Houlsby adapters ...")
    free_gpu_cache()
    model = HoulsbyAdapterVulnerabilityClassifier(model_name=MODEL_NAME, num_labels=NUM_LABELS, reduction_factor=ADAPTER_REDUCTION_FACTOR, non_linearity=NON_LINEARITY, classifier_dropout=CLASSIFIER_DROPOUT)
    print("Adapter injection complete.\n")
    print("GPU state after model load:")
    print_gpu_memory()
    print_parameter_summary(model)
    free_gpu_cache()
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.999), eps=1e-8)
    total_steps = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS) * NUM_EPOCHS
    warmup_steps = int(WARMUP_RATIO * total_steps)
    print(f"Total optimiser steps : {total_steps}")
    print(f"Warmup steps          : {warmup_steps}\n")
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    best_f1 = 0.0
    best_epoch = 0
    history = []
    print("=" * 68)
    print("TRAINING")
    print("=" * 68)
    for epoch in range(1, NUM_EPOCHS + 1):
        free_gpu_cache()
        print(f"\nGPU state at start of epoch {epoch}:")
        print_gpu_memory()
        train_loss, train_metrics = train_epoch(model=model, dataloader=train_loader, optimizer=optimizer, scheduler=scheduler, class_weights=class_weights, epoch=epoch, gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS)
        print_metrics("TRAIN", train_loss, train_metrics, epoch)
        free_gpu_cache()
        test_loss, test_metrics = evaluate(model=model, dataloader=test_loader, class_weights=class_weights)
        print_metrics("TEST", test_loss, test_metrics, epoch)
        history.append({"epoch": epoch, "train_loss": train_loss, "test_loss": test_loss, **{f"train_{k}": v for k, v in train_metrics.items()}, **{f"test_{k}": v for k, v in test_metrics.items()}})
        if test_metrics["f1"] > best_f1:
            best_f1 = test_metrics["f1"]
            best_epoch = epoch
            trainable_state = {k: v for k, v in model.state_dict().items() if any(tag in k for tag in ("adapter", "norm", "classifier"))}
            torch.save({"epoch": epoch, "trainable_state_dict": trainable_state, "optimizer_state_dict": optimizer.state_dict(), "scheduler_state_dict": scheduler.state_dict(), "best_f1": best_f1, "test_metrics": test_metrics, "config": {"model_name": MODEL_NAME, "reduction_factor": ADAPTER_REDUCTION_FACTOR, "non_linearity": NON_LINEARITY, "max_length": MAX_LENGTH, "num_labels": NUM_LABELS}}, os.path.join(SAVE_DIR, "best_adapter_model.pt"))
            print(f"  ✓ Best checkpoint saved  (epoch {epoch} | F1={best_f1*100:.2f}%)")
    pd.DataFrame(history).to_csv(os.path.join(SAVE_DIR, "training_history.csv"), index=False)
    print("\n" + "=" * 68)
    print("FINAL EVALUATION — BEST CHECKPOINT")
    print("=" * 68)
    free_gpu_cache()
    ckpt = torch.load(os.path.join(SAVE_DIR, "best_adapter_model.pt"), map_location="cpu")
    model.load_state_dict(ckpt["trainable_state_dict"], strict=False)
    final_loss, final_metrics = evaluate(model=model, dataloader=test_loader, class_weights=class_weights)
    print_metrics("FINAL TEST", final_loss, final_metrics)
    _, trainable, _, pct = count_parameters(model)
    d_model = model.backbone.config.hidden_size
    bottleneck_dim = d_model // ADAPTER_REDUCTION_FACTOR
    num_layers = model.backbone.config.num_hidden_layers
    print("\n" + "=" * 68)
    print("HOULSBY ADAPTER-TUNING — COMPLETE RESULTS SUMMARY")
    print("=" * 68)
    print(f"  Model               : {MODEL_NAME}")
    print(f"  Best Epoch          : {best_epoch} / {NUM_EPOCHS}")
    print(f"  Accuracy            : {final_metrics['accuracy']*100:.2f}%")
    print(f"  Precision           : {final_metrics['precision']*100:.2f}%")
    print(f"  Recall              : {final_metrics['recall']*100:.2f}%")
    print(f"  F1 (Binary)         : {final_metrics['f1']*100:.2f}%")
    print(f"  F1 (Macro)          : {final_metrics['f1_macro']*100:.2f}%")
    print(f"  MCC                 : {final_metrics['mcc']:.4f}")
    print(f"  AUC-ROC             : {final_metrics['auc_roc']:.4f}")
    print(f"  FPR                 : {final_metrics['fpr']*100:.2f}%")
    print(f"  TP={final_metrics['tp']} | TN={final_metrics['tn']} | FP={final_metrics['fp']} | FN={final_metrics['fn']}")
    print(f"\n  Decoder layers      : {num_layers}")
    print(f"  Adapters injected   : {num_layers * 2}  (2 per layer × {num_layers})")
    print(f"  d_model             : {d_model}")
    print(f"  Bottleneck dim (m)  : {bottleneck_dim}  (d={d_model} / {ADAPTER_REDUCTION_FACTOR})")
    print(f"  Trainable params    : {trainable:,}  ({pct:.4f}% of total)")
    print(f"  Backbone dtype      : {BACKBONE_DTYPE}")
    print(f"  Gradient ckpt       : ENABLED")
    print("=" * 68)
    results_df = pd.DataFrame([{"method": "Adapter-Tuning (Houlsby et al., ICML 2019)", "backbone": MODEL_NAME, "reduction_factor": ADAPTER_REDUCTION_FACTOR, "non_linearity": NON_LINEARITY, "bottleneck_dim": bottleneck_dim, "num_adapter_pairs": num_layers, "total_adapters": num_layers * 2, "backbone_dtype": str(BACKBONE_DTYPE), "grad_checkpointing": True, "max_length": MAX_LENGTH, "effective_batch": BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS, "trainable_params": trainable, "trainable_pct": round(pct, 6), "best_epoch": best_epoch, **{f"final_{k}": round(v, 6) if isinstance(v, float) else v for k, v in final_metrics.items()}}])
    results_df.to_csv(os.path.join(SAVE_DIR, "final_results.csv"), index=False)
    print(f"\nArtifacts saved to : {SAVE_DIR}/")
    print(f"  final_results.csv")
    print(f"  training_history.csv")
    print(f"  best_adapter_model.pt")
if __name__ == "__main__":
    main()

# LoRA Qwen2.5-Coder-1.5B, 1024token

In [ ]:
import warnings
import os
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix,
)
warnings.filterwarnings('ignore')
os.environ["CUDA_VISIBLE_DEVICES"]    = "0,1,2"
os.environ["TOKENIZERS_PARALLELISM"]  = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
MODEL_NAME     = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
TRAIN_CSV      = '/code vul/trqw.csv'
TEST_CSV       = 'code vul/teqw.csv'
MAX_LENGTH     = 1024
LORA_R         = 8
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.1
TARGET_MODULES = ["q_proj", "v_proj"]
NUM_EPOCHS     = 5
BATCH_SIZE     = 4
GRAD_ACCUM     = 2
LEARNING_RATE  = 3e-4
WEIGHT_DECAY   = 0.1
WARMUP_RATIO   = 0.06
VAL_SPLIT      = 0.1
SEED           = 42
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
set_seed(SEED)
class LoRALinear(nn.Module):
    def __init__(self, base_linear: nn.Linear, r: int, lora_alpha: int, lora_dropout: float):
        super().__init__()
        assert isinstance(base_linear, nn.Linear)
        self.base_linear  = base_linear
        in_features       = base_linear.in_features
        out_features      = base_linear.out_features
        self.r            = r
        self.scaling      = lora_alpha / r
        self.lora_dropout = nn.Dropout(p=lora_dropout) if lora_dropout > 0.0 else nn.Identity()
        self.lora_A       = nn.Parameter(torch.empty((r, in_features),   dtype=torch.float32))
        self.lora_B       = nn.Parameter(torch.empty((out_features, r),  dtype=torch.float32))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)
        self.base_linear.weight.requires_grad_(False)
        if self.base_linear.bias is not None:
            self.base_linear.bias.requires_grad_(False)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = self.base_linear(x)
        lora_A   = self.lora_A.to(dtype=x.dtype)
        lora_B   = self.lora_B.to(dtype=x.dtype)
        lora_out = (self.lora_dropout(x) @ lora_A.t() @ lora_B.t()) * self.scaling
        return base_out + lora_out
def inject_lora(model: nn.Module, target_modules: list, r: int, lora_alpha: int, lora_dropout: float) -> int:
    replaced = 0
    for name, module in list(model.named_modules()):
        for target in target_modules:
            if name.endswith(target) and isinstance(module, nn.Linear):
                parts  = name.split('.')
                parent = model
                for part in parts[:-1]:
                    parent = getattr(parent, part)
                setattr(parent, parts[-1], LoRALinear(module, r=r, lora_alpha=lora_alpha, lora_dropout=lora_dropout))
                replaced += 1
    return replaced
def freeze_base_and_mark_lora(model: nn.Module):
    for name, param in model.named_parameters():
        if 'lora_A' in name or 'lora_B' in name:
            param.data = param.data.to(torch.float32)
            param.requires_grad_(True)
        elif 'score' in name:
            param.data = param.data.to(torch.float32)
            param.requires_grad_(True)
        else:
            param.requires_grad_(False)
def register_score_fp32_hook(model: nn.Module):
    def _cast_input_to_fp32(module, args):
        return tuple(t.to(torch.float32) if isinstance(t, torch.Tensor) else t for t in args)
    model.score.register_forward_pre_hook(_cast_input_to_fp32)
def count_parameters(model: nn.Module):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    return trainable, total
class VulnDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(
            texts,
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx],
        }
class ModelWrapper(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.model = base_model
    def forward(self, input_ids, attention_mask):
        out = self.model(input_ids=input_ids, attention_mask=attention_mask)
        return out.logits
def compute_metrics(labels, preds, probs):
    cm = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel()
    return {
        'Accuracy':      accuracy_score(labels, preds),
        'Balanced_Acc':  balanced_accuracy_score(labels, preds),
        'Precision':     precision_score(labels, preds, zero_division=0),
        'Recall':        recall_score(labels, preds, zero_division=0),
        'F1':            f1_score(labels, preds, zero_division=0),
        'ROC_AUC':       roc_auc_score(labels, probs),
        'Avg_Precision': average_precision_score(labels, probs),
        'MCC':           matthews_corrcoef(labels, preds),
        'Specificity':   tn / (tn + fp + 1e-9),
        'Sensitivity':   tp / (tp + fn + 1e-9),
        'FPR':           fp / (fp + tn + 1e-9),
        'FNR':           fn / (fn + tp + 1e-9),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn),
    }
def run_eval(model, loader, criterion, device):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device, non_blocking=True)
            attention_mask = batch['attention_mask'].to(device, non_blocking=True)
            labels         = batch['labels'].to(device, non_blocking=True)
            logits         = model(input_ids, attention_mask)
            loss           = criterion(logits, labels)
            total_loss    += loss.item()
            probs          = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            preds          = logits.argmax(dim=-1).cpu().numpy()
            all_probs.extend(probs.tolist())
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
    return (
        total_loss / len(loader),
        np.array(all_labels),
        np.array(all_preds),
        np.array(all_probs),
    )
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)
train_df, val_df = train_test_split(
    train_df,
    test_size=VAL_SPLIT,
    random_state=SEED,
    stratify=train_df['label'],
)
print(f"Train size : {len(train_df)}")
print(f"Val size   : {len(val_df)}")
print(f"Test size  : {len(test_df)}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
train_dataset = VulnDataset(train_df['func_cleaned'].tolist(), train_df['label'].tolist(), tokenizer)
val_dataset   = VulnDataset(val_df['func_cleaned'].tolist(),   val_df['label'].tolist(),   tokenizer)
test_dataset  = VulnDataset(test_df['func_cleaned'].tolist(),  test_df['label'].tolist(),  tokenizer)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True, drop_last=False)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpus = torch.cuda.device_count()
print(f"Primary device : {device}")
print(f"Visible GPUs   : {n_gpus}")
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    free, total_mem = torch.cuda.mem_get_info(i)
    print(f"  GPU {i} — {props.name} | Free: {free/1024**3:.1f} GB / {total_mem/1024**3:.1f} GB")
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    trust_remote_code=True,
    pad_token_id=tokenizer.pad_token_id,
    torch_dtype=torch.float16,
)
base_model.config.use_cache = False
n_replaced = inject_lora(base_model, TARGET_MODULES, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT)
print(f"LoRA injected into {n_replaced} linear projections {TARGET_MODULES}")
freeze_base_and_mark_lora(base_model)
register_score_fp32_hook(base_model)
trainable_params, total_params = count_parameters(base_model)
print(f"Trainable parameters : {trainable_params:,}")
print(f"Total parameters     : {total_params:,}")
print(f"Trainable percentage : {100.0 * trainable_params / total_params:.4f}%")
model = ModelWrapper(base_model).to(device)
if n_gpus > 1:
    model = nn.DataParallel(model)
    print(f"DataParallel across {n_gpus} GPUs | Per-GPU batch = {BATCH_SIZE}")
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=WEIGHT_DECAY,
)
effective_batch = BATCH_SIZE * GRAD_ACCUM * max(1, n_gpus)
total_steps     = (len(train_loader) // GRAD_ACCUM) * NUM_EPOCHS
warmup_steps    = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)
criterion    = nn.CrossEntropyLoss()
train_losses = []
val_losses   = []
print(f"\nEffective batch size : {effective_batch}")
print(f"Total optim steps    : {total_steps}")
print(f"Warmup steps         : {warmup_steps}")
print(f"Starting training for {NUM_EPOCHS} epochs ...")
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    accum_loss = 0.0
    optimizer.zero_grad()
    for step, batch in enumerate(train_loader, 1):
        input_ids      = batch['input_ids'].to(device, non_blocking=True)
        attention_mask = batch['attention_mask'].to(device, non_blocking=True)
        labels         = batch['labels'].to(device, non_blocking=True)
        logits = model(input_ids, attention_mask)
        loss   = criterion(logits, labels) / GRAD_ACCUM
        loss.backward()
        accum_loss += loss.item()
        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                max_norm=1.0,
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            epoch_loss += accum_loss
            accum_loss  = 0.0
        if step % max(1, len(train_loader) // 5) == 0:
            print(f"  Epoch {epoch} | Step {step}/{len(train_loader)} | Loss: {loss.item() * GRAD_ACCUM:.4f}")
    avg_train = epoch_loss / (len(train_loader) / GRAD_ACCUM)
    avg_val, _, _, _ = run_eval(model, val_loader, criterion, device)
    train_losses.append(avg_train)
    val_losses.append(avg_val)
    print(f"Epoch {epoch}/{NUM_EPOCHS} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")
epochs_range = list(range(1, NUM_EPOCHS + 1))
plt.figure(figsize=(10, 6))
plt.plot(epochs_range, train_losses, 'b-o', linewidth=2, markersize=6, label='Train Loss')
plt.plot(epochs_range, val_losses,   'r-o', linewidth=2, markersize=6, label='Val Loss')
plt.xlabel('Epoch', fontsize=13)
plt.ylabel('Loss', fontsize=13)
plt.title('Train vs Validation Loss per Epoch\n(Scratch LoRA — Qwen2.5-Coder-1.5B)', fontsize=13)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.xticks(epochs_range)
plt.tight_layout()
plt.savefig('train_val_loss.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nLoss plot saved to train_val_loss.png")
print("\nRunning evaluation on test set ...")
_, test_labels, test_preds, test_probs = run_eval(model, test_loader, criterion, device)
metrics = compute_metrics(test_labels, test_preds, test_probs)
print("\n" + "=" * 50)
print("         TEST SET RESULTS")
print("=" * 50)
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k:<20s}: {v:.4f}")
    else:
        print(f"  {k:<20s}: {v}")
print("=" * 50)

# LoRA codet5+ 220, 1024 token

In [ ]:
import warnings
import os
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, matthews_corrcoef, confusion_matrix,
)
warnings.filterwarnings('ignore')
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
MODEL_NAME      = "Salesforce/codet5p-220m"
TRAIN_CSV                   = "/code vul/trqw.csv"
TEST_CSV                    = "/code vul/teqw.csv"
MAX_LENGTH = 1024
LORA_R = 8
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
TARGET_MODULES = ["q", "v"]
NUM_EPOCHS = 5
BATCH_SIZE = 4
GRAD_ACCUM = 2
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
WARMUP_RATIO = 0.06
VAL_SPLIT = 0.1
SEED = 42
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(SEED)
class LoRALinear(nn.Module):
    def __init__(self, base_linear: nn.Linear, r: int, lora_alpha: int, lora_dropout: float):
        super().__init__()
        assert isinstance(base_linear, nn.Linear)
        self.base_linear = base_linear
        in_features = base_linear.in_features
        out_features = base_linear.out_features
        self.r = r
        self.scaling = lora_alpha / r
        self.lora_dropout = nn.Dropout(p=lora_dropout) if lora_dropout > 0.0 else nn.Identity()
        self.lora_A = nn.Parameter(torch.empty((r, in_features), dtype=torch.float32))
        self.lora_B = nn.Parameter(torch.empty((out_features, r), dtype=torch.float32))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)
        self.base_linear.weight.requires_grad_(False)
        if self.base_linear.bias is not None:
            self.base_linear.bias.requires_grad_(False)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = self.base_linear(x)
        lora_A = self.lora_A.to(dtype=x.dtype)
        lora_B = self.lora_B.to(dtype=x.dtype)
        lora_out = (self.lora_dropout(x) @ lora_A.t() @ lora_B.t()) * self.scaling
        return base_out + lora_out
def inject_lora(model: nn.Module, target_modules: list, r: int, lora_alpha: int, lora_dropout: float) -> int:
    replaced = 0
    for name, module in list(model.named_modules()):
        for target in target_modules:
            if name.endswith(target) and isinstance(module, nn.Linear):
                parts = name.split('.')
                parent = model
                for part in parts[:-1]:
                    parent = getattr(parent, part)
                setattr(parent, parts[-1], LoRALinear(module, r=r, lora_alpha=lora_alpha, lora_dropout=lora_dropout))
                replaced += 1
    return replaced
def freeze_base_and_mark_lora(model: nn.Module):
    for name, param in model.named_parameters():
        if 'lora_A' in name or 'lora_B' in name:
            param.data = param.data.to(torch.float32)
            param.requires_grad_(True)
        elif 'classifier' in name:
            param.data = param.data.to(torch.float32)
            param.requires_grad_(True)
        else:
            param.requires_grad_(False)
def register_classifier_fp32_hook(classifier_layer: nn.Linear):
    def _cast_input_to_fp32(module, args):
        return tuple(t.to(torch.float32) if isinstance(t, torch.Tensor) else t for t in args)
    classifier_layer.register_forward_pre_hook(_cast_input_to_fp32)
def count_parameters(model: nn.Module):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total
class VulnDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(
            texts,
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx],
        }
class CodeT5pClassifier(nn.Module):
    def __init__(self, encoder, hidden_size: int, num_labels: int = 2):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Linear(hidden_size, num_labels, bias=True)
        nn.init.trunc_normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)
    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        encoder_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden = encoder_out.last_hidden_state
        mask_expanded = attention_mask.unsqueeze(-1).to(dtype=hidden.dtype)
        pooled = (hidden * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1e-9)
        pooled = pooled.to(torch.float32)
        return self.classifier(pooled)
def compute_metrics(labels, preds, probs):
    cm = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel()
    return {
        'Accuracy': accuracy_score(labels, preds),
        'Balanced_Acc': balanced_accuracy_score(labels, preds),
        'Precision': precision_score(labels, preds, zero_division=0),
        'Recall': recall_score(labels, preds, zero_division=0),
        'F1': f1_score(labels, preds, zero_division=0),
        'ROC_AUC': roc_auc_score(labels, probs),
        'Avg_Precision': average_precision_score(labels, probs),
        'MCC': matthews_corrcoef(labels, preds),
        'Specificity': tn / (tn + fp + 1e-9),
        'Sensitivity': tp / (tp + fn + 1e-9),
        'FPR': fp / (fp + tn + 1e-9),
        'FNR': fn / (fn + tp + 1e-9),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn),
    }
def run_eval(model, loader, criterion, device):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device, non_blocking=True)
            attention_mask = batch['attention_mask'].to(device, non_blocking=True)
            labels = batch['labels'].to(device, non_blocking=True)
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            preds = logits.argmax(dim=-1).cpu().numpy()
            all_probs.extend(probs.tolist())
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
    return (
        total_loss / len(loader),
        np.array(all_labels),
        np.array(all_preds),
        np.array(all_probs),
    )
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
train_df, val_df = train_test_split(
    train_df, test_size=VAL_SPLIT, random_state=SEED, stratify=train_df['label'],
)
print(f"Train size : {len(train_df)}")
print(f"Val size   : {len(val_df)}")
print(f"Test size  : {len(test_df)}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
train_dataset = VulnDataset(train_df['func_cleaned'].tolist(), train_df['label'].tolist(), tokenizer)
val_dataset = VulnDataset(val_df['func_cleaned'].tolist(), val_df['label'].tolist(), tokenizer)
test_dataset = VulnDataset(test_df['func_cleaned'].tolist(), test_df['label'].tolist(), tokenizer)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpus = torch.cuda.device_count()
print(f"Primary device : {device}")
print(f"Visible GPUs   : {n_gpus}")
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    free, total_mem = torch.cuda.mem_get_info(i)
    print(f"  GPU {i} — {props.name} | Free: {free/1024**3:.1f} GB / {total_mem/1024**3:.1f} GB")
seq2seq_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
seq2seq_model.config.use_cache = False
encoder = seq2seq_model.encoder
hidden_size = seq2seq_model.config.d_model
n_replaced = inject_lora(encoder, TARGET_MODULES, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT)
print(f"LoRA injected into {n_replaced} linear projections {TARGET_MODULES}")
base_model = CodeT5pClassifier(encoder, hidden_size=hidden_size, num_labels=2)
freeze_base_and_mark_lora(base_model)
register_classifier_fp32_hook(base_model.classifier)
trainable_params, total_params = count_parameters(base_model)
print(f"Trainable parameters : {trainable_params:,}")
print(f"Total parameters     : {total_params:,}")
print(f"Trainable percentage : {100.0 * trainable_params / total_params:.4f}%")
model = base_model.to(device)
if n_gpus > 1:
    model = nn.DataParallel(model)
    print(f"DataParallel across {n_gpus} GPUs | Per-GPU batch = {BATCH_SIZE}")
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=WEIGHT_DECAY,
)
effective_batch = BATCH_SIZE * GRAD_ACCUM * max(1, n_gpus)
total_steps = (len(train_loader) // GRAD_ACCUM) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)
criterion = nn.CrossEntropyLoss()
train_losses = []
val_losses = []
print(f"\nEffective batch size : {effective_batch}")
print(f"Total optim steps    : {total_steps}")
print(f"Warmup steps         : {warmup_steps}")
print(f"Starting training for {NUM_EPOCHS} epochs ...")
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    accum_loss = 0.0
    optimizer.zero_grad()
    for step, batch in enumerate(train_loader, 1):
        input_ids = batch['input_ids'].to(device, non_blocking=True)
        attention_mask = batch['attention_mask'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels) / GRAD_ACCUM
        loss.backward()
        accum_loss += loss.item()
        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                max_norm=1.0,
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            epoch_loss += accum_loss
            accum_loss = 0.0
        if step % max(1, len(train_loader) // 5) == 0:
            print(f"  Epoch {epoch} | Step {step}/{len(train_loader)} | Loss: {loss.item() * GRAD_ACCUM:.4f}")
    avg_train = epoch_loss / (len(train_loader) / GRAD_ACCUM)
    avg_val, _, _, _ = run_eval(model, val_loader, criterion, device)
    train_losses.append(avg_train)
    val_losses.append(avg_val)
    print(f"Epoch {epoch}/{NUM_EPOCHS} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")
epochs_range = list(range(1, NUM_EPOCHS + 1))
plt.figure(figsize=(10, 6))
plt.plot(epochs_range, train_losses, 'b-o', linewidth=2, markersize=6, label='Train Loss')
plt.plot(epochs_range, val_losses, 'r-o', linewidth=2, markersize=6, label='Val Loss')
plt.xlabel('Epoch', fontsize=13)
plt.ylabel('Loss', fontsize=13)
plt.title('Train vs Validation Loss per Epoch\n(Scratch LoRA — CodeT5p-770m)', fontsize=13)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.xticks(epochs_range)
plt.tight_layout()
plt.savefig('train_val_loss.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nLoss plot saved to train_val_loss.png")
print("\nRunning evaluation on test set ...")
_, test_labels, test_preds, test_probs = run_eval(model, test_loader, criterion, device)
metrics = compute_metrics(test_labels, test_preds, test_probs)
print("\n" + "=" * 50)
print("         TEST SET RESULTS")
print("=" * 50)

for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k:<20s}: {v:.4f}")
    else:
        print(f"  {k:<20s}: {v}")
        
print("=" * 50)

# Lora 3B codequen codevul 1024 token

In [ ]:
import warnings
import os
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, matthews_corrcoef, confusion_matrix,
)
warnings.filterwarnings('ignore')
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"
TRAIN_CSV       = 'code vul/trqw.csv'
TEST_CSV        =  '/code vul/teqw.csv'
MAX_LENGTH = 512
LORA_R = 8
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
TARGET_MODULES = ["q_proj", "v_proj"]
NUM_EPOCHS = 5
BATCH_SIZE = 4
GRAD_ACCUM = 4
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
WARMUP_RATIO = 0.06
VAL_SPLIT = 0.1
SEED = 42
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(SEED)
class LoRALinear(nn.Module):
    def __init__(self, base_linear: nn.Linear, r: int, lora_alpha: int, lora_dropout: float):
        super().__init__()
        assert isinstance(base_linear, nn.Linear)
        self.base_linear = base_linear
        in_features = base_linear.in_features
        out_features = base_linear.out_features
        self.r = r
        self.scaling = lora_alpha / r
        self.lora_dropout = nn.Dropout(p=lora_dropout) if lora_dropout > 0.0 else nn.Identity()
        self.lora_A = nn.Parameter(torch.empty((r, in_features), dtype=torch.float32))
        self.lora_B = nn.Parameter(torch.empty((out_features, r), dtype=torch.float32))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)
        self.base_linear.weight.requires_grad_(False)
        if self.base_linear.bias is not None:
            self.base_linear.bias.requires_grad_(False)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = self.base_linear(x)
        lora_A = self.lora_A.to(device=x.device, dtype=x.dtype)
        lora_B = self.lora_B.to(device=x.device, dtype=x.dtype)
        lora_out = (self.lora_dropout(x) @ lora_A.t() @ lora_B.t()) * self.scaling
        return base_out + lora_out
def inject_lora(model: nn.Module, target_modules: list, r: int, lora_alpha: int, lora_dropout: float) -> int:
    replaced = 0
    for name, module in list(model.named_modules()):
        for target in target_modules:
            if name.endswith(target) and isinstance(module, nn.Linear):
                parts = name.split('.')
                parent = model
                for part in parts[:-1]:
                    parent = getattr(parent, part)
                setattr(parent, parts[-1], LoRALinear(module, r=r, lora_alpha=lora_alpha, lora_dropout=lora_dropout))
                replaced += 1
    return replaced
def sync_lora_to_device(model: nn.Module):
    for module in model.modules():
        if isinstance(module, LoRALinear):
            target_device = module.base_linear.weight.device
            module.lora_A.data = module.lora_A.data.to(device=target_device, dtype=torch.float32)
            module.lora_B.data = module.lora_B.data.to(device=target_device, dtype=torch.float32)
def freeze_base_and_mark_lora(model: nn.Module):
    for name, param in model.named_parameters():
        if 'lora_A' in name or 'lora_B' in name:
            param.requires_grad_(True)
        elif 'score' in name:
            param.data = param.data.to(torch.float32)
            param.requires_grad_(True)
        else:
            param.requires_grad_(False)
def register_score_fp32_hook(model: nn.Module):
    def _cast_input_to_fp32(module, args):
        return tuple(t.to(torch.float32) if isinstance(t, torch.Tensor) else t for t in args)
    model.score.register_forward_pre_hook(_cast_input_to_fp32)
def count_parameters(model: nn.Module):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total
class VulnDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(
            texts,
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx],
        }
def compute_metrics(labels, preds, probs):
    cm = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel()
    return {
        'Accuracy': accuracy_score(labels, preds),
        'Balanced_Acc': balanced_accuracy_score(labels, preds),
        'Precision': precision_score(labels, preds, zero_division=0),
        'Recall': recall_score(labels, preds, zero_division=0),
        'F1': f1_score(labels, preds, zero_division=0),
        'ROC_AUC': roc_auc_score(labels, probs),
        'Avg_Precision': average_precision_score(labels, probs),
        'MCC': matthews_corrcoef(labels, preds),
        'Specificity': tn / (tn + fp + 1e-9),
        'Sensitivity': tp / (tp + fn + 1e-9),
        'FPR': fp / (fp + tn + 1e-9),
        'FNR': fn / (fn + tp + 1e-9),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn),
    }
def run_eval(model, loader, criterion, input_device, output_device):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(input_device, non_blocking=True)
            attention_mask = batch['attention_mask'].to(input_device, non_blocking=True)
            labels = batch['labels'].to(output_device, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                out = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = out.logits.float().to(output_device)
                loss = criterion(logits, labels)
            total_loss += loss.item()
            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            preds = logits.argmax(dim=-1).cpu().numpy()
            all_probs.extend(probs.tolist())
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
    return (
        total_loss / len(loader),
        np.array(all_labels),
        np.array(all_preds),
        np.array(all_probs),
    )
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
train_df, val_df = train_test_split(
    train_df, test_size=VAL_SPLIT, random_state=SEED, stratify=train_df['label'],
)
print(f"Train size : {len(train_df)}")
print(f"Val size   : {len(val_df)}")
print(f"Test size  : {len(test_df)}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
train_dataset = VulnDataset(train_df['func_cleaned'].tolist(), train_df['label'].tolist(), tokenizer)
val_dataset = VulnDataset(val_df['func_cleaned'].tolist(), val_df['label'].tolist(), tokenizer)
test_dataset = VulnDataset(test_df['func_cleaned'].tolist(), test_df['label'].tolist(), tokenizer)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
n_gpus = torch.cuda.device_count()
input_device = torch.device("cuda:0")
print(f"Visible GPUs   : {n_gpus}")
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    free, total_mem = torch.cuda.mem_get_info(i)
    print(f"  GPU {i} — {props.name} | Free: {free/1024**3:.1f} GB / {total_mem/1024**3:.1f} GB")
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    trust_remote_code=True,
    pad_token_id=tokenizer.pad_token_id,
    torch_dtype=torch.float16,
    device_map="balanced",
)
base_model.config.use_cache = False
if hasattr(base_model, 'gradient_checkpointing_enable'):
    base_model.gradient_checkpointing_enable()
    print("Gradient checkpointing enabled")
n_replaced = inject_lora(base_model, TARGET_MODULES, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT)
print(f"LoRA injected into {n_replaced} linear projections {TARGET_MODULES}")
sync_lora_to_device(base_model)
freeze_base_and_mark_lora(base_model)
register_score_fp32_hook(base_model)
output_device = next(base_model.score.parameters()).device
print(f"Input device  : {input_device}")
print(f"Output device : {output_device}")
trainable_params, total_params = count_parameters(base_model)
print(f"Trainable parameters : {trainable_params:,}")
print(f"Total parameters     : {total_params:,}")
print(f"Trainable percentage : {100.0 * trainable_params / total_params:.4f}%")
optimizer = torch.optim.AdamW(
    [p for p in base_model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=WEIGHT_DECAY,
)
effective_batch = BATCH_SIZE * GRAD_ACCUM
total_steps = (len(train_loader) // GRAD_ACCUM) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)
scaler = torch.cuda.amp.GradScaler()
criterion = nn.CrossEntropyLoss()
train_losses = []
val_losses = []
print(f"\nEffective batch size : {effective_batch}")
print(f"Total optim steps    : {total_steps}")
print(f"Warmup steps         : {warmup_steps}")
print(f"Starting training for {NUM_EPOCHS} epochs ...")
for epoch in range(1, NUM_EPOCHS + 1):
    base_model.train()
    epoch_loss = 0.0
    accum_loss = 0.0
    optimizer.zero_grad()
    for step, batch in enumerate(train_loader, 1):
        input_ids = batch['input_ids'].to(input_device, non_blocking=True)
        attention_mask = batch['attention_mask'].to(input_device, non_blocking=True)
        labels = batch['labels'].to(output_device, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            out = base_model(input_ids=input_ids, attention_mask=attention_mask)
            logits = out.logits.float().to(output_device)
            loss = criterion(logits, labels) / GRAD_ACCUM
        scaler.scale(loss).backward()
        accum_loss += loss.item()
        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in base_model.parameters() if p.requires_grad],
                max_norm=1.0,
            )
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            torch.cuda.empty_cache()
            epoch_loss += accum_loss
            accum_loss = 0.0
        if step % max(1, len(train_loader) // 5) == 0:
            print(f"  Epoch {epoch} | Step {step}/{len(train_loader)} | Loss: {loss.item() * GRAD_ACCUM:.4f}")
    avg_train = epoch_loss / (len(train_loader) / GRAD_ACCUM)
    avg_val, _, _, _ = run_eval(base_model, val_loader, criterion, input_device, output_device)
    train_losses.append(avg_train)
    val_losses.append(avg_val)
    print(f"Epoch {epoch}/{NUM_EPOCHS} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")
    torch.cuda.empty_cache()
epochs_range = list(range(1, NUM_EPOCHS + 1))
plt.figure(figsize=(10, 6))
plt.plot(epochs_range, train_losses, 'b-o', linewidth=2, markersize=6, label='Train Loss')
plt.plot(epochs_range, val_losses, 'r-o', linewidth=2, markersize=6, label='Val Loss')
plt.xlabel('Epoch', fontsize=13)
plt.ylabel('Loss', fontsize=13)
plt.title('Train vs Validation Loss per Epoch\n(Scratch LoRA — Qwen2.5-Coder-3B-Instruct)', fontsize=13)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.xticks(epochs_range)
plt.tight_layout()
plt.savefig('train_val_loss.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nLoss plot saved to train_val_loss.png")
print("\nRunning evaluation on test set ...")
_, test_labels, test_preds, test_probs = run_eval(base_model, test_loader, criterion, input_device, output_device)
metrics = compute_metrics(test_labels, test_preds, test_probs)
print("\n" + "=" * 50)
print("         TEST SET RESULTS")
print("=" * 50)
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k:<20s}: {v:.4f}")
    else:
        print(f"  {k:<20s}: {v}")
print("=" * 50)

# Lora 1024 token codet5 770m, code vul

In [ ]:
import warnings
import os
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, matthews_corrcoef, confusion_matrix,
)
warnings.filterwarnings('ignore')
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
MODEL_NAME = "Salesforce/codet5p-770m"
TRAIN_CSV     = "code vul/trqw.csv"
TEST_CSV      = "code vul/teqw.csv"
MAX_LENGTH = 1024
LORA_R = 8
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
TARGET_MODULES = ["q", "v"]
NUM_EPOCHS = 5
BATCH_SIZE = 4
GRAD_ACCUM = 2
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
WARMUP_RATIO = 0.06
VAL_SPLIT = 0.1
SEED = 42
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(SEED)
class LoRALinear(nn.Module):
    def __init__(self, base_linear: nn.Linear, r: int, lora_alpha: int, lora_dropout: float):
        super().__init__()
        assert isinstance(base_linear, nn.Linear)
        self.base_linear = base_linear
        in_features = base_linear.in_features
        out_features = base_linear.out_features
        self.r = r
        self.scaling = lora_alpha / r
        self.lora_dropout = nn.Dropout(p=lora_dropout) if lora_dropout > 0.0 else nn.Identity()
        self.lora_A = nn.Parameter(torch.empty((r, in_features), dtype=torch.float32))
        self.lora_B = nn.Parameter(torch.empty((out_features, r), dtype=torch.float32))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)
        self.base_linear.weight.requires_grad_(False)
        if self.base_linear.bias is not None:
            self.base_linear.bias.requires_grad_(False)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = self.base_linear(x)
        lora_A = self.lora_A.to(dtype=x.dtype)
        lora_B = self.lora_B.to(dtype=x.dtype)
        lora_out = (self.lora_dropout(x) @ lora_A.t() @ lora_B.t()) * self.scaling
        return base_out + lora_out
def inject_lora(model: nn.Module, target_modules: list, r: int, lora_alpha: int, lora_dropout: float) -> int:
    replaced = 0
    for name, module in list(model.named_modules()):
        for target in target_modules:
            if name.endswith(target) and isinstance(module, nn.Linear):
                parts = name.split('.')
                parent = model
                for part in parts[:-1]:
                    parent = getattr(parent, part)
                setattr(parent, parts[-1], LoRALinear(module, r=r, lora_alpha=lora_alpha, lora_dropout=lora_dropout))
                replaced += 1
    return replaced
def freeze_base_and_mark_lora(model: nn.Module):
    for name, param in model.named_parameters():
        if 'lora_A' in name or 'lora_B' in name:
            param.data = param.data.to(torch.float32)
            param.requires_grad_(True)
        elif 'classifier' in name:
            param.data = param.data.to(torch.float32)
            param.requires_grad_(True)
        else:
            param.requires_grad_(False)
def register_classifier_fp32_hook(classifier_layer: nn.Linear):
    def _cast_input_to_fp32(module, args):
        return tuple(t.to(torch.float32) if isinstance(t, torch.Tensor) else t for t in args)
    classifier_layer.register_forward_pre_hook(_cast_input_to_fp32)
def count_parameters(model: nn.Module):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total
class VulnDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(
            texts,
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx],
        }
class CodeT5pClassifier(nn.Module):
    def __init__(self, encoder, hidden_size: int, num_labels: int = 2):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Linear(hidden_size, num_labels, bias=True)
        nn.init.trunc_normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)
    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        encoder_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden = encoder_out.last_hidden_state
        mask_expanded = attention_mask.unsqueeze(-1).to(dtype=hidden.dtype)
        pooled = (hidden * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1e-9)
        pooled = pooled.to(torch.float32)
        return self.classifier(pooled)
def compute_metrics(labels, preds, probs):
    cm = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel()
    return {
        'Accuracy': accuracy_score(labels, preds),
        'Balanced_Acc': balanced_accuracy_score(labels, preds),
        'Precision': precision_score(labels, preds, zero_division=0),
        'Recall': recall_score(labels, preds, zero_division=0),
        'F1': f1_score(labels, preds, zero_division=0),
        'ROC_AUC': roc_auc_score(labels, probs),
        'Avg_Precision': average_precision_score(labels, probs),
        'MCC': matthews_corrcoef(labels, preds),
        'Specificity': tn / (tn + fp + 1e-9),
        'Sensitivity': tp / (tp + fn + 1e-9),
        'FPR': fp / (fp + tn + 1e-9),
        'FNR': fn / (fn + tp + 1e-9),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn),
    }
def run_eval(model, loader, criterion, device):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device, non_blocking=True)
            attention_mask = batch['attention_mask'].to(device, non_blocking=True)
            labels = batch['labels'].to(device, non_blocking=True)
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            preds = logits.argmax(dim=-1).cpu().numpy()
            all_probs.extend(probs.tolist())
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
    return (
        total_loss / len(loader),
        np.array(all_labels),
        np.array(all_preds),
        np.array(all_probs),
    )
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
train_df, val_df = train_test_split(
    train_df, test_size=VAL_SPLIT, random_state=SEED, stratify=train_df['label'],
)
print(f"Train size : {len(train_df)}")
print(f"Val size   : {len(val_df)}")
print(f"Test size  : {len(test_df)}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
train_dataset = VulnDataset(train_df['func_cleaned'].tolist(), train_df['label'].tolist(), tokenizer)
val_dataset = VulnDataset(val_df['func_cleaned'].tolist(), val_df['label'].tolist(), tokenizer)
test_dataset = VulnDataset(test_df['func_cleaned'].tolist(), test_df['label'].tolist(), tokenizer)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpus = torch.cuda.device_count()
print(f"Primary device : {device}")
print(f"Visible GPUs   : {n_gpus}")
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    free, total_mem = torch.cuda.mem_get_info(i)
    print(f"  GPU {i} — {props.name} | Free: {free/1024**3:.1f} GB / {total_mem/1024**3:.1f} GB")
seq2seq_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
seq2seq_model.config.use_cache = False
encoder = seq2seq_model.encoder
hidden_size = seq2seq_model.config.d_model
n_replaced = inject_lora(encoder, TARGET_MODULES, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT)
print(f"LoRA injected into {n_replaced} linear projections {TARGET_MODULES}")
base_model = CodeT5pClassifier(encoder, hidden_size=hidden_size, num_labels=2)
freeze_base_and_mark_lora(base_model)
register_classifier_fp32_hook(base_model.classifier)
trainable_params, total_params = count_parameters(base_model)
print(f"Trainable parameters : {trainable_params:,}")
print(f"Total parameters     : {total_params:,}")
print(f"Trainable percentage : {100.0 * trainable_params / total_params:.4f}%")
model = base_model.to(device)
if n_gpus > 1:
    model = nn.DataParallel(model)
    print(f"DataParallel across {n_gpus} GPUs | Per-GPU batch = {BATCH_SIZE}")
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=WEIGHT_DECAY,
)
effective_batch = BATCH_SIZE * GRAD_ACCUM * max(1, n_gpus)
total_steps = (len(train_loader) // GRAD_ACCUM) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)
criterion = nn.CrossEntropyLoss()
train_losses = []
val_losses = []
print(f"\nEffective batch size : {effective_batch}")
print(f"Total optim steps    : {total_steps}")
print(f"Warmup steps         : {warmup_steps}")
print(f"Starting training for {NUM_EPOCHS} epochs ...")
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    accum_loss = 0.0
    optimizer.zero_grad()
    for step, batch in enumerate(train_loader, 1):
        input_ids = batch['input_ids'].to(device, non_blocking=True)
        attention_mask = batch['attention_mask'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels) / GRAD_ACCUM
        loss.backward()
        accum_loss += loss.item()
        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                max_norm=1.0,
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            epoch_loss += accum_loss
            accum_loss = 0.0
        if step % max(1, len(train_loader) // 5) == 0:
            print(f"  Epoch {epoch} | Step {step}/{len(train_loader)} | Loss: {loss.item() * GRAD_ACCUM:.4f}")
    avg_train = epoch_loss / (len(train_loader) / GRAD_ACCUM)
    avg_val, _, _, _ = run_eval(model, val_loader, criterion, device)
    train_losses.append(avg_train)
    val_losses.append(avg_val)
    print(f"Epoch {epoch}/{NUM_EPOCHS} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")
epochs_range = list(range(1, NUM_EPOCHS + 1))
plt.figure(figsize=(10, 6))
plt.plot(epochs_range, train_losses, 'b-o', linewidth=2, markersize=6, label='Train Loss')
plt.plot(epochs_range, val_losses, 'r-o', linewidth=2, markersize=6, label='Val Loss')
plt.xlabel('Epoch', fontsize=13)
plt.ylabel('Loss', fontsize=13)
plt.title('Train vs Validation Loss per Epoch\n(Scratch LoRA — CodeT5p-770m)', fontsize=13)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.xticks(epochs_range)
plt.tight_layout()
plt.savefig('train_val_loss.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nLoss plot saved to train_val_loss.png")
print("\nRunning evaluation on test set ...")
_, test_labels, test_preds, test_probs = run_eval(model, test_loader, criterion, device)
metrics = compute_metrics(test_labels, test_preds, test_probs)
print("\n" + "=" * 50)
print("         TEST SET RESULTS")
print("=" * 50)

for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k:<20s}: {v:.4f}")
    else:
        print(f"  {k:<20s}: {v}")
        
print("=" * 50)

# Prefix Qwen2.5-Coder-1.5B 1024token 

In [ ]:
import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from transformers.cache_utils import DynamicCache
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix,
    balanced_accuracy_score, average_precision_score
)
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings


warnings.filterwarnings('ignore')
os.environ["CUDA_VISIBLE_DEVICES"]     = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"]   = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

MODEL_NAME      = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
TRAIN_CSV       = '/code vul/trqw.csv'
TEST_CSV        = '/code vul/teqw.csv'

MAX_LENGTH      = 1024
PRE_SEQ_LEN     = 100
MID_DIM         = 512
BATCH_SIZE      = 8
GRAD_ACCUM      = 3
EPOCHS          = 30
LR              = 5e-5
WEIGHT_DECAY    = 0.0
WARMUP_RATIO    = 0.15
VAL_SPLIT       = 0.20
SEED            = 42
SWA_START_RATIO = 0.6
PATIENCE        = 7
NUM_WORKERS     = 4
PREFIX_DROPOUT  = 0.0
PRIMARY_DEVICE  = torch.device("cuda:0")

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


class CodeDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.codes     = df['func_cleaned'].astype(str).tolist()
        self.labels    = df['label'].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.codes[idx],
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


class PrefixEncoder(nn.Module):


    def __init__(self, pre_seq_len, n_layers, n_kv_heads, head_dim, mid_dim, dropout_rate):
        super().__init__()
        self.pre_seq_len = pre_seq_len
        self.n_layers    = n_layers
        self.n_kv_heads  = n_kv_heads
        self.head_dim    = head_dim
        total_kv_dim     = n_layers * 2 * n_kv_heads * head_dim
        self.register_buffer('prefix_tokens', torch.arange(pre_seq_len).long())
        self.embedding      = nn.Embedding(pre_seq_len, mid_dim)
        self.trans          = nn.Sequential(
            nn.Linear(mid_dim, mid_dim),
            nn.Tanh(),
            nn.Linear(mid_dim, total_kv_dim)
        )
        self.prefix_dropout = nn.Dropout(dropout_rate)

    def forward(self, batch_size):
        prefix_tokens   = self.prefix_tokens.unsqueeze(0).expand(batch_size, -1)
        past_key_values = self.embedding(prefix_tokens)
        past_key_values = self.trans(past_key_values)
        past_key_values = self.prefix_dropout(past_key_values)
        past_key_values = past_key_values.view(
            batch_size, self.pre_seq_len,
            self.n_layers * 2, self.n_kv_heads, self.head_dim
        )
        past_key_values = past_key_values.permute(2, 0, 3, 1, 4).contiguous()
        past_key_values = past_key_values.split(2, dim=0)
        return tuple((pkv[0], pkv[1]) for pkv in past_key_values)


def _resolve_layer_devices(backbone, n_layers):

    layer_list = None
    candidate  = getattr(backbone, 'layers', None)
    if candidate is not None and hasattr(candidate, '__getitem__'):
        layer_list = candidate
    else:
        sub = getattr(backbone, 'model', None)
        if sub is not None:
            candidate = getattr(sub, 'layers', None)
            if candidate is not None and hasattr(candidate, '__getitem__'):
                layer_list = candidate
    devices = []
    for i in range(n_layers):
        try:
            device = next(layer_list[i].parameters()).device
        except Exception:
            device = PRIMARY_DEVICE
        devices.append(device)
    return devices


def _build_prefix_cache(prefix_kvs_fp32, layer_devices):

    cache = DynamicCache()
    for layer_idx, (k, v) in enumerate(prefix_kvs_fp32):
        device = layer_devices[layer_idx]
        cache.update(
            k.to(device=device, dtype=torch.float16),
            v.to(device=device, dtype=torch.float16),
            layer_idx
        )
    return cache


class PrefixTuningClassifier(nn.Module):


    def __init__(self, model_name, pre_seq_len, mid_dim, num_labels=2):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(
            model_name,
            device_map='auto',
            torch_dtype=torch.float16,
            trust_remote_code=True
        )
        for param in self.backbone.parameters():
            param.requires_grad = False

        cfg         = self.backbone.config
        n_layers    = cfg.num_hidden_layers
        n_heads     = cfg.num_attention_heads
        n_kv_heads  = getattr(cfg, 'num_key_value_heads', n_heads)
        hidden_size = cfg.hidden_size
        head_dim    = getattr(cfg, 'head_dim', hidden_size // n_heads)

        self.pre_seq_len    = pre_seq_len
        self.hidden_size    = hidden_size
        self._layer_devices = _resolve_layer_devices(self.backbone, n_layers)
        self._first_device  = self._layer_devices[0]

        self.prefix_encoder = PrefixEncoder(
            pre_seq_len  = pre_seq_len,
            n_layers     = n_layers,
            n_kv_heads   = n_kv_heads,
            head_dim     = head_dim,
            mid_dim      = mid_dim,
            dropout_rate = PREFIX_DROPOUT
        ).to(PRIMARY_DEVICE)

        self.classifier = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_labels)
        ).to(PRIMARY_DEVICE)

    def forward(self, input_ids, attention_mask):
        batch_size = input_ids.size(0)

        input_ids      = input_ids.to(self._first_device)
        attention_mask = attention_mask.to(self._first_device)

        prefix_kvs_fp32 = self.prefix_encoder(batch_size)
        past_kv         = _build_prefix_cache(prefix_kvs_fp32, self._layer_devices)

        prefix_mask = torch.ones(
            batch_size, self.pre_seq_len,
            dtype=attention_mask.dtype,
            device=self._first_device
        )
        ext_mask = torch.cat([prefix_mask, attention_mask], dim=1)

        outputs = self.backbone(
            input_ids       = input_ids,
            attention_mask  = ext_mask,
            past_key_values = past_kv,
            use_cache       = True,
            return_dict     = True
        )

        last_hs  = outputs.last_hidden_state.to(dtype=torch.float32, device=PRIMARY_DEVICE)
        seq_lens = attention_mask.sum(dim=1).long() - 1
        pooled   = last_hs[
            torch.arange(batch_size, device=PRIMARY_DEVICE),
            seq_lens.to(PRIMARY_DEVICE)
        ]
        return self.classifier(pooled)


def compute_metrics(labels, preds, probs):
    labels = np.array(labels)
    preds  = np.array(preds)
    probs  = np.array(probs)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        'Accuracy':      accuracy_score(labels, preds),
        'Balanced_Acc':  balanced_accuracy_score(labels, preds),
        'Precision':     precision_score(labels, preds, zero_division=0),
        'Recall':        recall_score(labels, preds, zero_division=0),
        'F1':            f1_score(labels, preds, zero_division=0),
        'ROC_AUC':       roc_auc_score(labels, probs),
        'Avg_Precision': average_precision_score(labels, probs),
        'MCC':           matthews_corrcoef(labels, preds),
        'Specificity':   tn / (tn + fp + 1e-9),
        'Sensitivity':   tp / (tp + fn + 1e-9),
        'FPR':           fp / (fp + tn + 1e-9),
        'FNR':           fn / (fn + tp + 1e-9),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn)
    }


def train_epoch(model, loader, criterion, optimizer, scheduler, scaler, step_counter):
    model.train()
    total_loss                       = 0.0
    all_labels, all_preds, all_probs = [], [], []
    optimizer.zero_grad()

    for batch_idx, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(PRIMARY_DEVICE)
        attention_mask = batch['attention_mask'].to(PRIMARY_DEVICE)
        labels         = batch['label'].to(PRIMARY_DEVICE)

        logits = model(input_ids, attention_mask)
        loss   = criterion(logits.float(), labels) / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (batch_idx + 1) % GRAD_ACCUM == 0 or (batch_idx + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0
            )
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            step_counter[0] += 1

        total_loss += (loss * GRAD_ACCUM).item()

        with torch.no_grad():
            logits_f = logits.float()
            probs    = torch.softmax(logits_f, dim=-1)[:, 1].cpu().numpy()
            preds    = logits_f.argmax(dim=-1).cpu().numpy()

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)

    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss                       = 0.0
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(PRIMARY_DEVICE)
            attention_mask = batch['attention_mask'].to(PRIMARY_DEVICE)
            labels         = batch['label'].to(PRIMARY_DEVICE)

            logits     = model(input_ids, attention_mask)
            logits_f   = logits.float()
            loss       = criterion(logits_f, labels)
            total_loss += loss.item()

            probs = torch.softmax(logits_f, dim=-1)[:, 1].cpu().numpy()
            preds = logits_f.argmax(dim=-1).cpu().numpy()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds)
            all_probs.extend(probs)

    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)


def get_trainable_state(model):
    return {
        'prefix_encoder': {k: v.cpu().clone() for k, v in model.prefix_encoder.state_dict().items()},
        'classifier':     {k: v.cpu().clone() for k, v in model.classifier.state_dict().items()}
    }


def accumulate_swa_state(swa_acc, model):
    src = get_trainable_state(model)
    if swa_acc is None:
        return copy.deepcopy(src)
    for module_key in ('prefix_encoder', 'classifier'):
        for k in swa_acc[module_key]:
            swa_acc[module_key][k] = swa_acc[module_key][k] + src[module_key][k]
    return swa_acc


def apply_swa_avg_state(swa_acc, n_swa):
    for module_key in ('prefix_encoder', 'classifier'):
        for k in swa_acc[module_key]:
            swa_acc[module_key][k] = swa_acc[module_key][k].float().div_(n_swa)
    return swa_acc


def load_trainable_state(model, state):
    model.prefix_encoder.load_state_dict(
        {k: v.to(PRIMARY_DEVICE) for k, v in state['prefix_encoder'].items()}
    )
    model.classifier.load_state_dict(
        {k: v.to(PRIMARY_DEVICE) for k, v in state['classifier'].items()}
    )


def plot_loss_curve(history, save_path='pt_loss_curve.png'):
    epochs = list(range(1, len(history['tr_loss']) + 1))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(epochs, history['tr_loss'], 'b-o', markersize=4, linewidth=1.8, label='Train Loss')
    ax.plot(epochs, history['va_loss'], 'r-o', markersize=4, linewidth=1.8, label='Val Loss')
    ax.set_title(
        'Prefix-Tuning (Qwen2.5-Coder-1.5B) — Train vs Validation Loss',
        fontsize=13, fontweight='bold'
    )
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Loss curve saved to {save_path}")


def plot_training_curves(history, save_path='pt_training_curves.png'):
    epochs = list(range(1, len(history['tr_loss']) + 1))
    fig, axes = plt.subplots(3, 3, figsize=(18, 14))
    fig.suptitle(
        'Prefix-Tuning (Qwen2.5-Coder-1.5B) Training Curves',
        fontsize=16, fontweight='bold'
    )

    def plot_ax(ax, key, title, ylabel):
        ax.plot(epochs, history[f'tr_{key}'], 'b-o', markersize=3, label='Train')
        ax.plot(epochs, history[f'va_{key}'], 'r-o', markersize=3, label='Val')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(True, alpha=0.3)

    plot_ax(axes[0][0], 'loss',    'Loss',          'Loss')
    plot_ax(axes[0][1], 'acc',     'Accuracy',      'Accuracy')
    plot_ax(axes[0][2], 'f1',      'F1 Score',      'F1')
    plot_ax(axes[1][0], 'auc',     'ROC AUC',       'AUC')
    plot_ax(axes[1][1], 'mcc',     'MCC',           'MCC')
    plot_ax(axes[1][2], 'prec',    'Precision',     'Precision')
    plot_ax(axes[2][0], 'rec',     'Recall',        'Recall')
    plot_ax(axes[2][1], 'bacc',    'Balanced Acc',  'Balanced Accuracy')
    plot_ax(axes[2][2], 'avgprec', 'Avg Precision', 'Avg Precision')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Training curves saved to {save_path}")


def plot_final_comparison(best_test_m, swa_test_m, save_path='pt_test_comparison.png'):
    keys      = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Avg_Precision']
    x         = np.arange(len(keys))
    w         = 0.35
    best_vals = [best_test_m[k] for k in keys]
    swa_vals  = [swa_test_m[k] for k in keys] if swa_test_m else None
    fig, ax   = plt.subplots(figsize=(14, 6))
    bars1     = ax.bar(x - w / 2, best_vals, w, label='Best Checkpoint', color='steelblue', alpha=0.85)
    if swa_vals:
        bars2 = ax.bar(x + w / 2, swa_vals, w, label='SWA Model', color='darkorange', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(keys, rotation=20, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title('Test Set: Best Checkpoint vs SWA Model', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
    if swa_vals:
        for bar in bars2:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Test comparison chart saved to {save_path}")


def plot_confusion_matrix(m, title, save_path):
    cm      = np.array([[m['TN'], m['FP']], [m['FN'], m['TP']]])
    fig, ax = plt.subplots(figsize=(5, 4))
    im      = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred 0', 'Pred 1'])
    ax.set_yticklabels(['True 0', 'True 1'])
    thresh = cm.max() / 2.0
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > thresh else 'black',
                    fontsize=14, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Confusion matrix saved to {save_path}")


def plot_metrics_radar(best_test_m, swa_test_m, save_path='pt_radar_chart.png'):
    keys   = ['Accuracy', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Specificity', 'Sensitivity']
    n      = len(keys)
    angles = [x / float(n) * 2 * np.pi for x in range(n)]
    angles += angles[:1]
    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

    def draw(m, color, label):
        vals  = [m[k] for k in keys]
        vals += vals[:1]
        ax.plot(angles, vals, color=color, linewidth=2, label=label)
        ax.fill(angles, vals, color=color, alpha=0.15)

    draw(best_test_m, 'steelblue', 'Best Checkpoint')
    if swa_test_m:
        draw(swa_test_m, 'darkorange', 'SWA Model')
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(keys, size=9)
    ax.set_ylim(0, 1)
    ax.set_title('Test Metrics Radar', fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Radar chart saved to {save_path}")


def main():
    n_gpus = torch.cuda.device_count()
    print(f"Using {n_gpus} GPU(s): {[torch.cuda.get_device_name(i) for i in range(n_gpus)]}")

    train_full = pd.read_csv(TRAIN_CSV)
    test_full  = pd.read_csv(TEST_CSV)

    train_df, val_df = train_test_split(
        train_full, test_size=VAL_SPLIT, random_state=SEED, stratify=train_full['label']
    )
    _, test_df = train_test_split(
        test_full, test_size=0.50, random_state=SEED, stratify=test_full['label']
    )
    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}\n")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = 'right'

    train_ds     = CodeDataset(train_df.reset_index(drop=True), tokenizer)
    val_ds       = CodeDataset(val_df.reset_index(drop=True),   tokenizer)
    test_ds      = CodeDataset(test_df.reset_index(drop=True),  tokenizer)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=False, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=False)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=False)

    counts        = train_df['label'].value_counts().sort_index().values
    raw_w         = torch.tensor([1.0 / c for c in counts], dtype=torch.float).to(PRIMARY_DEVICE)
    class_weights = raw_w / raw_w.sum() * len(counts)

    print("Loading backbone with device_map='auto' to shard across all available GPUs...")
    model = PrefixTuningClassifier(
        model_name  = MODEL_NAME,
        pre_seq_len = PRE_SEQ_LEN,
        mid_dim     = MID_DIM,
        num_labels  = 2
    )
    print(f"Backbone layer device map: {[str(d) for d in model._layer_devices]}\n")

    total_p   = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Params     : {total_p:,}")
    print(f"Trainable Params : {trainable:,}")
    print(f"Trainable %      : {100.0 * trainable / total_p:.4f}%\n")

    criterion        = nn.CrossEntropyLoss(weight=class_weights)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer        = torch.optim.AdamW(
        trainable_params, lr=LR, weight_decay=WEIGHT_DECAY,
        betas=(0.9, 0.999), eps=1e-8
    )
    effective_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    warmup_steps    = int(WARMUP_RATIO * effective_steps)
    scheduler       = get_cosine_schedule_with_warmup(optimizer, warmup_steps, effective_steps)
    scaler          = torch.cuda.amp.GradScaler()

    swa_start_epoch = int(SWA_START_RATIO * EPOCHS)
    swa_acc         = None
    n_swa           = 0
    step_counter    = [0]
    best_val_f1     = -1.0
    best_epoch      = 0
    best_state      = None
    no_improve      = 0

    history = {
        'tr_loss': [], 'tr_acc': [], 'tr_f1': [], 'tr_auc': [], 'tr_mcc': [],
        'tr_prec': [], 'tr_rec': [], 'tr_bacc': [], 'tr_avgprec': [],
        'va_loss': [], 'va_acc': [], 'va_f1': [], 'va_auc': [], 'va_mcc': [],
        'va_prec': [], 'va_rec': [], 'va_bacc': [], 'va_avgprec': []
    }

    hdr = (f"{'Ep':>3} | {'TrLoss':>8} | {'TrAcc':>7} | {'TrF1':>7} | {'TrAUC':>7} | {'TrMCC':>7} | "
           f"{'VaLoss':>8} | {'VaAcc':>7} | {'VaF1':>7} | {'VaAUC':>7} | {'VaMCC':>7} | {'Note':<12}")
    sep = "-" * len(hdr)
    print(sep)
    print(hdr)
    print(sep)

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_m = train_epoch(
            model, train_loader, criterion, optimizer, scheduler, scaler, step_counter
        )
        va_loss, va_m = eval_epoch(model, val_loader, criterion)

        history['tr_loss'].append(tr_loss)
        history['tr_acc'].append(tr_m['Accuracy'])
        history['tr_f1'].append(tr_m['F1'])
        history['tr_auc'].append(tr_m['ROC_AUC'])
        history['tr_mcc'].append(tr_m['MCC'])
        history['tr_prec'].append(tr_m['Precision'])
        history['tr_rec'].append(tr_m['Recall'])
        history['tr_bacc'].append(tr_m['Balanced_Acc'])
        history['tr_avgprec'].append(tr_m['Avg_Precision'])
        history['va_loss'].append(va_loss)
        history['va_acc'].append(va_m['Accuracy'])
        history['va_f1'].append(va_m['F1'])
        history['va_auc'].append(va_m['ROC_AUC'])
        history['va_mcc'].append(va_m['MCC'])
        history['va_prec'].append(va_m['Precision'])
        history['va_rec'].append(va_m['Recall'])
        history['va_bacc'].append(va_m['Balanced_Acc'])
        history['va_avgprec'].append(va_m['Avg_Precision'])

        note = ""
        if epoch > swa_start_epoch:
            swa_acc = accumulate_swa_state(swa_acc, model)
            n_swa  += 1
            note   += "SWA "

        is_best = va_m['F1'] > best_val_f1
        if is_best:
            best_val_f1 = va_m['F1']
            best_epoch  = epoch
            best_state  = get_trainable_state(model)
            no_improve  = 0
            note       += "<--"
        else:
            no_improve += 1

        print(
            f"{epoch:>3} | {tr_loss:>8.4f} | {tr_m['Accuracy']:>7.4f} | {tr_m['F1']:>7.4f} | "
            f"{tr_m['ROC_AUC']:>7.4f} | {tr_m['MCC']:>7.4f} | "
            f"{va_loss:>8.4f} | {va_m['Accuracy']:>7.4f} | {va_m['F1']:>7.4f} | "
            f"{va_m['ROC_AUC']:>7.4f} | {va_m['MCC']:>7.4f} | {note:<12}"
        )

        if no_improve >= PATIENCE:
            print(f"\n  Early stopping at epoch {epoch} (no val F1 improvement for {PATIENCE} epochs)")
            break

    print(sep)
    print(f"\nBest Epoch: {best_epoch}  |  Best Val F1: {best_val_f1:.4f}")

    print("\n--- Evaluating: Best Checkpoint Model ---")
    load_trainable_state(model, best_state)
    _, best_test_m = eval_epoch(model, test_loader, criterion)

    print("\n--- Evaluating: SWA Model ---")
    swa_test_m = None
    if n_swa > 0:
        swa_avg_state = apply_swa_avg_state(swa_acc, n_swa)
        load_trainable_state(model, swa_avg_state)
        _, swa_test_m = eval_epoch(model, test_loader, criterion)
        load_trainable_state(model, best_state)

    print(f"\n{'='*60}")
    print(f"  Val F1 trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_f1'][-5:]]}")
    print(f"  Val Loss trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_loss'][-5:]]}")
    print(f"\n{'='*60}")
    print(f"  TEST RESULTS — Best Checkpoint (Epoch {best_epoch})")
    print(f"{'='*60}")
    scalar_keys = [k for k in best_test_m if k not in ('TP', 'TN', 'FP', 'FN')]
    for k in scalar_keys:
        print(f"  {k:<22}: {best_test_m[k]:.4f}")
    print(f"  {'TP':<22}: {best_test_m['TP']}")
    print(f"  {'TN':<22}: {best_test_m['TN']}")
    print(f"  {'FP':<22}: {best_test_m['FP']}")
    print(f"  {'FN':<22}: {best_test_m['FN']}")

    if swa_test_m is not None:
        print(f"\n{'='*60}")
        print(f"  TEST RESULTS — SWA Model (averaged over last {n_swa} epochs)")
        print(f"{'='*60}")
        for k in scalar_keys:
            print(f"  {k:<22}: {swa_test_m[k]:.4f}")
        print(f"  {'TP':<22}: {swa_test_m['TP']}")
        print(f"  {'TN':<22}: {swa_test_m['TN']}")
        print(f"  {'FP':<22}: {swa_test_m['FP']}")
        print(f"  {'FN':<22}: {swa_test_m['FN']}")
        print(f"\n{'='*60}")
        print(f"  COMPARISON: Best Checkpoint vs SWA")
        print(f"{'='*60}")
        compare_keys = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall']
        print(f"  {'Metric':<22} {'BestCkpt':>12} {'SWA':>12} {'Delta':>12}")
        print(f"  {'-'*58}")
        for k in compare_keys:
            delta  = swa_test_m[k] - best_test_m[k]
            symbol = "+" if delta >= 0 else ""
            print(f"  {k:<22} {best_test_m[k]:>12.4f} {swa_test_m[k]:>12.4f} {symbol+f'{delta:.4f}':>12}")

    plot_loss_curve(history,      save_path='pt_loss_curve.png')
    plot_training_curves(history, save_path='pt_training_curves.png')
    plot_final_comparison(best_test_m, swa_test_m, save_path='pt_test_comparison.png')
    plot_confusion_matrix(best_test_m,
                          f'Confusion Matrix — Best Ckpt (Ep {best_epoch})',
                          'pt_cm_best.png')
    if swa_test_m:
        plot_confusion_matrix(swa_test_m,
                              f'Confusion Matrix — SWA ({n_swa} epochs)',
                              'pt_cm_swa.png')
    plot_metrics_radar(best_test_m, swa_test_m, save_path='pt_radar_chart.png')
    print("\nAll figures saved.")


if __name__ == "__main__":
    main()

# prefix tubing Qwen2.5-Coder-3b 1024 token code vul

In [ ]:
import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from transformers.cache_utils import DynamicCache
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix,
    balanced_accuracy_score, average_precision_score
)
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"
TRAIN_CSV       = '/code vul/trqw.csv'
TEST_CSV        =  '/code vul/teqw.csv'
MAX_LENGTH = 512
PRE_SEQ_LEN = 100
MID_DIM = 512
BATCH_SIZE = 8
GRAD_ACCUM = 3
EPOCHS = 30
LR = 5e-5
WEIGHT_DECAY = 0.0
WARMUP_RATIO = 0.15
VAL_SPLIT = 0.20
SEED = 42
SWA_START_RATIO = 0.6
PATIENCE = 7
NUM_WORKERS = 4
PREFIX_DROPOUT = 0.0
PRIMARY_DEVICE = torch.device("cuda:0")
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
class CodeDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.codes = df['code'].astype(str).tolist()
        self.labels = df['label'].tolist()
        self.tokenizer = tokenizer
    def __len__(self):
        return len(self.codes)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.codes[idx],
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }
class PrefixEncoder(nn.Module):
    def __init__(self, pre_seq_len, n_layers, n_kv_heads, head_dim, mid_dim, dropout_rate):
        super().__init__()
        self.pre_seq_len = pre_seq_len
        self.n_layers = n_layers
        self.n_kv_heads = n_kv_heads
        self.head_dim = head_dim
        total_kv_dim = n_layers * 2 * n_kv_heads * head_dim
        self.register_buffer('prefix_tokens', torch.arange(pre_seq_len).long())
        self.embedding = nn.Embedding(pre_seq_len, mid_dim)
        self.trans = nn.Sequential(
            nn.Linear(mid_dim, mid_dim),
            nn.Tanh(),
            nn.Linear(mid_dim, total_kv_dim)
        )
        self.prefix_dropout = nn.Dropout(dropout_rate)
    def forward(self, batch_size):
        prefix_tokens = self.prefix_tokens.unsqueeze(0).expand(batch_size, -1)
        past_key_values = self.embedding(prefix_tokens)
        past_key_values = self.trans(past_key_values)
        past_key_values = self.prefix_dropout(past_key_values)
        past_key_values = past_key_values.view(
            batch_size, self.pre_seq_len,
            self.n_layers * 2, self.n_kv_heads, self.head_dim
        )
        past_key_values = past_key_values.permute(2, 0, 3, 1, 4).contiguous()
        past_key_values = past_key_values.split(2, dim=0)
        return tuple((pkv[0], pkv[1]) for pkv in past_key_values)
def _resolve_layer_devices(backbone, n_layers):
    layer_list = None
    candidate = getattr(backbone, 'layers', None)
    if candidate is not None and hasattr(candidate, '__getitem__'):
        layer_list = candidate
    else:
        sub = getattr(backbone, 'model', None)
        if sub is not None:
            candidate = getattr(sub, 'layers', None)
            if candidate is not None and hasattr(candidate, '__getitem__'):
                layer_list = candidate
    devices = []
    for i in range(n_layers):
        try:
            device = next(layer_list[i].parameters()).device
        except Exception:
            device = PRIMARY_DEVICE
        devices.append(device)
    return devices
def _build_prefix_cache(prefix_kvs_fp32, layer_devices):
    cache = DynamicCache()
    for layer_idx, (k, v) in enumerate(prefix_kvs_fp32):
        device = layer_devices[layer_idx]
        cache.update(
            k.to(device=device, dtype=torch.float16),
            v.to(device=device, dtype=torch.float16),
            layer_idx
        )
    return cache
class PrefixTuningClassifier(nn.Module):
    def __init__(self, model_name, pre_seq_len, mid_dim, num_labels=2):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(
            model_name,
            device_map='auto',
            torch_dtype=torch.float16,
            trust_remote_code=True
        )
        for param in self.backbone.parameters():
            param.requires_grad = False
        cfg = self.backbone.config
        n_layers = cfg.num_hidden_layers
        n_heads = cfg.num_attention_heads
        n_kv_heads = getattr(cfg, 'num_key_value_heads', n_heads)
        hidden_size = cfg.hidden_size
        head_dim = getattr(cfg, 'head_dim', hidden_size // n_heads)
        self.pre_seq_len = pre_seq_len
        self.hidden_size = hidden_size
        self._layer_devices = _resolve_layer_devices(self.backbone, n_layers)
        self._first_device = self._layer_devices[0]
        self.prefix_encoder = PrefixEncoder(
            pre_seq_len=pre_seq_len,
            n_layers=n_layers,
            n_kv_heads=n_kv_heads,
            head_dim=head_dim,
            mid_dim=mid_dim,
            dropout_rate=PREFIX_DROPOUT
        ).to(PRIMARY_DEVICE)
        self.classifier = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_labels)
        ).to(PRIMARY_DEVICE)
    def forward(self, input_ids, attention_mask):
        batch_size = input_ids.size(0)
        input_ids = input_ids.to(self._first_device)
        attention_mask = attention_mask.to(self._first_device)
        prefix_kvs_fp32 = self.prefix_encoder(batch_size)
        past_kv = _build_prefix_cache(prefix_kvs_fp32, self._layer_devices)
        prefix_mask = torch.ones(
            batch_size, self.pre_seq_len,
            dtype=attention_mask.dtype,
            device=self._first_device
        )
        ext_mask = torch.cat([prefix_mask, attention_mask], dim=1)
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=ext_mask,
            past_key_values=past_kv,
            use_cache=True,
            return_dict=True
        )
        last_hs = outputs.last_hidden_state.to(dtype=torch.float32, device=PRIMARY_DEVICE)
        seq_lens = attention_mask.sum(dim=1).long() - 1
        pooled = last_hs[
            torch.arange(batch_size, device=PRIMARY_DEVICE),
            seq_lens.to(PRIMARY_DEVICE)
        ]
        return self.classifier(pooled)
def compute_metrics(labels, preds, probs):
    labels = np.array(labels)
    preds = np.array(preds)
    probs = np.array(probs)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        'Accuracy': accuracy_score(labels, preds),
        'Balanced_Acc': balanced_accuracy_score(labels, preds),
        'Precision': precision_score(labels, preds, zero_division=0),
        'Recall': recall_score(labels, preds, zero_division=0),
        'F1': f1_score(labels, preds, zero_division=0),
        'ROC_AUC': roc_auc_score(labels, probs),
        'Avg_Precision': average_precision_score(labels, probs),
        'MCC': matthews_corrcoef(labels, preds),
        'Specificity': tn / (tn + fp + 1e-9),
        'Sensitivity': tp / (tp + fn + 1e-9),
        'FPR': fp / (fp + tn + 1e-9),
        'FNR': fn / (fn + tp + 1e-9),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn)
    }
def train_epoch(model, loader, criterion, optimizer, scheduler, scaler, step_counter):
    model.train()
    total_loss = 0.0
    all_labels, all_preds, all_probs = [], [], []
    optimizer.zero_grad()
    for batch_idx, batch in enumerate(loader):
        input_ids = batch['input_ids'].to(PRIMARY_DEVICE)
        attention_mask = batch['attention_mask'].to(PRIMARY_DEVICE)
        labels = batch['label'].to(PRIMARY_DEVICE)
        logits = model(input_ids, attention_mask)
        loss = criterion(logits.float(), labels) / GRAD_ACCUM
        scaler.scale(loss).backward()
        if (batch_idx + 1) % GRAD_ACCUM == 0 or (batch_idx + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0
            )
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            step_counter[0] += 1
        total_loss += (loss * GRAD_ACCUM).item()
        with torch.no_grad():
            logits_f = logits.float()
            probs = torch.softmax(logits_f, dim=-1)[:, 1].cpu().numpy()
            preds = logits_f.argmax(dim=-1).cpu().numpy()
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)
    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(PRIMARY_DEVICE)
            attention_mask = batch['attention_mask'].to(PRIMARY_DEVICE)
            labels = batch['label'].to(PRIMARY_DEVICE)
            logits = model(input_ids, attention_mask)
            logits_f = logits.float()
            loss = criterion(logits_f, labels)
            total_loss += loss.item()
            probs = torch.softmax(logits_f, dim=-1)[:, 1].cpu().numpy()
            preds = logits_f.argmax(dim=-1).cpu().numpy()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds)
            all_probs.extend(probs)
    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)
def get_trainable_state(model):
    return {
        'prefix_encoder': {k: v.cpu().clone() for k, v in model.prefix_encoder.state_dict().items()},
        'classifier': {k: v.cpu().clone() for k, v in model.classifier.state_dict().items()}
    }
def accumulate_swa_state(swa_acc, model):
    src = get_trainable_state(model)
    if swa_acc is None:
        return copy.deepcopy(src)
    for module_key in ('prefix_encoder', 'classifier'):
        for k in swa_acc[module_key]:
            swa_acc[module_key][k] = swa_acc[module_key][k] + src[module_key][k]
    return swa_acc
def apply_swa_avg_state(swa_acc, n_swa):
    for module_key in ('prefix_encoder', 'classifier'):
        for k in swa_acc[module_key]:
            swa_acc[module_key][k] = swa_acc[module_key][k].float().div_(n_swa)
    return swa_acc
def load_trainable_state(model, state):
    model.prefix_encoder.load_state_dict(
        {k: v.to(PRIMARY_DEVICE) for k, v in state['prefix_encoder'].items()}
    )
    model.classifier.load_state_dict(
        {k: v.to(PRIMARY_DEVICE) for k, v in state['classifier'].items()}
    )
def plot_loss_curve(history, save_path='pt_loss_curve.png'):
    epochs = list(range(1, len(history['tr_loss']) + 1))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(epochs, history['tr_loss'], 'b-o', markersize=4, linewidth=1.8, label='Train Loss')
    ax.plot(epochs, history['va_loss'], 'r-o', markersize=4, linewidth=1.8, label='Val Loss')
    ax.set_title('Prefix-Tuning (Qwen2.5-Coder-3B) — Train vs Validation Loss', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Loss curve saved to {save_path}")
def plot_training_curves(history, save_path='pt_training_curves.png'):
    epochs = list(range(1, len(history['tr_loss']) + 1))
    fig, axes = plt.subplots(3, 3, figsize=(18, 14))
    fig.suptitle('Prefix-Tuning (Qwen2.5-Coder-3B) Training Curves', fontsize=16, fontweight='bold')
    def plot_ax(ax, key, title, ylabel):
        ax.plot(epochs, history[f'tr_{key}'], 'b-o', markersize=3, label='Train')
        ax.plot(epochs, history[f'va_{key}'], 'r-o', markersize=3, label='Val')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(True, alpha=0.3)
    plot_ax(axes[0][0], 'loss', 'Loss', 'Loss')
    plot_ax(axes[0][1], 'acc', 'Accuracy', 'Accuracy')
    plot_ax(axes[0][2], 'f1', 'F1 Score', 'F1')
    plot_ax(axes[1][0], 'auc', 'ROC AUC', 'AUC')
    plot_ax(axes[1][1], 'mcc', 'MCC', 'MCC')
    plot_ax(axes[1][2], 'prec', 'Precision', 'Precision')
    plot_ax(axes[2][0], 'rec', 'Recall', 'Recall')
    plot_ax(axes[2][1], 'bacc', 'Balanced Acc', 'Balanced Accuracy')
    plot_ax(axes[2][2], 'avgprec', 'Avg Precision', 'Avg Precision')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Training curves saved to {save_path}")
def plot_final_comparison(best_test_m, swa_test_m, save_path='pt_test_comparison.png'):
    keys = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Avg_Precision']
    x = np.arange(len(keys))
    w = 0.35
    best_vals = [best_test_m[k] for k in keys]
    swa_vals = [swa_test_m[k] for k in keys] if swa_test_m else None
    fig, ax = plt.subplots(figsize=(14, 6))
    bars1 = ax.bar(x - w / 2, best_vals, w, label='Best Checkpoint', color='steelblue', alpha=0.85)
    if swa_vals:
        bars2 = ax.bar(x + w / 2, swa_vals, w, label='SWA Model', color='darkorange', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(keys, rotation=20, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title('Test Set: Best Checkpoint vs SWA Model', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
    if swa_vals:
        for bar in bars2:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Test comparison chart saved to {save_path}")
def plot_confusion_matrix(m, title, save_path):
    cm = np.array([[m['TN'], m['FP']], [m['FN'], m['TP']]])
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred 0', 'Pred 1'])
    ax.set_yticklabels(['True 0', 'True 1'])
    thresh = cm.max() / 2.0
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > thresh else 'black',
                    fontsize=14, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Confusion matrix saved to {save_path}")
def plot_metrics_radar(best_test_m, swa_test_m, save_path='pt_radar_chart.png'):
    keys = ['Accuracy', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Specificity', 'Sensitivity']
    n = len(keys)
    angles = [x / float(n) * 2 * np.pi for x in range(n)]
    angles += angles[:1]
    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    def draw(m, color, label):
        vals = [m[k] for k in keys]
        vals += vals[:1]
        ax.plot(angles, vals, color=color, linewidth=2, label=label)
        ax.fill(angles, vals, color=color, alpha=0.15)
    draw(best_test_m, 'steelblue', 'Best Checkpoint')
    if swa_test_m:
        draw(swa_test_m, 'darkorange', 'SWA Model')
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(keys, size=9)
    ax.set_ylim(0, 1)
    ax.set_title('Test Metrics Radar', fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Radar chart saved to {save_path}")
def main():
    n_gpus = torch.cuda.device_count()
    print(f"Using {n_gpus} GPU(s): {[torch.cuda.get_device_name(i) for i in range(n_gpus)]}")
    train_full = pd.read_csv(TRAIN_CSV)
    test_full = pd.read_csv(TEST_CSV)
    train_df, val_df = train_test_split(
        train_full, test_size=VAL_SPLIT, random_state=SEED, stratify=train_full['label']
    )
    _, test_df = train_test_split(
        test_full, test_size=0.50, random_state=SEED, stratify=test_full['label']
    )
    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}\n")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = 'right'
    train_ds = CodeDataset(train_df.reset_index(drop=True), tokenizer)
    val_ds = CodeDataset(val_df.reset_index(drop=True), tokenizer)
    test_ds = CodeDataset(test_df.reset_index(drop=True), tokenizer)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=False, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=False)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=False)
    counts = train_df['label'].value_counts().sort_index().values
    raw_w = torch.tensor([1.0 / c for c in counts], dtype=torch.float).to(PRIMARY_DEVICE)
    class_weights = raw_w / raw_w.sum() * len(counts)
    print("Loading backbone with device_map='auto' to shard across all available GPUs...")
    model = PrefixTuningClassifier(
        model_name=MODEL_NAME,
        pre_seq_len=PRE_SEQ_LEN,
        mid_dim=MID_DIM,
        num_labels=2
    )
    print(f"Backbone layer device map: {[str(d) for d in model._layer_devices]}\n")
    total_p = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Params     : {total_p:,}")
    print(f"Trainable Params : {trainable:,}")
    print(f"Trainable %      : {100.0 * trainable / total_p:.4f}%\n")
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        trainable_params, lr=LR, weight_decay=WEIGHT_DECAY,
        betas=(0.9, 0.999), eps=1e-8
    )
    effective_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    warmup_steps = int(WARMUP_RATIO * effective_steps)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, effective_steps)
    scaler = torch.cuda.amp.GradScaler()
    swa_start_epoch = int(SWA_START_RATIO * EPOCHS)
    swa_acc = None
    n_swa = 0
    step_counter = [0]
    best_val_f1 = -1.0
    best_epoch = 0
    best_state = None
    no_improve = 0
    history = {
        'tr_loss': [], 'tr_acc': [], 'tr_f1': [], 'tr_auc': [], 'tr_mcc': [],
        'tr_prec': [], 'tr_rec': [], 'tr_bacc': [], 'tr_avgprec': [],
        'va_loss': [], 'va_acc': [], 'va_f1': [], 'va_auc': [], 'va_mcc': [],
        'va_prec': [], 'va_rec': [], 'va_bacc': [], 'va_avgprec': []
    }
    hdr = (f"{'Ep':>3} | {'TrLoss':>8} | {'TrAcc':>7} | {'TrF1':>7} | {'TrAUC':>7} | {'TrMCC':>7} | "
           f"{'VaLoss':>8} | {'VaAcc':>7} | {'VaF1':>7} | {'VaAUC':>7} | {'VaMCC':>7} | {'Note':<12}")
    sep = "-" * len(hdr)
    print(sep)
    print(hdr)
    print(sep)
    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_m = train_epoch(
            model, train_loader, criterion, optimizer, scheduler, scaler, step_counter
        )
        va_loss, va_m = eval_epoch(model, val_loader, criterion)
        history['tr_loss'].append(tr_loss)
        history['tr_acc'].append(tr_m['Accuracy'])
        history['tr_f1'].append(tr_m['F1'])
        history['tr_auc'].append(tr_m['ROC_AUC'])
        history['tr_mcc'].append(tr_m['MCC'])
        history['tr_prec'].append(tr_m['Precision'])
        history['tr_rec'].append(tr_m['Recall'])
        history['tr_bacc'].append(tr_m['Balanced_Acc'])
        history['tr_avgprec'].append(tr_m['Avg_Precision'])
        history['va_loss'].append(va_loss)
        history['va_acc'].append(va_m['Accuracy'])
        history['va_f1'].append(va_m['F1'])
        history['va_auc'].append(va_m['ROC_AUC'])
        history['va_mcc'].append(va_m['MCC'])
        history['va_prec'].append(va_m['Precision'])
        history['va_rec'].append(va_m['Recall'])
        history['va_bacc'].append(va_m['Balanced_Acc'])
        history['va_avgprec'].append(va_m['Avg_Precision'])
        note = ""
        if epoch > swa_start_epoch:
            swa_acc = accumulate_swa_state(swa_acc, model)
            n_swa += 1
            note += "SWA "
        is_best = va_m['F1'] > best_val_f1
        if is_best:
            best_val_f1 = va_m['F1']
            best_epoch = epoch
            best_state = get_trainable_state(model)
            no_improve = 0
            note += "<--"
        else:
            no_improve += 1
        print(
            f"{epoch:>3} | {tr_loss:>8.4f} | {tr_m['Accuracy']:>7.4f} | {tr_m['F1']:>7.4f} | "
            f"{tr_m['ROC_AUC']:>7.4f} | {tr_m['MCC']:>7.4f} | "
            f"{va_loss:>8.4f} | {va_m['Accuracy']:>7.4f} | {va_m['F1']:>7.4f} | "
            f"{va_m['ROC_AUC']:>7.4f} | {va_m['MCC']:>7.4f} | {note:<12}"
        )
        if no_improve >= PATIENCE:
            print(f"\n  Early stopping at epoch {epoch} (no val F1 improvement for {PATIENCE} epochs)")
            break
    print(sep)
    print(f"\nBest Epoch: {best_epoch}  |  Best Val F1: {best_val_f1:.4f}")
    print("\n--- Evaluating: Best Checkpoint Model ---")
    load_trainable_state(model, best_state)
    _, best_test_m = eval_epoch(model, test_loader, criterion)
    print("\n--- Evaluating: SWA Model ---")
    swa_test_m = None
    if n_swa > 0:
        swa_avg_state = apply_swa_avg_state(swa_acc, n_swa)
        load_trainable_state(model, swa_avg_state)
        _, swa_test_m = eval_epoch(model, test_loader, criterion)
        load_trainable_state(model, best_state)
    print(f"\n{'='*60}")
    print(f"  Val F1 trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_f1'][-5:]]}")
    print(f"  Val Loss trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_loss'][-5:]]}")
    print(f"\n{'='*60}")
    print(f"  TEST RESULTS — Best Checkpoint (Epoch {best_epoch})")
    print(f"{'='*60}")
    scalar_keys = [k for k in best_test_m if k not in ('TP', 'TN', 'FP', 'FN')]
    for k in scalar_keys:
        print(f"  {k:<22}: {best_test_m[k]:.4f}")
    print(f"  {'TP':<22}: {best_test_m['TP']}")
    print(f"  {'TN':<22}: {best_test_m['TN']}")
    print(f"  {'FP':<22}: {best_test_m['FP']}")
    print(f"  {'FN':<22}: {best_test_m['FN']}")
    if swa_test_m is not None:
        print(f"\n{'='*60}")
        print(f"  TEST RESULTS — SWA Model (averaged over last {n_swa} epochs)")
        print(f"{'='*60}")
        for k in scalar_keys:
            print(f"  {k:<22}: {swa_test_m[k]:.4f}")
        print(f"  {'TP':<22}: {swa_test_m['TP']}")
        print(f"  {'TN':<22}: {swa_test_m['TN']}")
        print(f"  {'FP':<22}: {swa_test_m['FP']}")
        print(f"  {'FN':<22}: {swa_test_m['FN']}")
        print(f"\n{'='*60}")
        print(f"  COMPARISON: Best Checkpoint vs SWA")
        print(f"{'='*60}")
        compare_keys = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall']
        print(f"  {'Metric':<22} {'BestCkpt':>12} {'SWA':>12} {'Delta':>12}")
        print(f"  {'-'*58}")
        for k in compare_keys:
            delta = swa_test_m[k] - best_test_m[k]
            symbol = "+" if delta >= 0 else ""
            print(f"  {k:<22} {best_test_m[k]:>12.4f} {swa_test_m[k]:>12.4f} {symbol+f'{delta:.4f}':>12}")
    plot_loss_curve(history, save_path='pt_loss_curve.png')
    plot_training_curves(history, save_path='pt_training_curves.png')
    plot_final_comparison(best_test_m, swa_test_m, save_path='pt_test_comparison.png')
    plot_confusion_matrix(best_test_m,
                          f'Confusion Matrix — Best Ckpt (Ep {best_epoch})',
                          'pt_cm_best.png')
    if swa_test_m:
        plot_confusion_matrix(swa_test_m,
                              f'Confusion Matrix — SWA ({n_swa} epochs)',
                              'pt_cm_swa.png')
    plot_metrics_radar(best_test_m, swa_test_m, save_path='pt_radar_chart.png')
    print("\nAll figures saved.")
if __name__ == "__main__":
    main()

# prefix 1024 token codet5 220M

In [ ]:
import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_cosine_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix,
    balanced_accuracy_score, average_precision_score
)
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings


warnings.filterwarnings('ignore')
os.environ["CUDA_VISIBLE_DEVICES"]     = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"]   = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

MODEL_NAME      = "Salesforce/codet5p-220m"
TRAIN_CSV                   = "/code vul/trqw.csv"
TEST_CSV                    = "/code vul/teqw.csv"

MAX_LENGTH      = 1024
PRE_SEQ_LEN     = 100
MID_DIM         = 512
BATCH_SIZE      = 8
GRAD_ACCUM      = 3
EPOCHS          = 30
LR              = 5e-5
WEIGHT_DECAY    = 0.0
WARMUP_RATIO    = 0.15
VAL_SPLIT       = 0.20
SEED            = 42
SWA_START_RATIO = 0.6
PATIENCE        = 7
NUM_WORKERS     = 4
PREFIX_DROPOUT  = 0.0
PRIMARY_DEVICE  = torch.device("cuda:0")

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


class CodeDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.codes     = df['func_cleaned'].astype(str).tolist()
        self.labels    = df['label'].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.codes[idx],
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


class PrefixEncoder(nn.Module):
    def __init__(self, pre_seq_len, hidden_size, mid_dim, dropout_rate):
        super().__init__()
        self.pre_seq_len = pre_seq_len
        self.register_buffer('prefix_tokens', torch.arange(pre_seq_len).long())
        self.embedding      = nn.Embedding(pre_seq_len, mid_dim)
        self.trans          = nn.Sequential(
            nn.Linear(mid_dim, mid_dim),
            nn.Tanh(),
            nn.Linear(mid_dim, hidden_size)
        )
        self.prefix_dropout = nn.Dropout(dropout_rate)

    def forward(self, batch_size):
        prefix_tokens   = self.prefix_tokens.unsqueeze(0).expand(batch_size, -1)
        prefix_embeds   = self.embedding(prefix_tokens)
        prefix_embeds   = self.trans(prefix_embeds)
        prefix_embeds   = self.prefix_dropout(prefix_embeds)
        return prefix_embeds


class PrefixTuningClassifier(nn.Module):
    def __init__(self, model_name, pre_seq_len, mid_dim, num_labels=2):
        super().__init__()
        self.backbone = AutoModelForSeq2SeqLM.from_pretrained(
            model_name,
            device_map='auto',
            torch_dtype=torch.float16,
            trust_remote_code=True
        )
        for param in self.backbone.parameters():
            param.requires_grad = False

        cfg         = self.backbone.config
        hidden_size = cfg.d_model

        self.pre_seq_len       = pre_seq_len
        self.hidden_size       = hidden_size
        self._encoder_device   = next(self.backbone.encoder.parameters()).device

        self.prefix_encoder = PrefixEncoder(
            pre_seq_len  = pre_seq_len,
            hidden_size  = hidden_size,
            mid_dim      = mid_dim,
            dropout_rate = PREFIX_DROPOUT
        ).to(PRIMARY_DEVICE)

        self.classifier = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_labels)
        ).to(PRIMARY_DEVICE)

    def forward(self, input_ids, attention_mask):
        batch_size = input_ids.size(0)

        input_ids      = input_ids.to(self._encoder_device)
        attention_mask = attention_mask.to(self._encoder_device)

        prefix_embeds = self.prefix_encoder(batch_size)
        prefix_embeds = prefix_embeds.to(device=self._encoder_device, dtype=torch.float16)

        inputs_embeds = self.backbone.encoder.embed_tokens(input_ids).to(dtype=torch.float16)
        inputs_embeds = torch.cat([prefix_embeds, inputs_embeds], dim=1)

        prefix_mask = torch.ones(
            batch_size, self.pre_seq_len,
            dtype=attention_mask.dtype,
            device=self._encoder_device
        )
        ext_mask = torch.cat([prefix_mask, attention_mask], dim=1)

        encoder_outputs = self.backbone.encoder(
            inputs_embeds  = inputs_embeds,
            attention_mask = ext_mask,
            return_dict    = True
        )

        last_hs  = encoder_outputs.last_hidden_state.to(dtype=torch.float32, device=PRIMARY_DEVICE)
        seq_lens = attention_mask.sum(dim=1).long() - 1 + self.pre_seq_len
        pooled   = last_hs[
            torch.arange(batch_size, device=PRIMARY_DEVICE),
            seq_lens.to(PRIMARY_DEVICE)
        ]
        return self.classifier(pooled)


def compute_metrics(labels, preds, probs):
    labels = np.array(labels)
    preds  = np.array(preds)
    probs  = np.array(probs)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        'Accuracy':      accuracy_score(labels, preds),
        'Balanced_Acc':  balanced_accuracy_score(labels, preds),
        'Precision':     precision_score(labels, preds, zero_division=0),
        'Recall':        recall_score(labels, preds, zero_division=0),
        'F1':            f1_score(labels, preds, zero_division=0),
        'ROC_AUC':       roc_auc_score(labels, probs),
        'Avg_Precision': average_precision_score(labels, probs),
        'MCC':           matthews_corrcoef(labels, preds),
        'Specificity':   tn / (tn + fp + 1e-9),
        'Sensitivity':   tp / (tp + fn + 1e-9),
        'FPR':           fp / (fp + tn + 1e-9),
        'FNR':           fn / (fn + tp + 1e-9),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn)
    }


def train_epoch(model, loader, criterion, optimizer, scheduler, scaler, step_counter):
    model.train()
    total_loss                       = 0.0
    all_labels, all_preds, all_probs = [], [], []
    optimizer.zero_grad()

    for batch_idx, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(PRIMARY_DEVICE)
        attention_mask = batch['attention_mask'].to(PRIMARY_DEVICE)
        labels         = batch['label'].to(PRIMARY_DEVICE)

        logits = model(input_ids, attention_mask)
        loss   = criterion(logits.float(), labels) / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (batch_idx + 1) % GRAD_ACCUM == 0 or (batch_idx + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0
            )
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            step_counter[0] += 1

        total_loss += (loss * GRAD_ACCUM).item()

        with torch.no_grad():
            logits_f = logits.float()
            probs    = torch.softmax(logits_f, dim=-1)[:, 1].cpu().numpy()
            preds    = logits_f.argmax(dim=-1).cpu().numpy()

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)

    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss                       = 0.0
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(PRIMARY_DEVICE)
            attention_mask = batch['attention_mask'].to(PRIMARY_DEVICE)
            labels         = batch['label'].to(PRIMARY_DEVICE)

            logits     = model(input_ids, attention_mask)
            logits_f   = logits.float()
            loss       = criterion(logits_f, labels)
            total_loss += loss.item()

            probs = torch.softmax(logits_f, dim=-1)[:, 1].cpu().numpy()
            preds = logits_f.argmax(dim=-1).cpu().numpy()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds)
            all_probs.extend(probs)

    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)


def get_trainable_state(model):
    return {
        'prefix_encoder': {k: v.cpu().clone() for k, v in model.prefix_encoder.state_dict().items()},
        'classifier':     {k: v.cpu().clone() for k, v in model.classifier.state_dict().items()}
    }


def accumulate_swa_state(swa_acc, model):
    src = get_trainable_state(model)
    if swa_acc is None:
        return copy.deepcopy(src)
    for module_key in ('prefix_encoder', 'classifier'):
        for k in swa_acc[module_key]:
            swa_acc[module_key][k] = swa_acc[module_key][k] + src[module_key][k]
    return swa_acc


def apply_swa_avg_state(swa_acc, n_swa):
    for module_key in ('prefix_encoder', 'classifier'):
        for k in swa_acc[module_key]:
            swa_acc[module_key][k] = swa_acc[module_key][k].float().div_(n_swa)
    return swa_acc


def load_trainable_state(model, state):
    model.prefix_encoder.load_state_dict(
        {k: v.to(PRIMARY_DEVICE) for k, v in state['prefix_encoder'].items()}
    )
    model.classifier.load_state_dict(
        {k: v.to(PRIMARY_DEVICE) for k, v in state['classifier'].items()}
    )


def plot_loss_curve(history, save_path='pt_loss_curve.png'):
    epochs = list(range(1, len(history['tr_loss']) + 1))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(epochs, history['tr_loss'], 'b-o', markersize=4, linewidth=1.8, label='Train Loss')
    ax.plot(epochs, history['va_loss'], 'r-o', markersize=4, linewidth=1.8, label='Val Loss')
    ax.set_title(
        'Prefix-Tuning (CodeT5p-220m) — Train vs Validation Loss',
        fontsize=13, fontweight='bold'
    )
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Loss curve saved to {save_path}")


def plot_training_curves(history, save_path='pt_training_curves.png'):
    epochs = list(range(1, len(history['tr_loss']) + 1))
    fig, axes = plt.subplots(3, 3, figsize=(18, 14))
    fig.suptitle(
        'Prefix-Tuning (CodeT5p-220m) Training Curves',
        fontsize=16, fontweight='bold'
    )

    def plot_ax(ax, key, title, ylabel):
        ax.plot(epochs, history[f'tr_{key}'], 'b-o', markersize=3, label='Train')
        ax.plot(epochs, history[f'va_{key}'], 'r-o', markersize=3, label='Val')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(True, alpha=0.3)

    plot_ax(axes[0][0], 'loss',    'Loss',          'Loss')
    plot_ax(axes[0][1], 'acc',     'Accuracy',      'Accuracy')
    plot_ax(axes[0][2], 'f1',      'F1 Score',      'F1')
    plot_ax(axes[1][0], 'auc',     'ROC AUC',       'AUC')
    plot_ax(axes[1][1], 'mcc',     'MCC',           'MCC')
    plot_ax(axes[1][2], 'prec',    'Precision',     'Precision')
    plot_ax(axes[2][0], 'rec',     'Recall',        'Recall')
    plot_ax(axes[2][1], 'bacc',    'Balanced Acc',  'Balanced Accuracy')
    plot_ax(axes[2][2], 'avgprec', 'Avg Precision', 'Avg Precision')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Training curves saved to {save_path}")


def plot_final_comparison(best_test_m, swa_test_m, save_path='pt_test_comparison.png'):
    keys      = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Avg_Precision']
    x         = np.arange(len(keys))
    w         = 0.35
    best_vals = [best_test_m[k] for k in keys]
    swa_vals  = [swa_test_m[k] for k in keys] if swa_test_m else None
    fig, ax   = plt.subplots(figsize=(14, 6))
    bars1     = ax.bar(x - w / 2, best_vals, w, label='Best Checkpoint', color='steelblue', alpha=0.85)
    if swa_vals:
        bars2 = ax.bar(x + w / 2, swa_vals, w, label='SWA Model', color='darkorange', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(keys, rotation=20, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title('Test Set: Best Checkpoint vs SWA Model', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
    if swa_vals:
        for bar in bars2:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Test comparison chart saved to {save_path}")


def plot_confusion_matrix(m, title, save_path):
    cm      = np.array([[m['TN'], m['FP']], [m['FN'], m['TP']]])
    fig, ax = plt.subplots(figsize=(5, 4))
    im      = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred 0', 'Pred 1'])
    ax.set_yticklabels(['True 0', 'True 1'])
    thresh = cm.max() / 2.0
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > thresh else 'black',
                    fontsize=14, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Confusion matrix saved to {save_path}")


def plot_metrics_radar(best_test_m, swa_test_m, save_path='pt_radar_chart.png'):
    keys   = ['Accuracy', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Specificity', 'Sensitivity']
    n      = len(keys)
    angles = [x / float(n) * 2 * np.pi for x in range(n)]
    angles += angles[:1]
    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

    def draw(m, color, label):
        vals  = [m[k] for k in keys]
        vals += vals[:1]
        ax.plot(angles, vals, color=color, linewidth=2, label=label)
        ax.fill(angles, vals, color=color, alpha=0.15)

    draw(best_test_m, 'steelblue', 'Best Checkpoint')
    if swa_test_m:
        draw(swa_test_m, 'darkorange', 'SWA Model')
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(keys, size=9)
    ax.set_ylim(0, 1)
    ax.set_title('Test Metrics Radar', fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Radar chart saved to {save_path}")


def main():
    n_gpus = torch.cuda.device_count()
    print(f"Using {n_gpus} GPU(s): {[torch.cuda.get_device_name(i) for i in range(n_gpus)]}")

    train_full = pd.read_csv(TRAIN_CSV)
    test_full  = pd.read_csv(TEST_CSV)

    train_df, val_df = train_test_split(
        train_full, test_size=VAL_SPLIT, random_state=SEED, stratify=train_full['label']
    )
    _, test_df = train_test_split(
        test_full, test_size=0.50, random_state=SEED, stratify=test_full['label']
    )
    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}\n")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = 'right'

    train_ds     = CodeDataset(train_df.reset_index(drop=True), tokenizer)
    val_ds       = CodeDataset(val_df.reset_index(drop=True),   tokenizer)
    test_ds      = CodeDataset(test_df.reset_index(drop=True),  tokenizer)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=False, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=False)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=False)

    counts        = train_df['label'].value_counts().sort_index().values
    raw_w         = torch.tensor([1.0 / c for c in counts], dtype=torch.float).to(PRIMARY_DEVICE)
    class_weights = raw_w / raw_w.sum() * len(counts)

    print("Loading backbone with device_map='auto' to shard across all available GPUs...")
    model = PrefixTuningClassifier(
        model_name  = MODEL_NAME,
        pre_seq_len = PRE_SEQ_LEN,
        mid_dim     = MID_DIM,
        num_labels  = 2
    )
    print(f"Encoder device: {model._encoder_device}\n")

    total_p   = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Params     : {total_p:,}")
    print(f"Trainable Params : {trainable:,}")
    print(f"Trainable %      : {100.0 * trainable / total_p:.4f}%\n")

    criterion        = nn.CrossEntropyLoss(weight=class_weights)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer        = torch.optim.AdamW(
        trainable_params, lr=LR, weight_decay=WEIGHT_DECAY,
        betas=(0.9, 0.999), eps=1e-8
    )
    effective_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    warmup_steps    = int(WARMUP_RATIO * effective_steps)
    scheduler       = get_cosine_schedule_with_warmup(optimizer, warmup_steps, effective_steps)
    scaler          = torch.cuda.amp.GradScaler()

    swa_start_epoch = int(SWA_START_RATIO * EPOCHS)
    swa_acc         = None
    n_swa           = 0
    step_counter    = [0]
    best_val_f1     = -1.0
    best_epoch      = 0
    best_state      = None
    no_improve      = 0

    history = {
        'tr_loss': [], 'tr_acc': [], 'tr_f1': [], 'tr_auc': [], 'tr_mcc': [],
        'tr_prec': [], 'tr_rec': [], 'tr_bacc': [], 'tr_avgprec': [],
        'va_loss': [], 'va_acc': [], 'va_f1': [], 'va_auc': [], 'va_mcc': [],
        'va_prec': [], 'va_rec': [], 'va_bacc': [], 'va_avgprec': []
    }

    hdr = (f"{'Ep':>3} | {'TrLoss':>8} | {'TrAcc':>7} | {'TrF1':>7} | {'TrAUC':>7} | {'TrMCC':>7} | "
           f"{'VaLoss':>8} | {'VaAcc':>7} | {'VaF1':>7} | {'VaAUC':>7} | {'VaMCC':>7} | {'Note':<12}")
    sep = "-" * len(hdr)
    print(sep)
    print(hdr)
    print(sep)

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_m = train_epoch(
            model, train_loader, criterion, optimizer, scheduler, scaler, step_counter
        )
        va_loss, va_m = eval_epoch(model, val_loader, criterion)

        history['tr_loss'].append(tr_loss)
        history['tr_acc'].append(tr_m['Accuracy'])
        history['tr_f1'].append(tr_m['F1'])
        history['tr_auc'].append(tr_m['ROC_AUC'])
        history['tr_mcc'].append(tr_m['MCC'])
        history['tr_prec'].append(tr_m['Precision'])
        history['tr_rec'].append(tr_m['Recall'])
        history['tr_bacc'].append(tr_m['Balanced_Acc'])
        history['tr_avgprec'].append(tr_m['Avg_Precision'])
        history['va_loss'].append(va_loss)
        history['va_acc'].append(va_m['Accuracy'])
        history['va_f1'].append(va_m['F1'])
        history['va_auc'].append(va_m['ROC_AUC'])
        history['va_mcc'].append(va_m['MCC'])
        history['va_prec'].append(va_m['Precision'])
        history['va_rec'].append(va_m['Recall'])
        history['va_bacc'].append(va_m['Balanced_Acc'])
        history['va_avgprec'].append(va_m['Avg_Precision'])

        note = ""
        if epoch > swa_start_epoch:
            swa_acc = accumulate_swa_state(swa_acc, model)
            n_swa  += 1
            note   += "SWA "

        is_best = va_m['F1'] > best_val_f1
        if is_best:
            best_val_f1 = va_m['F1']
            best_epoch  = epoch
            best_state  = get_trainable_state(model)
            no_improve  = 0
            note       += "<--"
        else:
            no_improve += 1

        print(
            f"{epoch:>3} | {tr_loss:>8.4f} | {tr_m['Accuracy']:>7.4f} | {tr_m['F1']:>7.4f} | "
            f"{tr_m['ROC_AUC']:>7.4f} | {tr_m['MCC']:>7.4f} | "
            f"{va_loss:>8.4f} | {va_m['Accuracy']:>7.4f} | {va_m['F1']:>7.4f} | "
            f"{va_m['ROC_AUC']:>7.4f} | {va_m['MCC']:>7.4f} | {note:<12}"
        )

        if no_improve >= PATIENCE:
            print(f"\n  Early stopping at epoch {epoch} (no val F1 improvement for {PATIENCE} epochs)")
            break

    print(sep)
    print(f"\nBest Epoch: {best_epoch}  |  Best Val F1: {best_val_f1:.4f}")

    print("\n--- Evaluating: Best Checkpoint Model ---")
    load_trainable_state(model, best_state)
    _, best_test_m = eval_epoch(model, test_loader, criterion)

    print("\n--- Evaluating: SWA Model ---")
    swa_test_m = None
    if n_swa > 0:
        swa_avg_state = apply_swa_avg_state(swa_acc, n_swa)
        load_trainable_state(model, swa_avg_state)
        _, swa_test_m = eval_epoch(model, test_loader, criterion)
        load_trainable_state(model, best_state)

    print(f"\n{'='*60}")
    print(f"  Val F1 trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_f1'][-5:]]}")
    print(f"  Val Loss trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_loss'][-5:]]}")
    print(f"\n{'='*60}")
    print(f"  TEST RESULTS — Best Checkpoint (Epoch {best_epoch})")
    print(f"{'='*60}")
    scalar_keys = [k for k in best_test_m if k not in ('TP', 'TN', 'FP', 'FN')]
    for k in scalar_keys:
        print(f"  {k:<22}: {best_test_m[k]:.4f}")
    print(f"  {'TP':<22}: {best_test_m['TP']}")
    print(f"  {'TN':<22}: {best_test_m['TN']}")
    print(f"  {'FP':<22}: {best_test_m['FP']}")
    print(f"  {'FN':<22}: {best_test_m['FN']}")

    if swa_test_m is not None:
        print(f"\n{'='*60}")
        print(f"  TEST RESULTS — SWA Model (averaged over last {n_swa} epochs)")
        print(f"{'='*60}")
        for k in scalar_keys:
            print(f"  {k:<22}: {swa_test_m[k]:.4f}")
        print(f"  {'TP':<22}: {swa_test_m['TP']}")
        print(f"  {'TN':<22}: {swa_test_m['TN']}")
        print(f"  {'FP':<22}: {swa_test_m['FP']}")
        print(f"  {'FN':<22}: {swa_test_m['FN']}")
        print(f"\n{'='*60}")
        print(f"  COMPARISON: Best Checkpoint vs SWA")
        print(f"{'='*60}")
        compare_keys = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall']
        print(f"  {'Metric':<22} {'BestCkpt':>12} {'SWA':>12} {'Delta':>12}")
        print(f"  {'-'*58}")
        for k in compare_keys:
            delta  = swa_test_m[k] - best_test_m[k]
            symbol = "+" if delta >= 0 else ""
            print(f"  {k:<22} {best_test_m[k]:>12.4f} {swa_test_m[k]:>12.4f} {symbol+f'{delta:.4f}':>12}")

    plot_loss_curve(history,      save_path='pt_loss_curve.png')
    plot_training_curves(history, save_path='pt_training_curves.png')
    plot_final_comparison(best_test_m, swa_test_m, save_path='pt_test_comparison.png')
    plot_confusion_matrix(best_test_m,
                          f'Confusion Matrix — Best Ckpt (Ep {best_epoch})',
                          'pt_cm_best.png')
    if swa_test_m:
        plot_confusion_matrix(swa_test_m,
                              f'Confusion Matrix — SWA ({n_swa} epochs)',
                              'pt_cm_swa.png')
    plot_metrics_radar(best_test_m, swa_test_m, save_path='pt_radar_chart.png')
    print("\nAll figures saved.")


if __name__ == "__main__":
    main()

# prefix codet5 770 1024 token

In [ ]:
import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix,
    balanced_accuracy_score, average_precision_score
)
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings


warnings.filterwarnings('ignore')
os.environ["CUDA_VISIBLE_DEVICES"]     = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"]   = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

torch.backends.cuda.matmul.allow_tf32                    = True
torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = True
torch.backends.cudnn.allow_tf32                          = True

MODEL_NAME      = "Salesforce/codet5p-770m"
TRAIN_CSV       = 'code vul/trqw.csv'
TEST_CSV        = 'code vul/teqw.csv'

MAX_LENGTH      = 512
PRE_SEQ_LEN     = 100
MID_DIM         = 512
BATCH_SIZE      = 4
GRAD_ACCUM      = 8
EPOCHS          = 30
LR              = 5e-5
WEIGHT_DECAY    = 0.0
WARMUP_RATIO    = 0.15
VAL_SPLIT       = 0.20
SEED            = 42
SWA_START_RATIO = 0.6
PATIENCE        = 7
NUM_WORKERS     = 4
PREFIX_DROPOUT  = 0.0
PRIMARY_DEVICE  = torch.device("cuda:0")

_PIN = torch.cuda.is_available()

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


class CodeDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.codes     = df['func_cleaned'].astype(str).tolist()
        self.labels    = df['label'].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.codes[idx],
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


class PrefixEncoder(nn.Module):
    def __init__(self, pre_seq_len, d_model, mid_dim, dropout_rate):
        super().__init__()
        self.pre_seq_len = pre_seq_len
        self.register_buffer('prefix_tokens', torch.arange(pre_seq_len).long())
        self.embedding      = nn.Embedding(pre_seq_len, mid_dim)
        self.trans          = nn.Sequential(
            nn.Linear(mid_dim, mid_dim),
            nn.Tanh(),
            nn.Linear(mid_dim, d_model)
        )
        self.prefix_dropout = nn.Dropout(dropout_rate)

    def forward(self, batch_size):
        prefix_tokens = self.prefix_tokens.unsqueeze(0).expand(batch_size, -1)
        prefix_embeds = self.embedding(prefix_tokens)
        prefix_embeds = self.trans(prefix_embeds)
        prefix_embeds = self.prefix_dropout(prefix_embeds)
        return prefix_embeds


def _get_encoder_embed_device(backbone):
    for attr_path in (
        ('encoder', 'embed_tokens'),
        ('shared',),
        ('encoder',),
    ):
        obj = backbone
        try:
            for attr in attr_path:
                obj = getattr(obj, attr)
            return next(obj.parameters()).device
        except Exception:
            continue
    return PRIMARY_DEVICE


class PrefixTuningClassifier(nn.Module):
    def __init__(self, model_name, pre_seq_len, mid_dim, num_labels=2):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(
            model_name,
            device_map='auto',
            torch_dtype=torch.float16,
            trust_remote_code=True
        )
        for param in self.backbone.parameters():
            param.requires_grad = False

        if hasattr(self.backbone, 'encoder') and hasattr(self.backbone.encoder, 'gradient_checkpointing_enable'):
            self.backbone.encoder.gradient_checkpointing_enable()
        elif hasattr(self.backbone, 'gradient_checkpointing_enable'):
            self.backbone.gradient_checkpointing_enable()

        cfg     = self.backbone.config
        d_model = getattr(cfg, 'd_model', getattr(cfg, 'hidden_size', 1024))

        self.pre_seq_len   = pre_seq_len
        self.d_model       = d_model
        self._embed_device = _get_encoder_embed_device(self.backbone)

        self.prefix_encoder = PrefixEncoder(
            pre_seq_len  = pre_seq_len,
            d_model      = d_model,
            mid_dim      = mid_dim,
            dropout_rate = PREFIX_DROPOUT
        ).to(PRIMARY_DEVICE)

        self.classifier = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(d_model, num_labels)
        ).to(PRIMARY_DEVICE)

    def forward(self, input_ids, attention_mask):
        batch_size = input_ids.size(0)

        input_ids_enc      = input_ids.to(self._embed_device)
        attention_mask_enc = attention_mask.to(self._embed_device)

        token_embeds  = self.backbone.encoder.embed_tokens(input_ids_enc)

        prefix_embeds = self.prefix_encoder(batch_size)
        prefix_embeds = prefix_embeds.to(
            device=self._embed_device, dtype=token_embeds.dtype
        )

        all_embeds = torch.cat([prefix_embeds, token_embeds], dim=1)

        prefix_mask = torch.ones(
            batch_size, self.pre_seq_len,
            dtype=attention_mask_enc.dtype,
            device=self._embed_device
        )
        ext_mask = torch.cat([prefix_mask, attention_mask_enc], dim=1)

        outputs = self.backbone.encoder(
            inputs_embeds  = all_embeds,
            attention_mask = ext_mask,
            return_dict    = True
        )

        last_hs    = outputs.last_hidden_state
        token_hs   = last_hs[:, self.pre_seq_len:, :]
        mask_f     = attention_mask_enc.unsqueeze(-1).to(
            dtype=torch.float32, device=token_hs.device
        )
        token_hs_f = token_hs.to(dtype=torch.float32)
        pooled     = (token_hs_f * mask_f).sum(1) / mask_f.sum(1).clamp(min=1e-9)
        pooled     = pooled.to(device=PRIMARY_DEVICE)

        return self.classifier(pooled)


def compute_metrics(labels, preds, probs):
    labels = np.array(labels)
    preds  = np.array(preds)
    probs  = np.array(probs)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        'Accuracy':      accuracy_score(labels, preds),
        'Balanced_Acc':  balanced_accuracy_score(labels, preds),
        'Precision':     precision_score(labels, preds, zero_division=0),
        'Recall':        recall_score(labels, preds, zero_division=0),
        'F1':            f1_score(labels, preds, zero_division=0),
        'ROC_AUC':       roc_auc_score(labels, probs),
        'Avg_Precision': average_precision_score(labels, probs),
        'MCC':           matthews_corrcoef(labels, preds),
        'Specificity':   tn / (tn + fp + 1e-9),
        'Sensitivity':   tp / (tp + fn + 1e-9),
        'FPR':           fp / (fp + tn + 1e-9),
        'FNR':           fn / (fn + tp + 1e-9),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn)
    }


def train_epoch(model, loader, criterion, optimizer, scheduler, scaler, step_counter):
    model.train()
    total_loss                       = 0.0
    all_labels, all_preds, all_probs = [], [], []
    optimizer.zero_grad()

    for batch_idx, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(PRIMARY_DEVICE)
        attention_mask = batch['attention_mask'].to(PRIMARY_DEVICE)
        labels         = batch['label'].to(PRIMARY_DEVICE)

        logits = model(input_ids, attention_mask)
        loss   = criterion(logits.float(), labels) / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (batch_idx + 1) % GRAD_ACCUM == 0 or (batch_idx + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0
            )
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            step_counter[0] += 1

        total_loss += (loss * GRAD_ACCUM).item()

        with torch.no_grad():
            logits_f = logits.float()
            probs    = torch.softmax(logits_f, dim=-1)[:, 1].cpu().numpy()
            preds    = logits_f.argmax(dim=-1).cpu().numpy()

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)

        del input_ids, attention_mask, labels, logits, loss
        if batch_idx % 50 == 0:
            torch.cuda.empty_cache()

    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss                       = 0.0
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(PRIMARY_DEVICE)
            attention_mask = batch['attention_mask'].to(PRIMARY_DEVICE)
            labels         = batch['label'].to(PRIMARY_DEVICE)

            logits     = model(input_ids, attention_mask)
            logits_f   = logits.float()
            loss       = criterion(logits_f, labels)
            total_loss += loss.item()

            probs = torch.softmax(logits_f, dim=-1)[:, 1].cpu().numpy()
            preds = logits_f.argmax(dim=-1).cpu().numpy()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds)
            all_probs.extend(probs)

            del input_ids, attention_mask, labels, logits, logits_f, loss

    torch.cuda.empty_cache()
    return total_loss / len(loader), compute_metrics(all_labels, all_preds, all_probs)


def get_trainable_state(model):
    return {
        'prefix_encoder': {k: v.cpu().clone() for k, v in model.prefix_encoder.state_dict().items()},
        'classifier':     {k: v.cpu().clone() for k, v in model.classifier.state_dict().items()}
    }


def accumulate_swa_state(swa_acc, model):
    src = get_trainable_state(model)
    if swa_acc is None:
        return copy.deepcopy(src)
    for module_key in ('prefix_encoder', 'classifier'):
        for k in swa_acc[module_key]:
            swa_acc[module_key][k] = swa_acc[module_key][k] + src[module_key][k]
    return swa_acc


def apply_swa_avg_state(swa_acc, n_swa):
    for module_key in ('prefix_encoder', 'classifier'):
        for k in swa_acc[module_key]:
            swa_acc[module_key][k] = swa_acc[module_key][k].float().div_(n_swa)
    return swa_acc


def load_trainable_state(model, state):
    model.prefix_encoder.load_state_dict(
        {k: v.to(PRIMARY_DEVICE) for k, v in state['prefix_encoder'].items()}
    )
    model.classifier.load_state_dict(
        {k: v.to(PRIMARY_DEVICE) for k, v in state['classifier'].items()}
    )


def plot_loss_curve(history, save_path='pt_loss_curve.png'):
    epochs = list(range(1, len(history['tr_loss']) + 1))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(epochs, history['tr_loss'], 'b-o', markersize=4, linewidth=1.8, label='Train Loss')
    ax.plot(epochs, history['va_loss'], 'r-o', markersize=4, linewidth=1.8, label='Val Loss')
    ax.set_title(
        'Prefix-Tuning (CodeT5+ 770M) — Train vs Validation Loss',
        fontsize=13, fontweight='bold'
    )
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Loss curve saved to {save_path}")


def plot_training_curves(history, save_path='pt_training_curves.png'):
    epochs = list(range(1, len(history['tr_loss']) + 1))
    fig, axes = plt.subplots(3, 3, figsize=(18, 14))
    fig.suptitle(
        'Prefix-Tuning (CodeT5+ 770M) Training Curves',
        fontsize=16, fontweight='bold'
    )

    def plot_ax(ax, key, title, ylabel):
        ax.plot(epochs, history[f'tr_{key}'], 'b-o', markersize=3, label='Train')
        ax.plot(epochs, history[f'va_{key}'], 'r-o', markersize=3, label='Val')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(True, alpha=0.3)

    plot_ax(axes[0][0], 'loss',    'Loss',          'Loss')
    plot_ax(axes[0][1], 'acc',     'Accuracy',      'Accuracy')
    plot_ax(axes[0][2], 'f1',      'F1 Score',      'F1')
    plot_ax(axes[1][0], 'auc',     'ROC AUC',       'AUC')
    plot_ax(axes[1][1], 'mcc',     'MCC',           'MCC')
    plot_ax(axes[1][2], 'prec',    'Precision',     'Precision')
    plot_ax(axes[2][0], 'rec',     'Recall',        'Recall')
    plot_ax(axes[2][1], 'bacc',    'Balanced Acc',  'Balanced Accuracy')
    plot_ax(axes[2][2], 'avgprec', 'Avg Precision', 'Avg Precision')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Training curves saved to {save_path}")


def plot_final_comparison(best_test_m, swa_test_m, save_path='pt_test_comparison.png'):
    keys      = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Avg_Precision']
    x         = np.arange(len(keys))
    w         = 0.35
    best_vals = [best_test_m[k] for k in keys]
    swa_vals  = [swa_test_m[k] for k in keys] if swa_test_m else None
    fig, ax   = plt.subplots(figsize=(14, 6))
    bars1     = ax.bar(x - w / 2, best_vals, w, label='Best Checkpoint', color='steelblue', alpha=0.85)
    if swa_vals:
        bars2 = ax.bar(x + w / 2, swa_vals, w, label='SWA Model', color='darkorange', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(keys, rotation=20, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title('Test Set: Best Checkpoint vs SWA Model', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
    if swa_vals:
        for bar in bars2:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Test comparison chart saved to {save_path}")


def plot_confusion_matrix(m, title, save_path):
    cm      = np.array([[m['TN'], m['FP']], [m['FN'], m['TP']]])
    fig, ax = plt.subplots(figsize=(5, 4))
    im      = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred 0', 'Pred 1'])
    ax.set_yticklabels(['True 0', 'True 1'])
    thresh = cm.max() / 2.0
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > thresh else 'black',
                    fontsize=14, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Confusion matrix saved to {save_path}")


def plot_metrics_radar(best_test_m, swa_test_m, save_path='pt_radar_chart.png'):
    keys   = ['Accuracy', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Specificity', 'Sensitivity']
    n      = len(keys)
    angles = [x / float(n) * 2 * np.pi for x in range(n)]
    angles += angles[:1]
    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

    def draw(m, color, label):
        vals  = [m[k] for k in keys]
        vals += vals[:1]
        ax.plot(angles, vals, color=color, linewidth=2, label=label)
        ax.fill(angles, vals, color=color, alpha=0.15)

    draw(best_test_m, 'steelblue', 'Best Checkpoint')
    if swa_test_m:
        draw(swa_test_m, 'darkorange', 'SWA Model')
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(keys, size=9)
    ax.set_ylim(0, 1)
    ax.set_title('Test Metrics Radar', fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Radar chart saved to {save_path}")


def main():
    n_gpus = torch.cuda.device_count()
    print(f"Using {n_gpus} GPU(s): {[torch.cuda.get_device_name(i) for i in range(n_gpus)]}")

    train_full = pd.read_csv(TRAIN_CSV)
    test_full  = pd.read_csv(TEST_CSV)

    train_df, val_df = train_test_split(
        train_full, test_size=VAL_SPLIT, random_state=SEED, stratify=train_full['label']
    )
    _, test_df = train_test_split(
        test_full, test_size=0.50, random_state=SEED, stratify=test_full['label']
    )
    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}\n")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = 'right'

    train_ds     = CodeDataset(train_df.reset_index(drop=True), tokenizer)
    val_ds       = CodeDataset(val_df.reset_index(drop=True),   tokenizer)
    test_ds      = CodeDataset(test_df.reset_index(drop=True),  tokenizer)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=_PIN, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=_PIN)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=_PIN)

    counts        = train_df['label'].value_counts().sort_index().values
    raw_w         = torch.tensor([1.0 / c for c in counts], dtype=torch.float).to(PRIMARY_DEVICE)
    class_weights = raw_w / raw_w.sum() * len(counts)

    print("Loading CodeT5+ 770M with device_map='auto' to shard across all available GPUs...")
    model = PrefixTuningClassifier(
        model_name  = MODEL_NAME,
        pre_seq_len = PRE_SEQ_LEN,
        mid_dim     = MID_DIM,
        num_labels  = 2
    )
    print(f"Encoder embed device : {model._embed_device}")
    print(f"Encoder d_model      : {model.d_model}\n")

    total_p   = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Params     : {total_p:,}")
    print(f"Trainable Params : {trainable:,}")
    print(f"Trainable %      : {100.0 * trainable / total_p:.4f}%\n")

    criterion        = nn.CrossEntropyLoss(weight=class_weights)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer        = torch.optim.AdamW(
        trainable_params, lr=LR, weight_decay=WEIGHT_DECAY,
        betas=(0.9, 0.999), eps=1e-8
    )
    effective_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    warmup_steps    = int(WARMUP_RATIO * effective_steps)
    scheduler       = get_cosine_schedule_with_warmup(optimizer, warmup_steps, effective_steps)
    scaler          = torch.cuda.amp.GradScaler()

    swa_start_epoch = int(SWA_START_RATIO * EPOCHS)
    swa_acc         = None
    n_swa           = 0
    step_counter    = [0]
    best_val_f1     = -1.0
    best_epoch      = 0
    best_state      = None
    no_improve      = 0

    history = {
        'tr_loss': [], 'tr_acc': [], 'tr_f1': [], 'tr_auc': [], 'tr_mcc': [],
        'tr_prec': [], 'tr_rec': [], 'tr_bacc': [], 'tr_avgprec': [],
        'va_loss': [], 'va_acc': [], 'va_f1': [], 'va_auc': [], 'va_mcc': [],
        'va_prec': [], 'va_rec': [], 'va_bacc': [], 'va_avgprec': []
    }

    hdr = (f"{'Ep':>3} | {'TrLoss':>8} | {'TrAcc':>7} | {'TrF1':>7} | {'TrAUC':>7} | {'TrMCC':>7} | "
           f"{'VaLoss':>8} | {'VaAcc':>7} | {'VaF1':>7} | {'VaAUC':>7} | {'VaMCC':>7} | {'Note':<12}")
    sep = "-" * len(hdr)
    print(sep)
    print(hdr)
    print(sep)

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_m = train_epoch(
            model, train_loader, criterion, optimizer, scheduler, scaler, step_counter
        )
        va_loss, va_m = eval_epoch(model, val_loader, criterion)

        history['tr_loss'].append(tr_loss)
        history['tr_acc'].append(tr_m['Accuracy'])
        history['tr_f1'].append(tr_m['F1'])
        history['tr_auc'].append(tr_m['ROC_AUC'])
        history['tr_mcc'].append(tr_m['MCC'])
        history['tr_prec'].append(tr_m['Precision'])
        history['tr_rec'].append(tr_m['Recall'])
        history['tr_bacc'].append(tr_m['Balanced_Acc'])
        history['tr_avgprec'].append(tr_m['Avg_Precision'])
        history['va_loss'].append(va_loss)
        history['va_acc'].append(va_m['Accuracy'])
        history['va_f1'].append(va_m['F1'])
        history['va_auc'].append(va_m['ROC_AUC'])
        history['va_mcc'].append(va_m['MCC'])
        history['va_prec'].append(va_m['Precision'])
        history['va_rec'].append(va_m['Recall'])
        history['va_bacc'].append(va_m['Balanced_Acc'])
        history['va_avgprec'].append(va_m['Avg_Precision'])

        note = ""
        if epoch > swa_start_epoch:
            swa_acc = accumulate_swa_state(swa_acc, model)
            n_swa  += 1
            note   += "SWA "

        is_best = va_m['F1'] > best_val_f1
        if is_best:
            best_val_f1 = va_m['F1']
            best_epoch  = epoch
            best_state  = get_trainable_state(model)
            no_improve  = 0
            note       += "<--"
        else:
            no_improve += 1

        print(
            f"{epoch:>3} | {tr_loss:>8.4f} | {tr_m['Accuracy']:>7.4f} | {tr_m['F1']:>7.4f} | "
            f"{tr_m['ROC_AUC']:>7.4f} | {tr_m['MCC']:>7.4f} | "
            f"{va_loss:>8.4f} | {va_m['Accuracy']:>7.4f} | {va_m['F1']:>7.4f} | "
            f"{va_m['ROC_AUC']:>7.4f} | {va_m['MCC']:>7.4f} | {note:<12}"
        )

        if no_improve >= PATIENCE:
            print(f"\n  Early stopping at epoch {epoch} (no val F1 improvement for {PATIENCE} epochs)")
            break

    print(sep)
    print(f"\nBest Epoch: {best_epoch}  |  Best Val F1: {best_val_f1:.4f}")

    print("\n--- Evaluating: Best Checkpoint Model ---")
    load_trainable_state(model, best_state)
    _, best_test_m = eval_epoch(model, test_loader, criterion)

    print("\n--- Evaluating: SWA Model ---")
    swa_test_m = None
    if n_swa > 0:
        swa_avg_state = apply_swa_avg_state(swa_acc, n_swa)
        load_trainable_state(model, swa_avg_state)
        _, swa_test_m = eval_epoch(model, test_loader, criterion)
        load_trainable_state(model, best_state)

    print(f"\n{'='*60}")
    print(f"  Val F1 trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_f1'][-5:]]}")
    print(f"  Val Loss trend (last 5 epochs): {[f'{x:.4f}' for x in history['va_loss'][-5:]]}")
    print(f"\n{'='*60}")
    print(f"  TEST RESULTS — Best Checkpoint (Epoch {best_epoch})")
    print(f"{'='*60}")
    scalar_keys = [k for k in best_test_m if k not in ('TP', 'TN', 'FP', 'FN')]
    for k in scalar_keys:
        print(f"  {k:<22}: {best_test_m[k]:.4f}")
    print(f"  {'TP':<22}: {best_test_m['TP']}")
    print(f"  {'TN':<22}: {best_test_m['TN']}")
    print(f"  {'FP':<22}: {best_test_m['FP']}")
    print(f"  {'FN':<22}: {best_test_m['FN']}")

    if swa_test_m is not None:
        print(f"\n{'='*60}")
        print(f"  TEST RESULTS — SWA Model (averaged over last {n_swa} epochs)")
        print(f"{'='*60}")
        for k in scalar_keys:
            print(f"  {k:<22}: {swa_test_m[k]:.4f}")
        print(f"  {'TP':<22}: {swa_test_m['TP']}")
        print(f"  {'TN':<22}: {swa_test_m['TN']}")
        print(f"  {'FP':<22}: {swa_test_m['FP']}")
        print(f"  {'FN':<22}: {swa_test_m['FN']}")
        print(f"\n{'='*60}")
        print(f"  COMPARISON: Best Checkpoint vs SWA")
        print(f"{'='*60}")
        compare_keys = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall']
        print(f"  {'Metric':<22} {'BestCkpt':>12} {'SWA':>12} {'Delta':>12}")
        print(f"  {'-'*58}")
        for k in compare_keys:
            delta  = swa_test_m[k] - best_test_m[k]
            symbol = "+" if delta >= 0 else ""
            print(f"  {k:<22} {best_test_m[k]:>12.4f} {swa_test_m[k]:>12.4f} {symbol+f'{delta:.4f}':>12}")

    plot_loss_curve(history,      save_path='pt_loss_curve.png')
    plot_training_curves(history, save_path='pt_training_curves.png')
    plot_final_comparison(best_test_m, swa_test_m, save_path='pt_test_comparison.png')
    plot_confusion_matrix(best_test_m,
                          f'Confusion Matrix — Best Ckpt (Ep {best_epoch})',
                          'pt_cm_best.png')
    if swa_test_m:
        plot_confusion_matrix(swa_test_m,
                              f'Confusion Matrix — SWA ({n_swa} epochs)',
                              'pt_cm_swa.png')
    plot_metrics_radar(best_test_m, swa_test_m, save_path='pt_radar_chart.png')
    print("\nAll figures saved.")


if __name__ == "__main__":
    main()